> **Edición pública anonimizada.** Nombres y rutas comerciales se sustituyeron por etiquetas genéricas; no se incluyen bases ni modelos originales. Los resultados históricos se consultan en la tesis adaptada. La demostración pública usa datos y modelos sintéticos.


In [ ]:
from pathlib import Path
import os, sys

# Find the repository independently of the computer or clone location.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / ".tesis-portafolio").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Abre Jupyter desde la carpeta de este repositorio.")
os.chdir(REPO_ROOT / '03 Prediccion')


# Modelado Predictivo de Leads Inmobiliarios — Dataset Calificado
## Predicción de Conversión y Abandono mediante Aprendizaje Automático Supervisado

**Autor:** Angie Flores
**Institución:** Universidad del Valle de Guatemala
**Directorio de trabajo:** `03 Prediccion`
**Última actualización:** 2026-06-30

---

### Resumen Ejecutivo

Este notebook constituye el entregable principal de la etapa de modelado predictivo de la tesis. Se abordan dos tareas de clasificación binaria:

1. **Predicción de Conversión:** dado un lead registrado en el CRM, ¿convertirá en venta?
2. **Predicción de Abandono:** dado un lead *en alcance del proyecto*, ¿abandonará sin convertir?

Se evalúan cinco algoritmos (Regresión Logística, Random Forest, Gradient Boosting, XGBoost y LightGBM) sobre cinco conjuntos de características, con y sin sobremuestreo SMOTE, aplicando la misma partición estratificada en todos los experimentos. Los mejores modelos son interpretados con SHAP y evaluados bajo criterios de negocio (Lift, Gain, Recall@K) y simulación Monte Carlo.

**Nota metodológica:** todo el código está escrito en inglés; toda la documentación narrativa está en español.

> **Importante:** esta versión conserva la estructura y metodología del notebook original, pero utiliza únicamente leads calificados. Las conclusiones cuantitativas heredadas deben actualizarse con los resultados obtenidos al ejecutar todas las celdas de esta versión.

---


# Parte I — Diagnóstico Inicial

## Sección 0 — Configuración e Importaciones

Se importan todas las librerías necesarias para el estudio. Las advertencias no críticas se suprimen para mayor legibilidad del notebook.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, time, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, HistGradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import shap
# shap.initjs()  # browser-only, skipped in nbconvert

# ── Reproducibility seed ──────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Output directories ────────────────────────────────────────────────────
BASE_DIR   = Path(f'{REPO_ROOT}/03 Prediccion')
RUN_DIR    = BASE_DIR / 'qualified_run'
DATA_PATH  = Path(f'{REPO_ROOT}/01 Limpieza Código/data/features/leads_enriched_qualified_final.csv')
KMEANS_PATH= Path(f'{REPO_ROOT}/02 K Means/Output/leads_calificados_clustered.csv')
FIGURES_DIR= RUN_DIR / 'figures'
MODELS_DIR = RUN_DIR / 'models'
RESULTS_DIR= RUN_DIR / 'results'
for d in [FIGURES_DIR, MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['font.size']   = 10
import sklearn
print('Librerias cargadas correctamente:')
print(f'  sklearn  : {sklearn.__version__}')
print(f'  imbalanced-learn: {__import__("imblearn").__version__}')
print(f'  shap     : {shap.__version__}')
print()
print('NOTA: XGBoost y LightGBM requieren libomp ARM64 que no esta disponible en')
print('este sistema. Se sustituyen por:')
print('  - HistGradientBoostingClassifier (sklearn) como alternativa a LightGBM')
print('  - ExtraTreesClassifier (sklearn) como alternativa a XGBoost')
print('  Ambos son mencionados explicitamente en los requisitos del estudio.')


## Sección 1 — Carga y Verificación de Integridad del Dataset

El único dataset utilizado para modelado es `leads_enriched_qualified_final.csv`, resultado de la etapa de limpieza y enriquecimiento de datos y filtrado a leads calificados. No se realiza ninguna limpieza adicional; únicamente se aplica el preprocesamiento mínimo requerido para el modelado (imputación, codificación, escalado).


In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Dataset principal  : {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"Conversiones (won=1): {df_raw['target_won'].sum():,}  "
      f"({df_raw['target_won'].mean()*100:.2f}%)")
print(f"No convertidos (won=0): {(df_raw['target_won']==0).sum():,}")
print()
# Integrity checks
assert df_raw['id_trato'].nunique() == len(df_raw), "ERROR: id_trato no es único"
assert df_raw['target_won'].isin([0,1]).all(), "ERROR: target_won tiene valores distintos de 0/1"
assert df_raw['proyecto_fuera_alcance'].isin([0,1]).all()
print("✓ Verificación de integridad superada")
print()
print("Distribución del target (Conversión):")
print(df_raw['target_won'].value_counts())


In [ ]:
# Load K-Means cluster assignments (subset of 42 K leads)
df_kmeans = pd.read_csv(KMEANS_PATH, low_memory=False)
print(f"Dataset K-Means    : {df_kmeans.shape[0]:,} filas × {df_kmeans.shape[1]} columnas")
print(f"Clusters disponibles: {sorted(df_kmeans['cluster'].dropna().unique())}")
print()
print("Distribución de clústeres:")
print(df_kmeans['cluster'].value_counts().sort_index())
print()
# Verify merge key
overlap = set(df_kmeans['id_trato']).intersection(set(df_raw['id_trato']))
print(f"IDs comunes con dataset principal: {len(overlap):,} ({len(overlap)/len(df_kmeans)*100:.1f}%)")


In [ ]:
# ── Target variable: Abandono ─────────────────────────────────────────────
# Task 2 abandonment target = complement of conversion (qualified dataset)
# proyecto_fuera_alcance is EXCLUDED as an irrelevant variable (see ALWAYS_EXCLUDE).
# No in-scope filtering is applied.
target_abandoned_check = (df_raw["target_won"] == 0)
print(f"Abandono (won=0): {target_abandoned_check.sum():,} leads ({target_abandoned_check.mean()*100:.2f}%)")
print(f"Conversion (won=1): {(~target_abandoned_check).sum():,} leads ({(~target_abandoned_check).mean()*100:.2f}%)")
print()
print("NOTA: la variable 'proyecto_fuera_alcance' es irrelevante y se excluye completamente.")


---
> **Hallazgos — Sección 1 (Carga de Datos):** El dataset calificado contiene **42,464 registros** de leads con **62 variables**. De estos, 739 son conversiones exitosas (1.74%), generando un desbalance de clases aproximado de **56.5:1**. La variable objetivo de la Tarea 2 se define como `target_abandoned = (1 - target_won)`, aplicando el mismo split estratificado sobre el dataset calificado. Los **42,464 leads** cuentan con asignación de clúster K-Means disponible para el conjunto de características F5. No se elimina ninguna fila; los valores faltantes se manejan exclusivamente dentro de los pipelines mediante imputación (mediana para numéricas, moda para categóricas).


## Sección 2 — Análisis de Valores Faltantes (Missingness)

El manejo de datos incompletos es un elemento metodológico relevante en cualquier estudio de modelado predictivo. Se analiza la distribución de valores faltantes en el dataset para comprender la cobertura real de cada variable y su impacto en los conjuntos de características utilizados.

Los *pipelines* de preprocesamiento gestionan los valores faltantes de forma automática mediante `SimpleImputer` (mediana para variables numéricas, moda para categóricas), garantizando que no se filtren registros del entrenamiento ni de la evaluación.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 2.1  Análisis de Missingness
# ════════════════════════════════════════════════════════════════════════

miss_count = df_raw.isnull().sum()
miss_pct   = (df_raw.isnull().mean() * 100).round(2)
df_miss = pd.DataFrame({'n_faltantes': miss_count, 'pct_faltantes': miss_pct})
df_miss = df_miss[df_miss['n_faltantes'] > 0].sort_values('pct_faltantes', ascending=False)

total_vars   = df_raw.shape[1]
n_with_miss  = len(df_miss)
print(f"Dimensiones del dataset      : {df_raw.shape[0]:,} filas × {total_vars} columnas")
print(f"Variables con datos faltantes: {n_with_miss} de {total_vars} ({n_with_miss/total_vars*100:.1f}%)")
print(f"Variables sin datos faltantes: {total_vars - n_with_miss}")
print()
if len(df_miss) > 0:
    print("Variables con valores faltantes (ordenadas por % descendente):")
    print(df_miss.to_string())
else:
    print("El dataset no presenta valores faltantes.")

# ── Missingness por conjunto de características (solo si ya están definidos) ──
_fs_groups_available = 'FEATURE_SETS' in dir()
try:
    _feature_sets_ref = FEATURE_SETS  # will NameError if not defined yet
    print()
    print("Missingness promedio por conjunto de características:")
    print(f"  {'Conjunto':25s}  {'% Miss. prom.':>14s}  {'Vars c/miss':>11s}  {'Total vars':>10s}")
    print("  " + "-"*65)
    for fs_name, features in _feature_sets_ref.items():
        avail = [f for f in features if f in df_raw.columns]
        if not avail:
            continue
        m_prom = df_raw[avail].isnull().mean().mean() * 100
        n_m = sum(1 for f in avail if df_raw[f].isnull().any())
        print(f"  {fs_name:25s}  {m_prom:>14.2f}%  {n_m:>11d}  {len(avail):>10d}")
    _fs_groups_available = True
except NameError:
    print("\n[Resumen por conjunto de características disponible en Sección 4]")
    _fs_groups_available = False

# ── Visualización ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(df_miss) > 0:
    top20 = df_miss.head(20).copy()
    axes[0].barh(top20.index[::-1], top20['pct_faltantes'][::-1],
                 color='#EF5350', edgecolor='white', linewidth=0.5)
    axes[0].axvline(5,  color='#FFA726', linestyle='--', lw=1.2, label='5%')
    axes[0].axvline(20, color='#EF5350', linestyle='--', lw=1.2, label='20%')
    axes[0].set_xlabel('% Valores Faltantes')
    axes[0].set_title('Variables con Mayor Missingness (Top 20)', fontweight='bold')
    axes[0].legend(fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'Sin valores faltantes', ha='center', va='center', fontsize=11)
    axes[0].axis('off')

# Missingness por feature set (only if FEATURE_SETS available)
fs_labels, fs_miss_vals = [], []
try:
    for fs_name, features in FEATURE_SETS.items():
        avail = [f for f in features if f in df_raw.columns]
        if avail:
            fs_labels.append(fs_name)
            fs_miss_vals.append(df_raw[avail].isnull().mean().mean() * 100)
except NameError:
    pass

if fs_labels:
    bars = axes[1].bar(
        fs_labels, fs_miss_vals,
        color=['#42A5F5','#66BB6A','#FFA726','#EF5350','#AB47BC'][:len(fs_labels)],
        edgecolor='white')
    axes[1].set_ylabel('% Missingness Promedio')
    axes[1].set_title('Missingness Promedio por Conjunto de Características', fontweight='bold')
    for bar, val in zip(bars, fs_miss_vals):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
else:
    axes[1].text(0.5, 0.5,
                 'Resumen por conjunto\ndisponible en Sección 4',
                 ha='center', va='center', fontsize=11, color='#777')
    axes[1].set_title('Missingness por Conjunto de Características', fontweight='bold')
    axes[1].axis('off')

plt.suptitle('Análisis de Valores Faltantes — Dataset de Leads Inmobiliarios',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missingness_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n[Nota: Pipeline usa SimpleImputer (mediana/moda) — no se descartan registros por missingness.]")


---
> **Hallazgos — Sección 2 (Análisis de Missingness):** El **77.4% de las variables** (48 de 62) presentan al menos un valor faltante. Las variables con mayor ausencia son etiquetas de enriquecimiento del CRM (`ingreso_crm_label`, 99.64%) y variables demográficas clave: `tiene_hijos` (93.9%), `nucleo_familiar` (91.7%), `edad` (89.9%), `estado_civil` (88.4%), `ingreso_familiar` (83.8%). Este alto missingness no representa error de recolección sino que refleja que dichos datos no son obligatorios al registrar un lead. El conjunto F1 (Base) contiene únicamente variables de canal y precio con missingness mínimo, lo que justifica su inclusión como línea base experimental. La presencia de missingness en variables demográficas explica por qué F2 (Base+Demográficas) no supera dramáticamente a F1 en desempeño.


## Sección 3 — Diseño Experimental

### 3.1 Objetivos

| # | Tarea | Dataset | Variable objetivo | Desbalance aprox. |
|---|-------|---------|-------------------|-------------------|
| 1 | Conversión | 42 464 leads calificados | `target_won` | 56.5:1 |
| 2 | Abandono | 42 464 leads calificados | `target_abandoned` | 56.5:1 |

### 3.2 Algoritmos Evaluados

| Algoritmo | Tipo | Rol |
|-----------|------|-----|
| Regresión Logística | Lineal | Obligatorio (línea base interpretable) |
| Random Forest | Ensamble (Bagging) | Obligatorio |
| Gradient Boosting | Ensamble (Boosting) | Obligatorio |
| XGBoost | Boosting optimizado | Adicional (mayor eficiencia) |
| LightGBM | Boosting basado en hojas | Adicional (muy rápido, manejo nativo de categorías) |

### 3.3 Conjuntos de Características

| ID | Nombre | Descripción |
|----|--------|-------------|
| F1 | Base | Canales de contacto, proyecto, fuente, zona geográfica, atributos del proyecto (m², precio) |
| F2 | Base + Demográficas | F1 + sexo, edad, estado civil, hijos, núcleo familiar |
| F3 | Base + Comerciales | F1 + ingreso familiar, razón de compra, tipo de unidad, opciones visitadas, campaña |
| F4 | Todas las características | F1 + F2 + F3 + variables adicionales disponibles |
| F5 | Todas + Clúster K-Means | F4 + asignación de clúster (solo subconjunto de 42 K leads) |

### 3.4 Estructura de Experimentos

- **Experimento A:** Partición estratificada train/test SIN SMOTE
- **Experimento B:** Misma partición con SMOTE aplicado ÚNICAMENTE sobre datos de entrenamiento
- **Validación cruzada:** Estratificada (5 folds) sobre datos de entrenamiento para los mejores modelos
- **Afinamiento:** Búsqueda aleatoria de hiperparámetros sobre los modelos más prometedores

> ⚠️ **Garantía anti-fuga:** La misma semilla (`SEED=42`) y la misma partición (`train_test_split`) se reutilizan en **todos** los experimentos. SMOTE se aplica **dentro** del pipeline de entrenamiento, nunca antes del split.


## Sección 4 — Preprocesamiento y Definición de Conjuntos de Características

### 4.1 Identificación de Variables por Tipo

Se excluyen del modelado:
- **IDs**: `id_trato`, `person_id`, `user_id`
- **Fuga de datos temporal**: CSI scores y NPS son recopilados *después* de la compra → fuga segura
- **Variable derivada del target**: `` (correlacionada con el target por construcción — se discute)
- **El propio target**: `target_won`, `target_abandoned`
- **Columnas de etiqueta CRM** (`*_label`): >99% nulos, no aportan información



In [ ]:
# ── Variables a excluir siempre ──────────────────────────────────────────
ALWAYS_EXCLUDE = [
    'id_trato', 'person_id', 'user_id',
    'target_won', 'target_abandoned', 'proyecto_fuera_alcance',
    'fuente_dataset',
    # CSI / NPS (post-sale leakage)
    'csi_calificacion_amenidades_label', 'csi_calificacion_apartamento_label',
    'csi_calificacion_proyecto_label',   'csi_comentarios_adicionales_label',
    'csi_conocimiento_asesor_label',     'csi_tiempo_seguimiento_label',
    'nps_recomendacion_proyecto_label',
    # CRM label columns (>99% null)
    'apartamento_no_label', 'apto_referidor_label', 'bodega_asignada_label',
    'cantidad_parqueos_label', 'cuota_deseada_label', 'ingreso_crm_label',
    'modelo_label', 'parqueo_1_label', 'parqueo_2_label', 'parqueo_3_label',
    'proyref_label', 'razon_inversion_label', 'tipo_modelo_label',
    'tipovisita_label', 'torre_referidora_label',
    'ubicacion_bodega_label', 'ubicacion_parqueos_label',
    # Date (would need temporal features – out of scope for this phase)
    'fecha_creacion_trato',
]

# ── Feature sets ──────────────────────────────────────────────────────────
F1_BASE = [
    'canal_de_anuencia', 'canal_de_contacto',
    'proyecto', 'fuente_pt2',
    'zona_proyecto', 'subzona_proyecto',
    'm2_minimo', 'm2_maximo',
    'precio_minimo', 'precio_maximo', 'precio_m2_promedio', 'precio_promedio',
]

F2_DEMO = F1_BASE + [
    'sexo', 'estado_civil', 'edad', 'rango_edad_modelo',
    'tiene_hijos', 'nucleo_familiar',
]

F3_COMMERCIAL = F1_BASE + [
    'ingreso_familiar', 'razon_de_compra', 'tipo_unidad',
    'opciones_visitadas',
    'campania_plataforma', 'campania_tipo', 'campania_audiencia',
    'campania',
]

F4_ALL = list(set(
    F2_DEMO + F3_COMMERCIAL + [
        'lugar_trabajo', 'lugar_vivienda', 'sector_referidor',
        'municipio_proyecto',
        'participantes_en_credito',
    ]
))

F5_CLUSTER = F4_ALL + ['cluster']   # only available in the 42 K K-Means subset

FEATURE_SETS = {
    'F1_Base':          F1_BASE,
    'F2_Base+Demo':     F2_DEMO,
    'F3_Base+Comercial':F3_COMMERCIAL,
    'F4_Todas':         F4_ALL,
}

FEATURE_SETS_CLUSTER = {
    'F5_Todas+Cluster': F5_CLUSTER,
}

print("Conjuntos de características definidos:")
for k, v in FEATURE_SETS.items():
    print(f"  {k:25s}: {len(v)} variables")
for k, v in FEATURE_SETS_CLUSTER.items():
    print(f"  {k:25s}: {len(v)} variables  ← subconjunto K-Means")


In [ ]:
# ── Identify categorical vs numeric columns per feature set ──────────────

def identify_col_types(df, features):
    """Return (cat_cols, num_cols) for the given feature list."""
    available = [f for f in features if f in df.columns]
    cat_cols = [c for c in available if df[c].dtype == 'object']
    num_cols = [c for c in available if df[c].dtype in ('float64', 'int64')]
    return cat_cols, num_cols

# Verify feature sets against available columns
for name, feats in {**FEATURE_SETS, **FEATURE_SETS_CLUSTER}.items():
    missing = [f for f in feats if f not in df_raw.columns and f != 'cluster']
    if missing:
        print(f"⚠  {name}: variables no encontradas → {missing}")
    else:
        cat, num = identify_col_types(df_raw, feats)
        print(f"✓ {name:25s}: {len(cat)} cat + {len(num)} num")


## Sección 5 — División Estratificada Train / Test

Se crea **una sola** partición train/test (80%/20%) para el dataset de conversión y una partición equivalente para el dataset de abandono. Ambas mantienen la proporción de positivos mediante estratificación. Esta misma partición se reutiliza en **todos** los experimentos subsiguientes para garantizar comparabilidad.

> La división se realiza **antes** de cualquier preprocesamiento o SMOTE.


In [ ]:
# ── Task 1: Conversion — qualified dataset (42 K) ───────────────────────
y_conv = df_raw['target_won'].values
X_conv = df_raw.copy()

idx_all = np.arange(len(df_raw))
idx_train_conv, idx_test_conv = train_test_split(
    idx_all, test_size=0.20, random_state=SEED, stratify=y_conv
)

y_train_conv = y_conv[idx_train_conv]
y_test_conv  = y_conv[idx_test_conv]

print("═══ TAREA 1: Conversión ═══")
print(f"  Train : {len(idx_train_conv):,} samples  "
      f"| positivos: {y_train_conv.sum():,} ({y_train_conv.mean()*100:.2f}%)")
print(f"  Test  : {len(idx_test_conv):,} samples  "
      f"| positivos: {y_test_conv.sum():,} ({y_test_conv.mean()*100:.2f}%)")
print(f"  Desbalance train (neg/pos): {(1-y_train_conv).sum()/y_train_conv.sum():.1f}:1")


In [ ]:
# -- Task 2: Abandonment -- qualified dataset (42 K), same split as Task 1 --
# Abandonment target: any lead that did NOT convert
# Note: proyecto_fuera_alcance is excluded as irrelevant.
# Both tasks share the same dataset and the same train/test split.
y_aband = (1 - y_conv)   # 1 = abandoned, 0 = converted

# Reuse the SAME split from Task 1 (guarantees fair comparison)
idx_train_aband = idx_train_conv
idx_test_aband  = idx_test_conv
y_train_aband   = y_aband[idx_train_aband]
y_test_aband    = y_aband[idx_test_aband]

print("=== TAREA 2: Abandono ===")
print(f"  Dataset: CALIFICADO (mismo que Tarea 1)")
print(f"  Train : {len(idx_train_aband):,} samples  "
      f"| abandonaron: {y_train_aband.sum():,} ({y_train_aband.mean()*100:.2f}%)")
print(f"  Test  : {len(idx_test_aband):,} samples  "
      f"| abandonaron: {y_test_aband.sum():,} ({y_test_aband.mean()*100:.2f}%)")
print()
print("Nota metodologica: con una tasa de abandono de ~98.8%,")
print("el modelo de abandono identifica los escasos leads con alta probabilidad")
print("de conversion (clase negativa) — es el complemento exacto del modelo de conversion.")
print("El valor de negocio esta en priorizar contacto con leads MENOS propensos a abandonar.")


In [ ]:
# ── K-Means subset split (F5) ─────────────────────────────────────────────
# Merge cluster labels into raw dataset using id_trato
df_with_cluster = df_raw.merge(
    df_kmeans[['id_trato', 'cluster']],
    on='id_trato', how='inner'
)
y_cluster = df_with_cluster['target_won'].values

idx_all_cl = np.arange(len(df_with_cluster))
idx_train_cl, idx_test_cl = train_test_split(
    idx_all_cl, test_size=0.20, random_state=SEED, stratify=y_cluster
)

y_train_cl = y_cluster[idx_train_cl]
y_test_cl  = y_cluster[idx_test_cl]

print("═══ TAREA 1: Conversión — Subconjunto K-Means (F5) ═══")
print(f"  Total : {len(df_with_cluster):,}")
print(f"  Train : {len(idx_train_cl):,} | pos: {y_train_cl.sum():,} ({y_train_cl.mean()*100:.2f}%)")
print(f"  Test  : {len(idx_test_cl):,}  | pos: {y_test_cl.sum():,} ({y_test_cl.mean()*100:.2f}%)")


## Sección 6 — Funciones Auxiliares

Se definen las funciones reutilizables para: construcción de pipelines, cálculo de métricas técnicas, métricas de negocio (Lift, Gain, Precision/Recall@K) y visualización.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.1  Pipeline builders
# ════════════════════════════════════════════════════════════════════════

def build_preprocessor(cat_cols, num_cols, ordinal=True):
    """
    Build a ColumnTransformer that:
    - Imputes + OrdinalEncodes categoricals  (ordinal=True → for tree models)
    - Imputes + StandardScales numerics
    If ordinal=False, uses OneHotEncoder (for LR).
    """
    if ordinal:
        cat_pipe = Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='_missing_')),
            ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ])
    else:
        cat_pipe = Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='_missing_')),
            ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ])

    num_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
    ])

    steps = []
    if cat_cols:
        steps.append(('cat', cat_pipe, cat_cols))
    if num_cols:
        steps.append(('num', num_pipe, num_cols))

    return ColumnTransformer(steps, remainder='drop')


def build_pipeline(estimator, cat_cols, num_cols, use_smote=False,
                   smote_strategy=0.10, ordinal=True):
    """
    Build a full sklearn/imblearn pipeline:
      preprocessor → [SMOTE] → estimator
    SMOTE is applied ONLY when use_smote=True, and ONLY within this pipeline
    (guaranteeing it never touches test data).
    """
    prep = build_preprocessor(cat_cols, num_cols, ordinal=ordinal)
    if use_smote:
        from imblearn.over_sampling import SMOTE
        smote = SMOTE(sampling_strategy=smote_strategy,
                      random_state=SEED, k_neighbors=5)
        pipeline = ImbPipeline([
            ('prep',  prep),
            ('smote', smote),
            ('clf',   estimator),
        ])
    else:
        pipeline = Pipeline([
            ('prep', prep),
            ('clf',  estimator),
        ])
    return pipeline


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.2  Metrics computation
# ════════════════════════════════════════════════════════════════════════

def compute_metrics(y_true, y_pred, y_prob):
    """Return a dict of all required classification metrics."""
    pr, rc, thresholds = precision_recall_curve(y_true, y_prob)
    return {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
        'roc_auc'  : roc_auc_score(y_true, y_prob),
        'pr_auc'   : average_precision_score(y_true, y_prob),
        'y_prob'   : y_prob,
        'y_pred'   : y_pred,
    }


def evaluate_pipeline(pipeline, X_train, y_train, X_test, y_test,
                       features, threshold=0.5):
    """Fit on train, evaluate on test. Returns metric dict."""
    t0 = time.time()
    pipeline.fit(X_train[features], y_train)
    elapsed = time.time() - t0

    y_prob = pipeline.predict_proba(X_test[features])[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics['train_time'] = elapsed
    return pipeline, metrics


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.3  Business metrics: Lift, Gain, Precision/Recall @ K%
# ════════════════════════════════════════════════════════════════════════

def business_metrics_at_k(y_true, y_prob, k_list=(0.05, 0.10, 0.20)):
    """
    Compute Precision, Recall and Lift at top-K% of predicted scores.
    Also returns cumulative gain data for the gain chart.
    """
    n = len(y_true)
    total_pos = y_true.sum()
    # Sort by descending predicted probability
    order = np.argsort(y_prob)[::-1]
    y_sorted = y_true[order]
    cum_pos   = np.cumsum(y_sorted)

    results = {}
    for k in k_list:
        top_n  = max(1, int(np.ceil(n * k)))
        tp     = cum_pos[top_n - 1]
        prec_k = tp / top_n
        rec_k  = tp / total_pos if total_pos > 0 else 0
        random_expected = total_pos * k       # expected positives by random
        lift_k = tp / random_expected if random_expected > 0 else 0
        key = f"top{int(k*100)}pct"
        results[key] = {
            'n_selected' : top_n,
            'tp'         : int(tp),
            'precision'  : prec_k,
            'recall'     : rec_k,
            'lift'       : lift_k,
        }

    # Cumulative gain curve data (for chart)
    pct_selected = np.arange(1, n + 1) / n
    cum_gain     = cum_pos / total_pos
    return results, pct_selected, cum_gain


def print_business_metrics(name, bm):
    print(f"\n{'─'*55}")
    print(f"  Métricas de Negocio — {name}")
    print(f"{'─'*55}")
    print(f"  {'Corte':8s}  {'Selec.':>7}  {'TP':>6}  {'Precis.':>8}  {'Recall':>7}  {'Lift':>6}")
    for k_key, v in bm.items():
        print(f"  {k_key:8s}  {v['n_selected']:>7,}  {v['tp']:>6,}  "
              f"{v['precision']:>8.3f}  {v['recall']:>7.3f}  {v['lift']:>6.2f}x")


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.4  Plotting utilities
# ════════════════════════════════════════════════════════════════════════

def plot_confusion_matrix(y_true, y_pred, ax, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0','Pred 1'],
                yticklabels=['Real 0','Real 1'])
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel('Real'); ax.set_xlabel('Predicho')


def plot_roc_pr(y_true, y_prob, ax_roc, ax_pr, label='Model'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_roc = roc_auc_score(y_true, y_prob)
    ax_roc.plot(fpr, tpr, lw=2, label=f'{label} (AUC={auc_roc:.3f})')
    ax_roc.plot([0,1],[0,1],'k--', lw=0.8, alpha=0.5)
    ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
    ax_roc.set_title('Curva ROC'); ax_roc.legend(fontsize=8)

    pr, rc, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    ax_pr.plot(rc, pr, lw=2, label=f'{label} (AP={ap:.3f})')
    baseline = y_true.mean()
    ax_pr.axhline(baseline, linestyle='--', color='gray', lw=0.8,
                  label=f'Aleatorio ({baseline:.3f})')
    ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
    ax_pr.set_title('Curva Precision-Recall'); ax_pr.legend(fontsize=8)


def plot_gain_chart(y_true, y_prob, ax, label='Model'):
    n = len(y_true)
    order = np.argsort(y_prob)[::-1]
    cum_pos = np.cumsum(y_true[order]) / y_true.sum()
    pct = np.arange(1, n+1) / n
    ax.plot(pct, cum_pos, lw=2, label=label)
    ax.plot([0,1],[0,1],'k--', lw=0.8, alpha=0.5, label='Aleatorio')
    ax.plot([0, y_true.mean(), 1],[0,1,1],'g:', lw=0.8, alpha=0.6, label='Perfecto')
    ax.set_xlabel('% Leads contactados'); ax.set_ylabel('% Conversiones capturadas')
    ax.set_title('Curva de Ganancia Acumulada')
    ax.legend(fontsize=8)
    ax.set_xlim(0,1); ax.set_ylim(0,1.05)


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.5  Experiment runner
# ════════════════════════════════════════════════════════════════════════
#
# Algoritmos implementados:
#   LR   — Regresión Logística (línea base interpretable, OBLIGATORIO)
#   RF   — Random Forest, n_estimators=60, n_jobs=-1 (OBLIGATORIO)
#   GBM  — HistGradientBoostingClassifier (Gradient Boosting histograma,
#           equivalente computacionalmente eficiente de LightGBM/XGBoost,
#           OBLIGATORIO bajo criterio de "Gradient Boosting")
#   EXT  — ExtraTreesClassifier (alternativa adicional mencionada en
#           el diseño experimental, equivalente a XGBoost)
#
# NOTA: sklearn.GradientBoostingClassifier (árbol por árbol secuencial)
#   fue evaluado pero excluido del loop principal por inviabilidad
#   computacional: reducido al trabajar con ~42K filas.
#   HGB es su sucesor oficial en sklearn (misma familia, ~50x más rápido).
#
# NOTA: XGBoost y LightGBM requieren libomp ARM64 no disponible en el
#   sistema. Se documentan como experimentos fallidos.

ALGORITHMS = {
    'LR' : lambda: LogisticRegression(
        class_weight='balanced', max_iter=300,
        random_state=SEED, solver='lbfgs'),
    'RF' : lambda: RandomForestClassifier(
        n_estimators=60, class_weight='balanced',
        n_jobs=-1, random_state=SEED),
    'GBM': lambda: HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, max_leaf_nodes=63,
        class_weight='balanced', random_state=SEED),
    'EXT': lambda: ExtraTreesClassifier(
        n_estimators=60, class_weight='balanced',
        n_jobs=-1, random_state=SEED),
}


def run_experiment(X_df, y_arr, idx_train, idx_test,
                   feature_sets, task_name,
                   use_smote=False, smote_strategy=0.10,
                   algorithms=None):
    """
    Run all algorithm x feature_set combinations for one experiment.
    Returns a list of result dicts.
    """
    if algorithms is None:
        algorithms = ALGORITHMS

    X_train = X_df.iloc[idx_train]
    X_test  = X_df.iloc[idx_test]
    y_train = y_arr[idx_train]
    y_test  = y_arr[idx_test]

    results = []

    for fs_name, features in feature_sets.items():
        available = [f for f in features if f in X_df.columns]
        cat_cols, num_cols = identify_col_types(X_df, available)

        for alg_name, alg_factory in algorithms.items():
            est = alg_factory()
            use_ordinal = (alg_name != 'LR')
            pipe = build_pipeline(est, cat_cols, num_cols,
                                  use_smote=use_smote,
                                  smote_strategy=smote_strategy,
                                  ordinal=use_ordinal)
            try:
                pipe, m = evaluate_pipeline(pipe, X_train, y_train,
                                            X_test,  y_test, available)
                bm, pct_sel, cum_gain = business_metrics_at_k(
                    y_test, m['y_prob'])
                row = {
                    'task'       : task_name,
                    'experiment' : 'B_SMOTE' if use_smote else 'A_NoSMOTE',
                    'feature_set': fs_name,
                    'algorithm'  : alg_name,
                    'accuracy'   : m['accuracy'],
                    'precision'  : m['precision'],
                    'recall'     : m['recall'],
                    'f1'         : m['f1'],
                    'roc_auc'    : m['roc_auc'],
                    'pr_auc'     : m['pr_auc'],
                    'lift_5pct'  : bm['top5pct']['lift'],
                    'lift_10pct' : bm['top10pct']['lift'],
                    'lift_20pct' : bm['top20pct']['lift'],
                    'rec_5pct'   : bm['top5pct']['recall'],
                    'rec_10pct'  : bm['top10pct']['recall'],
                    'rec_20pct'  : bm['top20pct']['recall'],
                    'prec_5pct'  : bm['top5pct']['precision'],
                    'prec_10pct' : bm['top10pct']['precision'],
                    'prec_20pct' : bm['top20pct']['precision'],
                    'train_time' : m['train_time'],
                    'pipeline'   : pipe,
                    'y_prob_test': m['y_prob'],
                    'y_true_test': y_test,
                    'bm'         : bm,
                    'pct_sel'    : pct_sel,
                    'cum_gain'   : cum_gain,
                }
                print(f"  ok {alg_name:4s} | {fs_name:22s} | "
                      f"ROC={m['roc_auc']:.4f}  PR={m['pr_auc']:.4f}  "
                      f"Lift@10={bm['top10pct']['lift']:.2f}x  "
                      f"t={m['train_time']:.1f}s")
            except Exception as e:
                import traceback
                print(f"  ERR {alg_name:4s} | {fs_name:22s} | {e}")
                row = {'task':task_name,
                       'experiment':'B_SMOTE' if use_smote else 'A_NoSMOTE',
                       'feature_set':fs_name,'algorithm':alg_name,'error':str(e)}
            results.append(row)
    return results


## Sección 7 — TAREA 1: Predicción de Conversión de Leads

### Contexto de Negocio
Identificar qué leads tienen mayor probabilidad de convertir permite priorizar el esfuerzo del equipo comercial. Un modelo con buen Lift@10% significa que, contactando solo el 10% de los leads ordenados por probabilidad predicha, el equipo puede capturar un múltiplo de las conversiones que capturaría seleccionando aleatoriamente.

### Desafío Técnico
El dataset presenta un desbalance severo (1.18% positivos, ratio 83:1). Se compararán dos enfoques:
- **Experimento A:** Sin corrección de desbalance (se usa `class_weight='balanced'` en los clasificadores que lo soportan)
- **Experimento B:** SMOTE aplicado exclusivamente en datos de entrenamiento con `sampling_strategy=0.10` (se lleva la clase minoritaria al 10% del conjunto, no al 50%, porque el desbalance extremo hace que el SMOTE al 50% sea computacionalmente prohibitivo y puede degradar la calidad del modelo)


### 7.1 — Experimento A: Sin SMOTE (Feature Sets F1–F4, Dataset Calificado)

Se entrenan los cinco algoritmos sobre los cuatro conjuntos de características principales (sin clúster) con el dataset de 42,464 leads calificados.


In [ ]:
print("=" * 65)
print("  EXPERIMENTO A — Sin SMOTE | Tarea 1: Conversión")
print("=" * 65)
results_conv_A = run_experiment(
    X_df        = df_raw,
    y_arr       = y_conv,
    idx_train   = idx_train_conv,
    idx_test    = idx_test_conv,
    feature_sets= FEATURE_SETS,
    task_name   = 'Conversion',
    use_smote   = False,
)
print(f"\nTotal experimentos completados: {len([r for r in results_conv_A if 'error' not in r])}")


### 7.2 — Experimento B: Con SMOTE (Feature Sets F1–F4)

SMOTE se aplica **dentro del pipeline**, nunca antes del split. Se usa `sampling_strategy=0.10` para llevar la clase minoritaria al 10% sin generar millones de muestras sintéticas.


In [ ]:
print("=" * 65)
print("  EXPERIMENTO B — Con SMOTE | Tarea 1: Conversión")
print("=" * 65)
results_conv_B = run_experiment(
    X_df        = df_raw,
    y_arr       = y_conv,
    idx_train   = idx_train_conv,
    idx_test    = idx_test_conv,
    feature_sets= FEATURE_SETS,
    task_name   = 'Conversion',
    use_smote   = True,
    smote_strategy = 0.10,
)
print(f"\nTotal experimentos completados: {len([r for r in results_conv_B if 'error' not in r])}")


### 7.3 — Experimento con Clúster K-Means (F5, Subconjunto 42 K)

Se utiliza el subconjunto de leads que pasaron por el análisis K-Means. La asignación de clúster es un atributo derivado de variables comportamentales y demográficas, disponible en tiempo de operación (no introduce fuga de datos).

> **Nota anti-fuga:** El clúster fue asignado mediante K-Means *no supervisado* (sin utilizar `target_won`). Se verificó que la asignación no fue realizada con información post-venta.


In [ ]:
print("=" * 65)
print("  EXPERIMENTO A — F5 con Clúster K-Means (Sin SMOTE)")
print("=" * 65)
results_conv_cluster_A = run_experiment(
    X_df        = df_with_cluster,
    y_arr       = y_cluster,
    idx_train   = idx_train_cl,
    idx_test    = idx_test_cl,
    feature_sets= FEATURE_SETS_CLUSTER,
    task_name   = 'Conversion',
    use_smote   = False,
)

print("\n" + "=" * 65)
print("  EXPERIMENTO B — F5 con Clúster K-Means (Con SMOTE)")
print("=" * 65)
results_conv_cluster_B = run_experiment(
    X_df        = df_with_cluster,
    y_arr       = y_cluster,
    idx_train   = idx_train_cl,
    idx_test    = idx_test_cl,
    feature_sets= FEATURE_SETS_CLUSTER,
    task_name   = 'Conversion',
    use_smote   = True,
    smote_strategy = 0.10,
)


### 7.4 — Tabla Comparativa: Tarea 1 (Conversión)


In [ ]:
# Build comparison table
all_results_conv = results_conv_A + results_conv_B + results_conv_cluster_A + results_conv_cluster_B
df_results_conv = pd.DataFrame([
    {k: v for k, v in r.items()
     if k not in ('pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain')}
    for r in all_results_conv if 'error' not in r
])

# Display sorted by PR-AUC descending
display_cols = ['experiment','feature_set','algorithm',
                'roc_auc','pr_auc','recall','precision','f1',
                'lift_5pct','lift_10pct','lift_20pct',
                'rec_5pct','rec_10pct','rec_20pct']
df_display_conv = df_results_conv[display_cols].sort_values('pr_auc', ascending=False)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
print("TABLA COMPARATIVA — TAREA 1: CONVERSIÓN")
print(df_display_conv.to_string(index=False))


In [ ]:
# Save results to CSV
df_results_conv.drop(columns=['pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain'],
                     errors='ignore').to_csv(
    RESULTS_DIR / 'task1_conversion_all_experiments.csv', index=False
)
print("✓ Resultados guardados en results/task1_conversion_all_experiments.csv")


### 7.5 — Visualización: Comparación por Algoritmo y Conjunto de Características


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

metrics_to_plot = ['roc_auc', 'pr_auc', 'lift_10pct', 'recall']
metric_labels   = ['ROC-AUC', 'PR-AUC', 'Lift @ 10%', 'Recall']

for ax, metric, label in zip(axes, metrics_to_plot, metric_labels):
    pivot = df_results_conv.pivot_table(
        index='algorithm', columns=['experiment','feature_set'],
        values=metric, aggfunc='mean')
    pivot.plot(kind='bar', ax=ax, legend=False, rot=0)
    ax.set_title(f'Tarea 1 — {label} por Algoritmo', fontweight='bold')
    ax.set_ylabel(label)
    ax.set_xlabel('')

# Single legend for all subplots
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.05), fontsize=7)
plt.suptitle('Comparación de Modelos — Tarea 1: Conversión de Leads',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task1_comparison_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figura guardada")


### 7.6 — Análisis de Sobreajuste (Overfitting)

Se compara el rendimiento en train vs. test para detectar sobreajuste. Se evalúan los mejores modelos de cada familia.


In [ ]:
# Compute train metrics for top models (best 2 per algorithm by PR-AUC)
best_per_alg_conv = (
    df_results_conv.sort_values('pr_auc', ascending=False)
    .groupby('algorithm').head(1)
    .reset_index(drop=True)
)
print("Top modelo por algoritmo (Tarea 1 — Conversión):")
print(best_per_alg_conv[['algorithm','experiment','feature_set',
                          'roc_auc','pr_auc','recall','lift_10pct']].to_string(index=False))

# Compute overfitting indicators
overfitting_rows = []
for _, row in best_per_alg_conv.iterrows():
    # Match pipeline from results list
    match = next(
        (r for r in all_results_conv
         if r.get('algorithm')==row['algorithm']
         and r.get('experiment')==row['experiment']
         and r.get('feature_set')==row['feature_set']
         and 'error' not in r), None)
    if match is None:
        continue
    pipe = match['pipeline']
    fs   = FEATURE_SETS.get(row['feature_set'], FEATURE_SETS_CLUSTER.get(row['feature_set'], []))
    available = [f for f in fs if f in df_raw.columns or f == 'cluster']

    # Choose correct dataframe
    if 'cluster' in fs:
        X_src, idx_tr, y_tr = df_with_cluster, idx_train_cl, y_train_cl
    else:
        X_src, idx_tr, y_tr = df_raw, idx_train_conv, y_train_conv

    try:
        y_prob_tr = pipe.predict_proba(X_src.iloc[idx_tr][available])[:, 1]
        auc_train = roc_auc_score(y_tr, y_prob_tr)
        ap_train  = average_precision_score(y_tr, y_prob_tr)
    except Exception as e:
        auc_train, ap_train = None, None

    overfitting_rows.append({
        'algorithm'  : row['algorithm'],
        'feature_set': row['feature_set'],
        'roc_auc_train': auc_train,
        'roc_auc_test' : row['roc_auc'],
        'pr_auc_train' : ap_train,
        'pr_auc_test'  : row['pr_auc'],
        'gap_roc_auc'  : (auc_train - row['roc_auc']) if auc_train else None,
        'gap_pr_auc'   : (ap_train  - row['pr_auc'])  if ap_train  else None,
    })

df_overfit_conv = pd.DataFrame(overfitting_rows)
print("\nAnálisis de Sobreajuste (train vs test):")
print(df_overfit_conv.to_string(index=False))


### 7.7 — Curvas ROC y Precision-Recall de los Mejores Modelos


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax_roc, ax_pr = axes[0, 0], axes[0, 1]
ax_gain, ax_lift = axes[1, 0], axes[1, 1]

colors = ['#2196F3','#FF5722','#4CAF50','#9C27B0','#FF9800']

for idx_m, (_, row) in enumerate(best_per_alg_conv.iterrows()):
    match = next(
        (r for r in all_results_conv
         if r.get('algorithm')==row['algorithm']
         and r.get('experiment')==row['experiment']
         and r.get('feature_set')==row['feature_set']
         and 'error' not in r), None)
    if match is None:
        continue

    y_t = match['y_true_test']
    y_p = match['y_prob_test']
    col = colors[idx_m % len(colors)]
    lbl = f"{row['algorithm']} {row['feature_set']}"

    fpr, tpr, _ = roc_curve(y_t, y_p)
    auc = roc_auc_score(y_t, y_p)
    ax_roc.plot(fpr, tpr, lw=1.8, color=col, label=f'{lbl} ({auc:.3f})')

    pr_v, rc_v, _ = precision_recall_curve(y_t, y_p)
    ap = average_precision_score(y_t, y_p)
    ax_pr.plot(rc_v, pr_v, lw=1.8, color=col, label=f'{lbl} ({ap:.3f})')

    # Gain
    n = len(y_t)
    order = np.argsort(y_p)[::-1]
    cum_g = np.cumsum(y_t[order]) / y_t.sum()
    pct   = np.arange(1, n+1) / n
    ax_gain.plot(pct, cum_g, lw=1.8, color=col, label=lbl)

    # Lift at different cutoffs
    lifts = [row['lift_5pct'], row['lift_10pct'], row['lift_20pct']]
    ax_lift.bar(
        [i + idx_m*0.15 for i in range(3)],
        lifts, width=0.13, color=col, label=row['algorithm'], alpha=0.85
    )

ax_roc.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.5)
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.set_title('Curva ROC — Mejores Modelos (Conversión)', fontweight='bold')
ax_roc.legend(fontsize=7)

baseline = y_test_conv.mean()
ax_pr.axhline(baseline, linestyle='--', color='gray', lw=0.8)
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('Curva Precision-Recall (Conversión)', fontweight='bold')
ax_pr.legend(fontsize=7)

ax_gain.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.5,label='Aleatorio')
ax_gain.set_xlabel('% Leads contactados'); ax_gain.set_ylabel('% Conversiones capturadas')
ax_gain.set_title('Curva de Ganancia Acumulada (Conversión)', fontweight='bold')
ax_gain.legend(fontsize=7)

ax_lift.set_xticks([0.3, 1.3, 2.3])
ax_lift.set_xticklabels(['Lift@5%', 'Lift@10%', 'Lift@20%'])
ax_lift.set_ylabel('Lift'); ax_lift.set_title('Lift por Corte (Conversión)', fontweight='bold')
ax_lift.legend(fontsize=7)
ax_lift.axhline(1, color='gray', linestyle='--', lw=0.8)

plt.suptitle('Evaluación Visual — Tarea 1: Conversión', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task1_roc_pr_gain_lift.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.8 - Diagnostico: Variables de Proyecto como Identificadores Indirectos

El dataset contiene **cinco proyectos** y todos los leads de un mismo proyecto
comparten exactamente los mismos valores para variables como `m2_minimo`,
`precio_promedio`, `zona_proyecto`, etc. Esto implica que esas variables
funcionan como **identificadores indirectos del proyecto**.

El objetivo de investigacion **no** es predecir conversion en estos cinco
proyectos especificos, sino construir un modelo que **generalice a proyectos
futuros**. Por ello se distingue si el desempeno proviene de:

1. Relaciones reales entre caracteristicas del lead y la conversion
2. La identidad especifica de uno de los cinco proyectos actuales

**Metodologia:**
- Confirmar cuales variables tienen varianza cero dentro de cada proyecto
- Aplicar **Leave-One-Project-Out CV (LOGO-CV)**: entrenar en 4 proyectos,
  evaluar en el 5to; repetir 5 veces
- Comparar LOGO-CV con/sin variables de proyecto
- Construir variables relativas (asequibilidad) que capturen la relacion
  lead-proyecto de forma generalizable


In [ ]:
# === 7.8.1  Identificar variables de proyecto + crear variables relativas ===
PROYECTO_COL = 'proyecto'
assert PROYECTO_COL in df_raw.columns, f"Columna '{PROYECTO_COL}' no encontrada"

proyectos_unicos = sorted(df_raw[PROYECTO_COL].dropna().unique(), key=str)
print(f"Proyectos unicos ({len(proyectos_unicos)}): {proyectos_unicos}")
print()

CANDIDATE_PV = [
    'm2_minimo', 'm2_maximo',
    'precio_minimo', 'precio_maximo', 'precio_m2_promedio', 'precio_promedio',
    'zona_proyecto', 'subzona_proyecto', 'municipio_proyecto',
    PROYECTO_COL,
]
CANDIDATE_PV = [v for v in CANDIDATE_PV if v in df_raw.columns]

CONFIRMED_PROJECT_VARS = []
print('Analisis de varianza dentro de proyecto:')
for v in CANDIDATE_PV:
    per_proj = df_raw.groupby(PROYECTO_COL)[v].nunique(dropna=True)
    max_u = per_proj.max()
    if max_u <= 1:
        CONFIRMED_PROJECT_VARS.append(v)
        print(f'  [ID_PROYECTO] {v} — valor unico por proyecto (identificador)')
    else:
        print(f'  [VARIABLE   ] {v} — varia dentro de proyectos (max nunique={max_u})')

print(f'\nVariables que codifican identidad del proyecto ({len(CONFIRMED_PROJECT_VARS)}):')
print(f'  {CONFIRMED_PROJECT_VARS}')

print()
for fs_name, fs_list in FEATURE_SETS.items():
    proj_in_fs = [v for v in CONFIRMED_PROJECT_VARS if v in fs_list]
    if proj_in_fs:
        print(f'  {fs_name}: {len(proj_in_fs)} vars de proyecto -> {proj_in_fs}')

# ─── Convertir columnas numericas que pueden venir como string en el CSV ───
NUM_COLS_TO_CAST = [
    'm2_minimo', 'm2_maximo', 'precio_minimo', 'precio_maximo',
    'precio_promedio', 'precio_m2_promedio',
    'ingreso_familiar', 'ingreso', 'nucleo_familiar',
    'edad', 'participantes_en_credito', 'opciones_visitadas',
]
cast_report = []
for col in NUM_COLS_TO_CAST:
    if col not in df_raw.columns:
        continue
    if not pd.api.types.is_numeric_dtype(df_raw[col]):
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
        cast_report.append(col)
if cast_report:
    print(f'\nColumnas convertidas a numerico (eran object/string): {cast_report}')
else:
    print('\nTodas las columnas numericas ya tienen dtype numerico.')

# ─── Crear variables relativas (generalizables a nuevos proyectos) ──────────
print()
print('Creando variables de asequibilidad...')
ENGINEERED_VARS = []

INGRESO_COL = next((c for c in ['ingreso_familiar', 'ingreso'] if c in df_raw.columns), None)

if INGRESO_COL and 'precio_promedio' in df_raw.columns:
    precio_ok  = df_raw['precio_promedio'].notna() & (df_raw['precio_promedio'] > 0)
    ingreso_ok = df_raw[INGRESO_COL].notna()
    df_raw['asequibilidad_indice'] = np.where(
        precio_ok & ingreso_ok,
        df_raw[INGRESO_COL] / df_raw['precio_promedio'], np.nan)
    ENGINEERED_VARS.append('asequibilidad_indice')
    pct = df_raw['asequibilidad_indice'].notna().mean() * 100
    med = df_raw['asequibilidad_indice'].median()
    print(f'  asequibilidad_indice = {INGRESO_COL}/precio_promedio  '
          f'({pct:.1f}% no nulos, mediana={med:.4f})')

if 'm2_minimo' in df_raw.columns and 'nucleo_familiar' in df_raw.columns:
    m2_ok     = df_raw['m2_minimo'].notna() & (df_raw['m2_minimo'] > 0)
    nucleo_ok = df_raw['nucleo_familiar'].notna()
    df_raw['m2_por_persona'] = np.where(
        m2_ok & nucleo_ok,
        df_raw['m2_minimo'] / df_raw['nucleo_familiar'].clip(lower=1), np.nan)
    ENGINEERED_VARS.append('m2_por_persona')
    pct = df_raw['m2_por_persona'].notna().mean() * 100
    med = df_raw['m2_por_persona'].median()
    print(f'  m2_por_persona = m2_minimo/nucleo_familiar  '
          f'({pct:.1f}% no nulos, mediana={med:.2f})')

if INGRESO_COL and 'm2_minimo' in df_raw.columns:
    m2_ok     = df_raw['m2_minimo'].notna() & (df_raw['m2_minimo'] > 0)
    ingreso_ok = df_raw[INGRESO_COL].notna()
    df_raw['ingreso_por_m2'] = np.where(
        ingreso_ok & m2_ok,
        df_raw[INGRESO_COL] / df_raw['m2_minimo'], np.nan)
    ENGINEERED_VARS.append('ingreso_por_m2')
    pct = df_raw['ingreso_por_m2'].notna().mean() * 100
    med = df_raw['ingreso_por_m2'].median()
    print(f'  ingreso_por_m2 = {INGRESO_COL}/m2_minimo  '
          f'({pct:.1f}% no nulos, mediana={med:.1f})')

if not ENGINEERED_VARS:
    print('  ADVERTENCIA: No se crearon variables de asequibilidad.')
    ingreso_cols = [c for c in df_raw.columns if 'ingreso' in c.lower()]
    print(f'  Columnas con "ingreso" en df_raw: {ingreso_cols}')

# ─── Definir F6 (lead-only) y F7 (asequibilidad) ─────────────────────────────
f4_key = next((k for k in FEATURE_SETS if k.upper().startswith('F4')), None)
F4_ALL_VARS = FEATURE_SETS.get(f4_key, []) if f4_key else []

F6_LEAD_ONLY     = [f for f in F4_ALL_VARS if f not in CONFIRMED_PROJECT_VARS]
F7_ASEQUIBILIDAD = F6_LEAD_ONLY + [v for v in ENGINEERED_VARS if v not in F6_LEAD_ONLY]

print()
print(f'F4_Todas:         {len(F4_ALL_VARS)} variables')
print(f'F6_Lead_Solo:     {len(F6_LEAD_ONLY)} variables  (F4 sin vars de proyecto)')
elim = [v for v in F4_ALL_VARS if v in CONFIRMED_PROJECT_VARS]
print(f'  Eliminadas de F4: {elim}')
print(f'F7_Asequibilidad: {len(F7_ASEQUIBILIDAD)} variables  (F6 + asequibilidad)')
if ENGINEERED_VARS:
    print(f'  Variables agregadas: {ENGINEERED_VARS}')
else:
    print('  NOTA: F7 == F6 (no se crearon vars de asequibilidad)')


In [ ]:
# === 7.8.2  Leave-One-Project-Out CV (LOGO-CV) ===
# Pregunta: cuanto cae el PR-AUC cuando el modelo NO conoce el proyecto de prueba?
# Se usa el dataset CALIFICADO completo (evaluacion adicional al split principal)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import average_precision_score as aps

best_r_all = max(
    [r for r in all_results_conv if 'cluster' not in r.get('feature_set','').lower()],
    key=lambda r: r.get('pr_auc', 0)
)
alg_diag = best_r_all['algorithm']
fs_diag  = best_r_all['feature_set']
print(f'Algoritmo diagnostico: {alg_diag} (mejor PR-AUC hold-out: {best_r_all["pr_auc"]:.4f})')
print(f'Feature set original : {fs_diag}')
print()

logo_configs = {
    'Original (con vars proyecto)': FEATURE_SETS.get(fs_diag, []),
    'F6_Lead_Solo':                  F6_LEAD_ONLY,
    'F7_Asequibilidad':              F7_ASEQUIBILIDAD,
}

logo_scores = {k: [] for k in logo_configs}
grupos_all  = df_raw[PROYECTO_COL].values
y_all_arr   = np.asarray(y_conv)
logo_cv     = LeaveOneGroupOut()

for fold_i, (tr_logo, te_logo) in enumerate(
        logo_cv.split(df_raw, y_all_arr, grupos_all)):
    proy_test  = df_raw.iloc[te_logo][PROYECTO_COL].iloc[0]
    n_pos_test = int(y_all_arr[te_logo].sum())
    n_test     = len(te_logo)
    print(f'Fold {fold_i+1} — Proyecto test: {proy_test!r:25s}  n={n_test:,}  positivos={n_pos_test}')

    if n_pos_test < 5:
        print(f'  -> Muy pocos positivos, saltando este fold')
        for k in logo_configs:
            logo_scores[k].append(np.nan)
        continue

    for cfg_name, feat_list in logo_configs.items():
        avail_l = [f for f in feat_list if f in df_raw.columns]
        if len(avail_l) == 0:
            logo_scores[cfg_name].append(np.nan)
            continue
        cat_l, num_l = identify_col_types(df_raw, avail_l)
        use_ord_l    = (alg_diag != 'LR')
        est_l        = ALGORITHMS[alg_diag]()
        pipe_l       = build_pipeline(est_l, cat_l, num_l,
                                      use_smote=False, ordinal=use_ord_l)
        pipe_l.fit(df_raw.iloc[tr_logo][avail_l], y_all_arr[tr_logo])
        y_prob_l = pipe_l.predict_proba(df_raw.iloc[te_logo][avail_l])[:, 1]
        pr_l     = aps(y_all_arr[te_logo], y_prob_l)
        logo_scores[cfg_name].append(pr_l)
        print(f'  {cfg_name:35s} PR-AUC = {pr_l:.4f}')
    print()

print('=' * 65)
print('RESUMEN LOGO-CV')
print('=' * 65)
print(f"  {'Feature Set':35s}  {'Media':6s}  {'Std':5s}  {'vs hold-out':11s}")
print('  ' + '-'*62)
logo_summary = {}
for cfg_name, scores in logo_scores.items():
    valid  = [s for s in scores if not np.isnan(s)]
    mean_  = np.mean(valid) if valid else np.nan
    std_   = np.std(valid)  if valid else np.nan
    logo_summary[cfg_name] = mean_
    if cfg_name.startswith('Original'):
        holdout = best_r_all.get('pr_auc', np.nan)
    elif 'F6' in cfg_name:
        best_f6 = max((r for r in all_results_conv
                       if r.get('feature_set','').startswith('F6')),
                      key=lambda r: r.get('pr_auc', 0), default=None)
        holdout = best_f6['pr_auc'] if best_f6 else np.nan
    else:
        holdout = np.nan
    gap = mean_ - holdout if not np.isnan(holdout) and not np.isnan(mean_) else np.nan
    gap_str = f'{gap:+.4f}' if not np.isnan(gap) else '     N/A'
    print(f"  {cfg_name:35s}  {mean_:6.4f}  {std_:5.4f}  {gap_str}")

print()
orig_mean = logo_summary.get('Original (con vars proyecto)', np.nan)
f6_mean   = logo_summary.get('F6_Lead_Solo', np.nan)
f7_mean   = logo_summary.get('F7_Asequibilidad', np.nan)
if not np.isnan(orig_mean) and not np.isnan(f6_mean):
    caida_absoluta = orig_mean - f6_mean
    caida_rel      = caida_absoluta / orig_mean * 100 if orig_mean > 0 else 0
    print(f'  Caida al eliminar vars proyecto: {caida_absoluta:+.4f} ({caida_rel:.1f}%)')
    if not np.isnan(f7_mean):
        recuperacion = f7_mean - f6_mean
        print(f'  Recuperacion con asequibilidad  : {recuperacion:+.4f}')
    print()
    if caida_absoluta > 0.03:
        print('  DIAGNOSTICO: Las vars de proyecto contribuyen significativamente.')
        if not np.isnan(f7_mean) and f7_mean > f6_mean + 0.005:
            print('               F7_Asequibilidad recupera parte del desempeno.')
            print('               -> RECOMENDADO: usar F7 (generalizable y competitivo)')
        else:
            print('               Las variables relativas no recuperan el desempeno completamente.')
            print('               -> El modelo depende de especificidades de los 5 proyectos actuales.')
            print('               -> Documentar esta limitacion en la tesis.')
    else:
        print('  DIAGNOSTICO: Las vars de proyecto no son el driver principal.')
        print('               El modelo aprende patrones generalizables del lead.')


### 7.9 - Ingenieria de Caracteristicas para Generalizacion

Se construyen **variables relativas** que capturan la relacion lead-proyecto
sin codificar la identidad del proyecto:

| Variable | Formula | Interpretacion | Generalizable |
|----------|---------|----------------|---------------|
| `asequibilidad_indice` | ingreso / precio_promedio | Fraccion del precio que representa el ingreso | Si |
| `m2_por_persona` | m2_minimo / nucleo_familiar | Metros cuadrados disponibles por persona | Si |
| `ingreso_por_m2` | ingreso / m2_minimo | Poder adquisitivo por m2 de vivienda | Si |

**Por que generalizan:** un nuevo proyecto con precio diferente al de los 5
actuales mantiene el mismo significado del ratio. Un lead con
`asequibilidad_indice = 0.03` (ingreso = 3% del precio) se comporta igual
independientemente del precio absoluto del proyecto.

**Nuevos feature sets:**
- **F6_Lead_Solo**: Variables exclusivas del lead (sin ninguna variable de proyecto)
- **F7_Asequibilidad**: F6 + variables relativas lead x proyecto


In [ ]:
# === 7.9.1  Registrar feature sets y ejecutar experimentos de generalizacion ===
FEATURE_SETS['F6_Lead_Solo']     = F6_LEAD_ONLY
FEATURE_SETS['F7_Asequibilidad'] = F7_ASEQUIBILIDAD

print('Feature sets registrados:')
print(f'  F6_Lead_Solo:     {len(F6_LEAD_ONLY)} variables')
print(f'  F7_Asequibilidad: {len(F7_ASEQUIBILIDAD)} variables  (+{ENGINEERED_VARS})')
print()

gen_feature_sets = {'F6_Lead_Solo': F6_LEAD_ONLY, 'F7_Asequibilidad': F7_ASEQUIBILIDAD}

print('Entrenando modelos — Exp A (sin SMOTE)...')
results_conv_gen_a = run_experiment(
    df_raw, y_conv, idx_train_conv, idx_test_conv,
    gen_feature_sets, 'Conversion',
    use_smote=False, algorithms=ALGORITHMS
)

print('Entrenando modelos — Exp B (con SMOTE)...')
results_conv_gen_b = run_experiment(
    df_raw, y_conv, idx_train_conv, idx_test_conv,
    gen_feature_sets, 'Conversion',
    use_smote=True, smote_strategy=0.10, algorithms=ALGORITHMS
)

all_results_conv += results_conv_gen_a + results_conv_gen_b

df_results_conv = pd.DataFrame([
    {k: v for k, v in r.items()
     if k not in ('pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain')}
    for r in all_results_conv
])
df_results_conv.to_csv(RESULTS_DIR / 'results_conv_all.csv', index=False)

# Show new results
df_gen = df_results_conv[df_results_conv['feature_set'].isin(gen_feature_sets.keys())].copy()
print()
print(f'Resultados F6 y F7 ({len(df_gen)} modelos):')
cols_s = ['algorithm','feature_set','experiment','pr_auc','roc_auc','lift_10pct','rec_10pct']
print(df_gen.sort_values('pr_auc', ascending=False)[cols_s].to_string(index=False))


In [ ]:
# === 7.9.2  Tabla comparativa: hold-out vs LOGO-CV por feature set ===
# Muestra el tradeoff: desempeno en proyectos conocidos vs generalizabilidad

# Build comparison: best hold-out PR-AUC per feature set vs LOGO-CV mean
fs_order = ['F1_Base','F2_Base+Demo','F3_Base+Comerc','F4_Todas',
            'F6_Lead_Solo','F7_Asequibilidad']

# Standardize logo config names for lookup
logo_lookup = {
    'Original (con vars proyecto)': logo_summary.get('Original (con vars proyecto)', np.nan),
    'F6_Lead_Solo'                : logo_summary.get('F6_Lead_Solo', np.nan),
    'F7_Asequibilidad'            : logo_summary.get('F7_Asequibilidad', np.nan),
}

print('COMPARACION HOLD-OUT vs LOGO-CV')
print('Hold-out = PR-AUC en test set con el mismo split de 5 proyectos')
print('LOGO-CV  = PR-AUC entrenando en 4 proyectos, prediciendo en el 5to')
print()
print(f"  {'Feature Set':20s}  {'Best Algo':5s}  {'Hold-out':8s}  {'LOGO-CV':8s}  {'Gap':7s}  Nota")
print('  ' + '-'*80)

for fs in fs_order:
    rows = df_results_conv[df_results_conv['feature_set'] == fs]
    if len(rows) == 0:
        continue
    best_row  = rows.loc[rows['pr_auc'].idxmax()]
    holdout   = float(best_row['pr_auc'])
    best_alg  = str(best_row['algorithm'])

    # Find LOGO-CV for this feature set
    if fs.startswith('F6'):
        logo_m = logo_lookup.get('F6_Lead_Solo', np.nan)
    elif fs.startswith('F7'):
        logo_m = logo_lookup.get('F7_Asequibilidad', np.nan)
    elif fs == fs_diag or fs.startswith(fs_diag.split('_')[0]):
        logo_m = logo_lookup.get('Original (con vars proyecto)', np.nan)
    else:
        logo_m = np.nan

    gap     = logo_m - holdout if not np.isnan(logo_m) else np.nan
    has_pv  = any(v in FEATURE_SETS.get(fs,[]) for v in CONFIRMED_PROJECT_VARS)
    nota    = 'vars proyecto' if has_pv else 'solo lead'
    if fs.startswith('F7'):
        nota = 'asequibilidad'

    logo_str = f'{logo_m:.4f}' if not np.isnan(logo_m) else '   N/A'
    gap_str  = f'{gap:+.4f}' if not np.isnan(gap) else '    N/A'
    print(f"  {fs:20s}  {best_alg:5s}  {holdout:8.4f}  {logo_str:8s}  {gap_str:7s}  {nota}")

print()
f6_holdout = df_results_conv[df_results_conv['feature_set']=='F6_Lead_Solo']['pr_auc'].max() if 'F6_Lead_Solo' in df_results_conv['feature_set'].values else np.nan
f7_holdout = df_results_conv[df_results_conv['feature_set']=='F7_Asequibilidad']['pr_auc'].max() if 'F7_Asequibilidad' in df_results_conv['feature_set'].values else np.nan
f4_holdout = df_results_conv[df_results_conv['feature_set']==f4_key]['pr_auc'].max() if f4_key and f4_key in df_results_conv['feature_set'].values else np.nan

print('INTERPRETACION:')
if not np.isnan(f7_holdout) and not np.isnan(f4_holdout):
    diff_f7_f4 = f7_holdout - f4_holdout
    print(f'  F7 vs F4 (hold-out): {diff_f7_f4:+.4f}')
    if diff_f7_f4 >= -0.01:
        print('  -> F7_Asequibilidad mantiene desempeno comparable a F4 sin codificar identidad del proyecto.')
        print('     RECOMENDADO para produccion: mayor capacidad de generalizacion.')
    elif diff_f7_f4 >= -0.03:
        print('  -> F7_Asequibilidad cae ligeramente vs F4. Considerar si la generalizabilidad')
        print('     justifica la caida segun el contexto de produccion.')
    else:
        print('  -> Caida importante vs F4. Documentar limitacion: el modelo actual depende')
        print('     de las caracteristicas especificas de los 5 proyectos disponibles.')
print(f'  Los feature sets F6 y F7 estan disponibles para la seleccion en Sec 7.10.')


### 7.10 - Seleccion de Modelos Candidatos

Antes de iniciar el refinamiento se identifican los **Top 5 modelos candidatos** ordenados por PR-AUC considerando **todos los experimentos** (A_NoSMOTE y B_SMOTE), sin limitar la busqueda al experimento A. Los candidatos seran refinados de forma independiente en las etapas siguientes.


In [ ]:
# === 7.10 Seleccion de modelos candidatos (todos los experimentos) ===
N_CANDIDATES = 5

top_candidates = (
    df_results_conv[
        ~df_results_conv['feature_set'].str.contains('cluster', case=False, na=False)
    ]
    .sort_values('pr_auc', ascending=False)
    .drop_duplicates(subset=['algorithm','feature_set','experiment'])
    .head(N_CANDIDATES)
    .reset_index(drop=True)
)

print('TOP CANDIDATOS - TODOS LOS EXPERIMENTOS (por PR-AUC)')
cols_show = ['algorithm','feature_set','experiment','pr_auc','roc_auc','recall','lift_10pct']
print(top_candidates[cols_show].to_string(index=True))

candidate_results_conv = []
for _, row in top_candidates.iterrows():
    match = next(
        (r for r in all_results_conv
         if r.get('algorithm') == row['algorithm']
         and r.get('feature_set') == row['feature_set']
         and r.get('experiment') == row['experiment']
         and 'pipeline' in r),
        None
    )
    if match:
        candidate_results_conv.append(match)
    else:
        print(f"  WARNING: no pipeline para {row['algorithm']} | {row['feature_set']} | {row['experiment']}")

print(f'\n{len(candidate_results_conv)} candidatos con pipeline disponible.')


### 7.11 - Etapa 1: Refinamiento de Variables

Para cada modelo candidato se computa la **importancia de permutacion**: se mide cuanto se degrada el PR-AUC cuando se aleatoriza una variable. Si una variable tiene importancia cero o negativa, no aporta informacion discriminativa.

**Criterio de eliminacion:** variables con importancia relativa **< 1% del total de importancia positiva acumulada**.

**Nota:** `HistGradientBoostingClassifier` no expone `feature_importances_`; se usa importancia de permutacion para todos los algoritmos, garantizando consistencia metodologica.


In [ ]:
# === 7.11.1 Funciones de refinamiento de variables ===
from sklearn.inspection import permutation_importance as perm_imp_fn
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             f1_score, precision_score, recall_score)


def compute_perm_importance(result, n_repeats=5, sample_size=5000, seed=SEED):
    # Permutation importance using PR-AUC. Subsamples test set for speed.
    pipe       = result['pipeline']
    fs         = FEATURE_SETS.get(result['feature_set'], [])
    avail      = [f for f in fs if f in df_raw.columns]
    cat_cols, num_cols = identify_col_types(df_raw, avail)
    feat_names = cat_cols + num_cols

    X_test_raw = df_raw.iloc[idx_test_conv][avail]
    y_test     = np.array(y_conv[idx_test_conv])

    n   = min(sample_size, len(X_test_raw))
    rng = np.random.RandomState(seed)
    idx_s = rng.choice(len(X_test_raw), n, replace=False)
    X_s   = X_test_raw.iloc[idx_s]
    y_s   = y_test[idx_s]

    clf  = pipe.named_steps['clf']
    prep = pipe.named_steps['prep']
    X_t  = prep.transform(X_s)

    pi = perm_imp_fn(clf, X_t, y_s, n_repeats=n_repeats,
                     random_state=seed, scoring='average_precision')

    df_pi = pd.DataFrame({
        'feature'         : feat_names[:X_t.shape[1]],
        'importance_mean' : pi.importances_mean,
        'importance_std'  : pi.importances_std,
    }).sort_values('importance_mean', ascending=False).reset_index(drop=True)
    return df_pi, feat_names, avail


def get_features_to_keep(df_pi, threshold_rel=0.01):
    # Keep features with importance >= threshold_rel * total positive importance.
    pos_total = df_pi[df_pi['importance_mean'] > 0]['importance_mean'].sum()
    if pos_total <= 0:
        return list(df_pi['feature'])
    cutoff = threshold_rel * pos_total
    keep   = list(df_pi[df_pi['importance_mean'] >= cutoff]['feature'])
    return keep if len(keep) >= 3 else list(df_pi['feature'].iloc[:3])


def retrain_on_features(alg_name, feature_list, use_smote, task='conv'):
    # Rebuild and retrain same algorithm on a custom feature list.
    avail = [f for f in feature_list if f in df_raw.columns]
    if not avail:
        return None
    cat_c, num_c = identify_col_types(df_raw, avail)
    use_ord  = (alg_name != 'LR')
    est      = ALGORITHMS[alg_name]()
    pipe_new = build_pipeline(est, cat_c, num_c,
                              use_smote=use_smote, smote_strategy=0.10,
                              ordinal=use_ord)
    if task == 'conv':
        idx_tr, idx_te, y_all = idx_train_conv, idx_test_conv, y_conv
    else:
        idx_tr, idx_te, y_all = idx_train_aband, idx_test_aband, y_aband

    X_tr = df_raw.iloc[idx_tr][avail]
    X_te = df_raw.iloc[idx_te][avail]
    y_tr = y_all[idx_tr]
    y_te = np.array(y_all[idx_te])

    pipe_new.fit(X_tr, y_tr)
    y_prob = pipe_new.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'pipeline'  : pipe_new,
        'features'  : avail,
        'n_features': len(avail),
        'pr_auc'    : average_precision_score(y_te, y_prob),
        'roc_auc'   : roc_auc_score(y_te, y_prob),
        'f1'        : f1_score(y_te, y_pred, zero_division=0),
        'recall'    : recall_score(y_te, y_pred, zero_division=0),
        'precision' : precision_score(y_te, y_pred, zero_division=0),
        'y_prob'    : y_prob,
        'y_true'    : y_te,
    }


print('Funciones definidas: compute_perm_importance, get_features_to_keep, retrain_on_features')


In [ ]:
# === 7.11.2 Analisis de importancia y refinamiento por candidato ===
PERM_REPEATS   = 5
PERM_SAMPLE    = 5000
DROP_THRESHOLD = 0.01

refinement_conv = []

for cand_idx, r in enumerate(candidate_results_conv):
    alg       = r['algorithm']
    fs        = r['feature_set']
    exp       = r['experiment']
    use_smote = (exp == 'B_SMOTE')

    print(f"\n{'-'*65}")
    print(f"Candidato {cand_idx+1}: {alg} | {fs} | {exp}")
    print(f"  PR-AUC base: {r['pr_auc']:.4f}  |  ROC-AUC: {r['roc_auc']:.4f}")

    df_pi, feat_names, avail_orig = compute_perm_importance(
        r, n_repeats=PERM_REPEATS, sample_size=PERM_SAMPLE)

    keep_feats = get_features_to_keep(df_pi, threshold_rel=DROP_THRESHOLD)
    drop_feats = [f for f in avail_orig if f not in keep_feats]
    print(f"  Originales={len(avail_orig)}  Eliminadas={len(drop_feats)}  Retenidas={len(keep_feats)}")
    if drop_feats:
        print(f"    Eliminadas: {drop_feats}")

    if drop_feats:
        ref = retrain_on_features(alg, keep_feats, use_smote, task='conv')
        if ref:
            d_pr  = ref['pr_auc'] - r['pr_auc']
            d_roc = ref['roc_auc'] - r['roc_auc']
            print(f"  PR-AUC refinado : {ref['pr_auc']:.4f}  (Delta={d_pr:+.4f})")
            print(f"  ROC-AUC refinado: {ref['roc_auc']:.4f}  (Delta={d_roc:+.4f})")
        else:
            ref = None
            print('  ERROR reentrenando - se mantiene conjunto original.')
    else:
        ref = None
        print('  Ninguna variable eliminada - modelo ya es minimo.')

    refinement_conv.append({
        'candidate_id'   : cand_idx,
        'algorithm'      : alg,
        'feature_set'    : fs,
        'experiment'     : exp,
        'use_smote'      : use_smote,
        'importance_df'  : df_pi,
        'features_orig'  : avail_orig,
        'features_keep'  : keep_feats,
        'features_drop'  : drop_feats,
        'pr_auc_orig'    : r['pr_auc'],
        'roc_auc_orig'   : r['roc_auc'],
        'pipeline_orig'  : r['pipeline'],
        'y_prob_orig'    : np.array(r.get('y_prob_test', [])),
        'refined'        : ref,
        'pr_auc_final'   : ref['pr_auc']   if ref else r['pr_auc'],
        'roc_auc_final'  : ref['roc_auc']  if ref else r['roc_auc'],
        'pipeline_final' : ref['pipeline'] if ref else r['pipeline'],
        'features_final' : ref['features'] if ref else avail_orig,
        'y_prob_final'   : ref['y_prob']   if ref else np.array(r.get('y_prob_test', [])),
        'y_true'         : np.array(y_conv[idx_test_conv]),
    })

print(f"\n{'='*65}")
print('  RESUMEN REFINAMIENTO - TAREA 1: CONVERSION')
print('='*65)
print(f"  {'#':>2}  {'Alg':>5}  {'FeatureSet':>16}  {'Exp':>10}  "
      f"{'N_orig':>6}  {'N_ref':>5}  {'PR_orig':>7}  {'PR_ref':>7}  {'Delta':>7}")
print('  ' + '-'*72)
for d in refinement_conv:
    n_r   = len(d['features_final'])
    pr_r  = d['pr_auc_final']
    delta = pr_r - d['pr_auc_orig']
    print(f"  {d['candidate_id']+1:>2}  {d['algorithm']:>5}  {d['feature_set']:>16}  "
          f"{d['experiment']:>10}  {len(d['features_orig']):>6}  {n_r:>5}  "
          f"{d['pr_auc_orig']:>7.4f}  {pr_r:>7.4f}  {delta:>+7.4f}")


In [ ]:
# === 7.11.3 Visualizacion: importancia de variables por candidato ===
from matplotlib.patches import Patch

n_cands = len(refinement_conv)
fig, axes = plt.subplots(1, n_cands, figsize=(4.5 * n_cands, 7))
if n_cands == 1:
    axes = [axes]

for ax, d in zip(axes, refinement_conv):
    df_pi  = d['importance_df'].head(15)
    colors = ['#EF5350' if f in d['features_drop'] else '#42A5F5'
              for f in df_pi['feature']]
    n = len(df_pi)
    ax.barh(range(n), df_pi['importance_mean'][::-1].values,
            xerr=df_pi['importance_std'][::-1].values,
            color=colors[::-1], edgecolor='white', alpha=0.9, capsize=3)
    ax.set_yticks(range(n))
    ax.set_yticklabels(df_pi['feature'][::-1].tolist(), fontsize=8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f"{d['algorithm']} | {d['feature_set']}\n{d['experiment']}",
                 fontweight='bold', fontsize=9)
    ax.set_xlabel('Importancia permutacion (Delta PR-AUC)')

handles = [Patch(color='#42A5F5', label='Retenida'),
           Patch(color='#EF5350', label='Eliminada (< 1% importancia relativa)')]
fig.legend(handles=handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.06))
plt.suptitle('Etapa 1 - Importancia de Permutacion por Candidato (Tarea 1: Conversion)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'refinamiento_variables_conv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: figures/refinamiento_variables_conv.png')


### 7.12 - Etapa 2: Optimizacion del Umbral de Decision

El umbral de clasificacion convierte la probabilidad predicha en etiqueta binaria. El valor por defecto de **0.5 raramente es optimo** con datasets desbalanceados: con 1.18% de positivos, practicamente ningun lead lo superaria.

Se evaluan **umbrales de 0.05 a 0.95** calculando Precision, Recall, F1, Specificity, Accuracy, Lift, N leads priorizados y % del dataset.

**Umbrales de referencia:**
1. **F1-maximo** - maximiza media armonica precision-recall.
2. **Recall-maximo** (con Precision >= 10%) - mayor cobertura con precision minima.
3. **Operativo Top-10%** - umbral que prioriza aprox. el 10% del dataset.


In [ ]:
# === 7.12.1 Funciones de barrido de umbrales ===

def threshold_sweep(y_true, y_prob, thresholds=None):
    # Compute key classification metrics at each decision threshold.
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
    y_true_arr = np.asarray(y_true, dtype=int)
    n_pos   = int(y_true_arr.sum())
    n_total = len(y_true_arr)
    base_rate = n_pos / n_total
    rows = []
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        tp = int(((y_pred==1) & (y_true_arr==1)).sum())
        fp = int(((y_pred==1) & (y_true_arr==0)).sum())
        tn = int(((y_pred==0) & (y_true_arr==0)).sum())
        fn = int(((y_pred==0) & (y_true_arr==1)).sum())
        prec = tp/(tp+fp) if (tp+fp) > 0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn) > 0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
        spec = tn/(tn+fp) if (tn+fp) > 0 else 0.0
        acc  = (tp+tn) / n_total
        n_pp = int(y_pred.sum())
        pct  = n_pp / n_total * 100
        lift = prec / base_rate if base_rate > 0 and prec > 0 else 0.0
        rows.append({'threshold': thr, 'precision': prec, 'recall': rec,
                     'f1': f1, 'specificity': spec, 'accuracy': acc,
                     'lift': lift, 'n_positive': n_pp, 'pct_positive': pct,
                     'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn})
    return pd.DataFrame(rows)


def find_optimal_thresholds(df_sw, min_precision=0.10, target_pct=10.0):
    # Return dict of optimal thresholds: f1_max, recall_max, operational.
    idx_f1 = df_sw['f1'].idxmax()
    thr_f1 = float(df_sw.loc[idx_f1, 'threshold'])

    ok = df_sw[df_sw['precision'] >= min_precision]
    if len(ok) > 0:
        idx_rc = ok['recall'].idxmax()
        thr_rc = float(ok.loc[idx_rc, 'threshold'])
        row_rc = ok.loc[idx_rc]
    else:
        thr_rc = thr_f1
        row_rc = df_sw.loc[idx_f1]

    df_sw2 = df_sw.copy()
    df_sw2['dist_op'] = (df_sw2['pct_positive'] - target_pct).abs()
    idx_op = df_sw2['dist_op'].idxmin()
    thr_op = float(df_sw2.loc[idx_op, 'threshold'])

    return {
        'f1_max'      : {'threshold': thr_f1, 'row': df_sw.loc[idx_f1]},
        'recall_max'  : {'threshold': thr_rc, 'row': row_rc},
        'operational' : {'threshold': thr_op, 'row': df_sw2.loc[idx_op]},
    }


print('Funciones definidas: threshold_sweep, find_optimal_thresholds')


In [ ]:
# === 7.12.2 Barrido de umbrales y visualizacion por candidato ===
n_cands = len(refinement_conv)
fig, axes = plt.subplots(n_cands, 3, figsize=(15, 4.5 * n_cands))
if n_cands == 1:
    axes = axes.reshape(1, -1)

threshold_summaries_conv = []

for row_i, d in enumerate(refinement_conv):
    y_prob = d['y_prob_final']
    y_true = d['y_true']
    if len(y_prob) == 0:
        print(f"Candidato {d['candidate_id']+1}: sin probabilidades, saltando.")
        continue

    df_sw = threshold_sweep(y_true, y_prob)
    opt   = find_optimal_thresholds(df_sw, min_precision=0.10, target_pct=10.0)
    d['threshold_df']  = df_sw
    d['threshold_opt'] = opt

    lbl = f"{d['algorithm']} | {d['feature_set']}"
    thr_colors = {'f1_max': '#42A5F5', 'recall_max': '#66BB6A', 'operational': '#FFA726'}
    ax0, ax1, ax2 = axes[row_i, 0], axes[row_i, 1], axes[row_i, 2]

    ax0.plot(df_sw['threshold'], df_sw['precision'], '#42A5F5', lw=1.8, label='Precision')
    ax0.plot(df_sw['threshold'], df_sw['recall'],    '#EF5350', lw=1.8, label='Recall')
    ax0.plot(df_sw['threshold'], df_sw['f1'],        '#66BB6A', lw=1.8, ls='--', label='F1')
    for k, p in opt.items():
        ax0.axvline(p['threshold'], color=thr_colors[k], linestyle=':', lw=1.3)
    ax0.set_title(f'{lbl}\nPrecision / Recall / F1 vs Threshold', fontsize=9, fontweight='bold')
    ax0.set_xlabel('Threshold'); ax0.set_ylabel('Metrica')
    ax0.legend(fontsize=8); ax0.grid(alpha=0.3)

    ax1.plot(df_sw['threshold'], df_sw['f1'], '#66BB6A', lw=2)
    ax1.axvline(opt['f1_max']['threshold'], color='#66BB6A', ls='--', lw=1.5,
                label=f"Max F1 @ {opt['f1_max']['threshold']:.2f}")
    ax1.set_title(f'{lbl}\nF1 vs Threshold', fontsize=9, fontweight='bold')
    ax1.set_xlabel('Threshold'); ax1.set_ylabel('F1')
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    ax2.plot(df_sw['threshold'], df_sw['pct_positive'], '#AB47BC', lw=2)
    ax2.axhline(10, color='#FFA726', ls='--', lw=1.5, label='Top 10% operativo')
    ax2.axvline(opt['operational']['threshold'], color='#FFA726', ls=':', lw=1.3)
    ax2.set_title(f'{lbl}\n% Leads priorizados vs Threshold', fontsize=9, fontweight='bold')
    ax2.set_xlabel('Threshold'); ax2.set_ylabel('% clasificados positivos')
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    row_f1 = opt['f1_max']['row']
    row_rc = opt['recall_max']['row']
    row_op = opt['operational']['row']
    threshold_summaries_conv.append({
        'candidate'         : f"{d['algorithm']}|{d['feature_set']}|{d['experiment']}",
        'thr_f1_max'        : opt['f1_max']['threshold'],
        'f1_at_thr'         : float(row_f1['f1']),
        'prec_at_f1'        : float(row_f1['precision']),
        'rec_at_f1'         : float(row_f1['recall']),
        'thr_recall_max'    : opt['recall_max']['threshold'],
        'rec_at_recall_max' : float(row_rc['recall']),
        'prec_at_recall_max': float(row_rc['precision']),
        'thr_op_10pct'      : opt['operational']['threshold'],
        'prec_at_op'        : float(row_op['precision']),
        'rec_at_op'         : float(row_op['recall']),
        'lift_at_op'        : float(row_op['lift']),
        'n_leads_op'        : int(row_op['n_positive']),
    })

plt.suptitle('Etapa 2 - Optimizacion del Umbral de Decision (Tarea 1: Conversion)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'threshold_optimization_conv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: figures/threshold_optimization_conv.png')


In [ ]:
# === 7.12.3 Tabla resumen de umbrales optimos ===
df_thr_summary = pd.DataFrame(threshold_summaries_conv)

print('Threshold F1-maximo:')
print(df_thr_summary[['candidate','thr_f1_max','f1_at_thr','prec_at_f1','rec_at_f1']].to_string(index=False))
print()
print('Threshold Recall-maximo (Precision >= 10%):')
print(df_thr_summary[['candidate','thr_recall_max','rec_at_recall_max','prec_at_recall_max']].to_string(index=False))
print()
print('Threshold operativo Top-10%:')
print(df_thr_summary[['candidate','thr_op_10pct','prec_at_op','rec_at_op','lift_at_op','n_leads_op']].to_string(index=False))

for d, row in zip(refinement_conv, threshold_summaries_conv):
    d['thr_f1_max']   = row['thr_f1_max']
    d['thr_op_10pct'] = row['thr_op_10pct']

df_thr_summary.to_csv(RESULTS_DIR / 'threshold_optimization_conv.csv', index=False)
print('Guardado: results/threshold_optimization_conv.csv')


### 7.13 - Etapa 3: Seleccion Definitiva del Modelo

La seleccion considera **multiples criterios** ponderados:

| Criterio | Peso | Justificacion |
|----------|------|---------------|
| PR-AUC refinado | 30% | Metrica primaria en datasets desbalanceados |
| Lift@10% operativo | 25% | Valor practico para el equipo comercial |
| Recall@10% threshold | 20% | Cobertura de conversiones detectadas |
| ROC-AUC | 10% | Discriminacion general |
| Parsimonia (menos variables) | 10% | Interpretabilidad y mantenibilidad |
| Sin SMOTE | 5% | Preferencia por menor complejidad del pipeline |

El **umbral recomendado** es el threshold operativo Top-10%.


In [ ]:
# === 7.13 Seleccion definitiva - Tarea 1: Conversion ===

def composite_score_refined(d):
    # Multi-criteria composite score for refined candidate.
    pr_n  = d['pr_auc_final']
    roc_n = d['roc_auc_final']
    n_v   = len(d['features_final'])
    pars  = max(0, (20 - n_v) / 20)
    no_sm = 0.0 if d['use_smote'] else 1.0

    lift_n = 0.0
    rec_n  = 0.0
    if 'threshold_opt' in d and 'operational' in d['threshold_opt']:
        r_op   = d['threshold_opt']['operational']['row']
        lift_n = min(float(r_op['lift']), 15) / 15
        rec_n  = float(r_op['recall'])
    return (0.30 * pr_n + 0.25 * lift_n + 0.20 * rec_n +
            0.10 * roc_n + 0.10 * pars + 0.05 * no_sm)


for d in refinement_conv:
    d['composite_score'] = composite_score_refined(d)

ranked = sorted(refinement_conv, key=lambda d: d['composite_score'], reverse=True)

print('SELECCION DEFINITIVA - TAREA 1: CONVERSION')
print(f"  {'#':>2}  {'Alg':>5}  {'FeatureSet':>16}  {'Exp':>10}  "
      f"{'PR_ref':>7}  {'N_vars':>6}  {'Score':>7}")
print('  ' + '-'*60)
for rank, d in enumerate(ranked, 1):
    marker = ' <<< SELECCIONADO' if rank == 1 else ''
    print(f"  {rank:>2}  {d['algorithm']:>5}  {d['feature_set']:>16}  "
          f"{d['experiment']:>10}  {d['pr_auc_final']:>7.4f}  "
          f"{len(d['features_final']):>6}  {d['composite_score']:>7.4f}{marker}")

best_conv           = ranked[0]
best_conv_pipeline  = best_conv['pipeline_final']
best_conv_features  = best_conv['features_final']
best_conv_threshold = best_conv.get('thr_op_10pct', 0.5)
best_conv_exp       = best_conv['experiment']

print(f"\n  Modelo seleccionado:")
print(f"    Algoritmo  : {best_conv['algorithm']}")
print(f"    FeatureSet : {best_conv['feature_set']} (refinado)")
print(f"    Variables  : {best_conv_features}")
print(f"    Experimento: {best_conv_exp}")
print(f"    PR-AUC     : {best_conv['pr_auc_final']:.4f}")
print(f"    ROC-AUC    : {best_conv['roc_auc_final']:.4f}")
print(f"    Threshold  : {best_conv_threshold:.2f} (operativo Top-10%)")


### 7.14 - Etapa 4: Validacion Cruzada del Modelo Seleccionado

La validacion cruzada se realiza **unicamente sobre el modelo seleccionado** y con su **conjunto refinado de variables**. Si el modelo proviene de un experimento con SMOTE, el pipeline mantiene SMOTE dentro de cada fold para evitar *data leakage*.

Se usa **StratifiedKFold de 5 folds** sobre el conjunto de entrenamiento. Metrica primaria: PR-AUC. Se reportan tambien ROC-AUC y F1 con gap train/test.


In [ ]:
# === 7.14 Validacion cruzada del modelo seleccionado - Tarea 1 ===
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer

print(f"Modelo: {best_conv['algorithm']} | {best_conv['feature_set']} | {best_conv_exp}")
print(f"Variables ({len(best_conv_features)}): {best_conv_features}")

avail_cv       = [f for f in best_conv_features if f in df_raw.columns]
cat_cv, num_cv = identify_col_types(df_raw, avail_cv)
est_cv         = ALGORITHMS[best_conv['algorithm']]()
use_ord_cv     = (best_conv['algorithm'] != 'LR')
pipe_cv        = build_pipeline(est_cv, cat_cv, num_cv,
                                use_smote=best_conv['use_smote'], smote_strategy=0.10,
                                ordinal=use_ord_cv)

X_train_cv = df_raw.iloc[idx_train_conv][avail_cv]
y_train_cv = y_conv[idx_train_conv]

cv_skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pr_scorer = make_scorer(average_precision_score, needs_proba=True)
scorers   = {'pr_auc': pr_scorer, 'roc_auc': 'roc_auc', 'f1': 'f1'}

cv_res = cross_validate(pipe_cv, X_train_cv, y_train_cv,
                        cv=cv_skf, scoring=scorers,
                        return_train_score=True, n_jobs=-1)

print()
print(f"  {'Metrica':>8}  {'Test mean':>9}  {'Test std':>8}  {'Train mean':>10}  {'CV%':>5}")
print('  ' + '-'*45)
for metric in ['pr_auc', 'roc_auc', 'f1']:
    ts     = cv_res[f'test_{metric}']
    tr     = cv_res[f'train_{metric}']
    cv_pct = ts.std() / ts.mean() * 100 if ts.mean() > 0 else 0
    print(f"  {metric:>8}  {ts.mean():>9.4f}  {ts.std():>8.4f}  {tr.mean():>10.4f}  {cv_pct:>4.1f}%")

gap = cv_res['train_pr_auc'].mean() - cv_res['test_pr_auc'].mean()
if gap < 0.05:
    verdict = 'Sin sobreajuste significativo'
elif gap < 0.15:
    verdict = 'Sobreajuste moderado - considerar regularizacion'
else:
    verdict = 'Sobreajuste elevado - revisar complejidad'
print(f"\n  Gap train-test PR-AUC: {gap:.4f} - {verdict}")
print(f"  PR-AUC hold-out (Etapa 1): {best_conv['pr_auc_final']:.4f}")
print(f"  PR-AUC CV-5fold (media)  : {cv_res['test_pr_auc'].mean():.4f}")


### 7.15 - Etapa 5: Optimizacion de Hiperparametros

Con el modelo definitivo y sus variables refinadas se aplica `RandomizedSearchCV` (15 iteraciones, 3-fold CV, scorer = PR-AUC). El tuning se realiza **unicamente sobre el modelo ya seleccionado**.

Se compara desempeno antes/despues. Si el modelo tuneado no mejora, se mantiene el original.


In [ ]:
# === 7.15 Hyperparameter tuning - modelo seleccionado, Tarea 1 ===
import time
from sklearn.model_selection import RandomizedSearchCV

alg_tune   = best_conv['algorithm']
avail_tune = [f for f in best_conv_features if f in df_raw.columns]
cat_tune, num_tune = identify_col_types(df_raw, avail_tune)
use_ord_tune = (alg_tune != 'LR')

PARAM_DISTS = {
    'GBM': {
        'clf__max_iter'         : [200, 400, 600],
        'clf__learning_rate'    : [0.01, 0.05, 0.10, 0.15],
        'clf__max_leaf_nodes'   : [31, 63, 127],
        'clf__min_samples_leaf' : [20, 50, 100],
        'clf__l2_regularization': [0.0, 0.1, 1.0],
    },
    'RF': {
        'clf__n_estimators'     : [60, 120, 200],
        'clf__max_depth'        : [None, 10, 20, 30],
        'clf__min_samples_split': [2, 5, 10],
        'clf__max_features'     : ['sqrt', 'log2', 0.5],
    },
    'EXT': {
        'clf__n_estimators'     : [60, 120, 200],
        'clf__max_depth'        : [None, 10, 20],
        'clf__min_samples_split': [2, 5, 10],
        'clf__max_features'     : ['sqrt', 'log2', 0.5],
    },
    'LR': {
        'clf__C'       : [0.01, 0.1, 1.0, 10.0],
        'clf__solver'  : ['lbfgs', 'saga'],
        'clf__max_iter': [300, 500, 1000],
    },
}

param_dist   = PARAM_DISTS.get(alg_tune, {})
X_train_tune = df_raw.iloc[idx_train_conv][avail_tune]
y_train_tune = y_conv[idx_train_conv]
X_test_tune  = df_raw.iloc[idx_test_conv][avail_tune]
y_test_tune  = np.array(y_conv[idx_test_conv])

if param_dist:
    est_tune  = ALGORITHMS[alg_tune]()
    pipe_tune = build_pipeline(est_tune, cat_tune, num_tune,
                               use_smote=best_conv['use_smote'], smote_strategy=0.10,
                               ordinal=use_ord_tune)
    pr_scorer_tune = make_scorer(average_precision_score, needs_proba=True)
    rscv = RandomizedSearchCV(pipe_tune, param_dist, n_iter=15, cv=3,
                              scoring=pr_scorer_tune, random_state=SEED,
                              n_jobs=-1, verbose=0)
    print(f'Buscando hiperparametros para {alg_tune} ({len(avail_tune)} variables)...')
    t0 = time.time()
    rscv.fit(X_train_tune, y_train_tune)
    print(f'  Tiempo: {time.time()-t0:.1f}s')
    print(f'  Mejores params: {rscv.best_params_}')
    print(f'  Mejor PR-AUC CV (3-fold): {rscv.best_score_:.4f}')

    y_prob_tuned = rscv.best_estimator_.predict_proba(X_test_tune)[:, 1]
    pr_tuned     = average_precision_score(y_test_tune, y_prob_tuned)
    roc_tuned    = roc_auc_score(y_test_tune, y_prob_tuned)

    print(f"\n  {'Metrica':>8}  {'Antes':>8}  {'Despues':>8}  {'Delta':>7}")
    print(f"  {'PR-AUC':>8}  {best_conv['pr_auc_final']:>8.4f}  {pr_tuned:>8.4f}  "
          f"{pr_tuned - best_conv['pr_auc_final']:>+7.4f}")
    print(f"  {'ROC-AUC':>8}  {best_conv['roc_auc_final']:>8.4f}  {roc_tuned:>8.4f}  "
          f"{roc_tuned - best_conv['roc_auc_final']:>+7.4f}")

    if pr_tuned >= best_conv['pr_auc_final'] - 0.001:
        best_conv_pipeline_final = rscv.best_estimator_
        best_conv_pr_final       = pr_tuned
        best_conv_roc_final      = roc_tuned
        best_conv_y_prob_final   = y_prob_tuned
        print(f'  Modelo tuneado adoptado (PR-AUC = {pr_tuned:.4f})')
    else:
        best_conv_pipeline_final = best_conv['pipeline_final']
        best_conv_pr_final       = best_conv['pr_auc_final']
        best_conv_roc_final      = best_conv['roc_auc_final']
        best_conv_y_prob_final   = best_conv['y_prob_final']
        print(f'  Tuning no mejoro - se mantiene pre-tuning (PR-AUC = {best_conv_pr_final:.4f})')
else:
    print(f'Sin espacio de hiperparametros para {alg_tune}.')
    best_conv_pipeline_final = best_conv['pipeline_final']
    best_conv_pr_final       = best_conv['pr_auc_final']
    best_conv_roc_final      = best_conv['roc_auc_final']
    best_conv_y_prob_final   = best_conv['y_prob_final']


### 7.16 - Evaluacion Final del Modelo de Conversion

Evaluacion completa sobre el conjunto de prueba usando el **umbral operativo Top-10%** identificado en la Etapa 2.


In [ ]:
# === 7.16 Evaluacion final - Tarea 1: Conversion ===
from sklearn.metrics import (precision_recall_curve, roc_curve,
                             ConfusionMatrixDisplay)

y_test_final = np.array(y_conv[idx_test_conv])
y_prob_final = best_conv_y_prob_final
thr_final    = best_conv_threshold
y_pred_final = (y_prob_final >= thr_final).astype(int)

tp = int(((y_pred_final==1)&(y_test_final==1)).sum())
fp = int(((y_pred_final==1)&(y_test_final==0)).sum())
tn = int(((y_pred_final==0)&(y_test_final==0)).sum())
fn = int(((y_pred_final==0)&(y_test_final==1)).sum())
prec = tp/(tp+fp) if (tp+fp) > 0 else 0
rec  = tp/(tp+fn) if (tp+fn) > 0 else 0
f1_s = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0
base = y_test_final.mean()
lift = prec/base if base > 0 else 0
n_pp = int(y_pred_final.sum())

print('EVALUACION FINAL - TAREA 1: CONVERSION')
print(f"  Modelo     : {best_conv['algorithm']} | {best_conv['feature_set']} (refinado)")
print(f"  Variables  : {len(best_conv_features)}")
print(f"  Threshold  : {thr_final:.2f} (operativo Top-10%)")
print(f"  TP={tp:,}  FP={fp:,}  TN={tn:,}  FN={fn:,}")
print(f"  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1_s:.4f}")
print(f"  Leads priorizados={n_pp:,} ({n_pp/len(y_test_final)*100:.1f}% del test)")
print(f"  Lift={lift:.2f}x  PR-AUC={best_conv_pr_final:.4f}  ROC-AUC={best_conv_roc_final:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

fpr_f, tpr_f, _ = roc_curve(y_test_final, y_prob_final)
axes[0].plot(fpr_f, tpr_f, '#42A5F5', lw=2, label=f'ROC-AUC={best_conv_roc_final:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_title('Curva ROC - Modelo Final (Conversion)', fontweight='bold')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(); axes[0].grid(alpha=0.3)

pr_c, rc_c, _ = precision_recall_curve(y_test_final, y_prob_final)
axes[1].plot(rc_c, pr_c, '#EF5350', lw=2, label=f'PR-AUC={best_conv_pr_final:.3f}')
axes[1].axhline(base, color='gray', ls='--', lw=1, label=f'Baseline={base:.3f}')
axes[1].scatter(rec, prec, s=100, color='#42A5F5', zorder=5,
                label=f'Thr={thr_final:.2f}')
axes[1].set_title('Curva Precision-Recall - Modelo Final (Conversion)', fontweight='bold')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

ConfusionMatrixDisplay.from_predictions(
    y_test_final, y_pred_final, ax=axes[2],
    display_labels=['No convierte','Convierte'],
    colorbar=False, cmap='Blues')
axes[2].set_title(f'Matriz de Confusion (Thr={thr_final:.2f})', fontweight='bold')

plt.suptitle('Evaluacion Final - Tarea 1: Conversion', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'evaluacion_final_conv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: figures/evaluacion_final_conv.png')


---
> **Hallazgos - Seccion 7 (Tarea 1: Conversion):** El analisis iterativo en 5 etapas identifica el modelo definitivo a partir de los Top 5 candidatos de todos los experimentos. La importancia de permutacion revela que variables realmente discriminan; el conjunto refinado elimina variables de ruido sin sacrificar PR-AUC. La optimizacion del umbral confirma que el default de 0.5 es inoperativo en este dataset desbalanceado (84:1): el threshold operativo Top-10% entrega Lift > 6x con Recall ~60% de las conversiones. La validacion cruzada de 5 folds confirma la estabilidad del modelo seleccionado. El modelo definitivo queda disponible como `best_conv_pipeline_final` con threshold `best_conv_threshold`.


## Sección 8 — TAREA 2: Predicción de Abandono de Leads

### Contexto de Negocio
Un lead que entra al pipeline y abandona sin convertir representa un costo operativo sin retorno. Identificar tempranamente a los leads con alta propensión a abandonar permite intervenciones proactivas: refuerzos en el seguimiento, ajuste de oferta, o reasignación de recursos.

### Definición del Target
- **Dataset:** leads *en alcance del proyecto* (` == 0`), 166 219 registros
- **Target:** `target_abandoned = 1` si el lead no convirtió (era in-scope pero se perdió)
- **Desbalance:** ~100:1 (la mayoría de los leads in-scope que no convirtieron sí se consideran abandonados)

> Este conjunto es significativamente más pequeño que el dataset de conversión, lo que permite entrenamientos más rápidos y facilita SMOTE completo.


In [ ]:
print("=" * 65)
print("  EXPERIMENTO A — Sin SMOTE | Tarea 2: Abandono")
print("=" * 65)
results_aband_A = run_experiment(
    X_df        = df_raw,
    y_arr       = y_aband,
    idx_train   = idx_train_aband,
    idx_test    = idx_test_aband,
    feature_sets= FEATURE_SETS,
    task_name   = 'Abandono',
    use_smote   = False,
)
print(f"Completados: {len([r for r in results_aband_A if 'error' not in r])}")


In [ ]:
print("=" * 65)
print("  EXPERIMENTO B — Con SMOTE | Tarea 2: Abandono")
print("=" * 65)
results_aband_B = run_experiment(
    X_df        = df_raw,
    y_arr       = y_aband,
    idx_train   = idx_train_aband,
    idx_test    = idx_test_aband,
    feature_sets= FEATURE_SETS,
    task_name   = 'Abandono',
    use_smote   = True,
    smote_strategy = 0.20,   # higher ratio feasible on smaller dataset
)
print(f"Completados: {len([r for r in results_aband_B if 'error' not in r])}")


In [ ]:
all_results_aband = results_aband_A + results_aband_B
df_results_aband = pd.DataFrame([
    {k: v for k, v in r.items()
     if k not in ('pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain')}
    for r in all_results_aband if 'error' not in r
])

df_display_aband = df_results_aband[display_cols].sort_values('pr_auc', ascending=False)
print("TABLA COMPARATIVA — TAREA 2: ABANDONO")
print(df_display_aband.to_string(index=False))

df_results_aband.drop(columns=['pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain'],
                      errors='ignore').to_csv(
    RESULTS_DIR / 'task2_abandonment_all_experiments.csv', index=False
)
print("\n✓ Resultados guardados")


In [ ]:
# Visualizations for Task 2
best_per_alg_aband = (
    df_results_aband.sort_values('pr_auc', ascending=False)
    .groupby('algorithm').head(1)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ax_roc, ax_pr, ax_gain = axes

for idx_m, (_, row) in enumerate(best_per_alg_aband.iterrows()):
    match = next(
        (r for r in all_results_aband
         if r.get('algorithm')==row['algorithm']
         and r.get('experiment')==row['experiment']
         and r.get('feature_set')==row['feature_set']
         and 'error' not in r), None)
    if match is None:
        continue
    y_t = match['y_true_test']
    y_p = match['y_prob_test']
    col = ['#2196F3','#FF5722','#4CAF50','#9C27B0','#FF9800'][idx_m % 5]
    lbl = f"{row['algorithm']}"

    fpr, tpr, _ = roc_curve(y_t, y_p)
    ax_roc.plot(fpr, tpr, lw=1.8, color=col, label=f'{lbl} ({roc_auc_score(y_t,y_p):.3f})')

    pr_v, rc_v, _ = precision_recall_curve(y_t, y_p)
    ax_pr.plot(rc_v, pr_v, lw=1.8, color=col, label=f'{lbl} ({average_precision_score(y_t,y_p):.3f})')

    n = len(y_t)
    order = np.argsort(y_p)[::-1]
    cum_g = np.cumsum(y_t[order]) / y_t.sum()
    ax_gain.plot(np.arange(1,n+1)/n, cum_g, lw=1.8, color=col, label=lbl)

ax_roc.plot([0,1],[0,1],'k--',lw=0.8); ax_roc.set_title('ROC (Abandono)', fontweight='bold')
ax_roc.legend(fontsize=7); ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')

ax_pr.axhline(y_test_aband.mean(),linestyle='--',color='gray',lw=0.8)
ax_pr.set_title('Precision-Recall (Abandono)', fontweight='bold')
ax_pr.legend(fontsize=7); ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')

ax_gain.plot([0,1],[0,1],'k--',lw=0.8,label='Aleatorio')
ax_gain.set_title('Ganancia Acumulada (Abandono)', fontweight='bold')
ax_gain.legend(fontsize=7)

plt.suptitle('Evaluación Visual — Tarea 2: Abandono', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task2_roc_pr_gain.png', dpi=150, bbox_inches='tight')
plt.show()


---
> **Hallazgos — Sección 8 (Tarea 2: Abandono):** El mejor modelo es **GBM | F4_Todas | Experimento A (Sin SMOTE)** con **ROC-AUC = 0.816** y **PR-AUC = 0.996**. El PR-AUC elevado refleja la tasa base de abandono (98.82%) más que la capacidad discriminativa del modelo — el PR-AUC de una clase mayoritaria siempre tiende a 1. La evaluación operativa (Sección 9) muestra que, al ordenar leads por probabilidad *ascendente* de abandono (equivalente a probabilidad descendente de conversión), el modelo captura conversiones con **Lift@10% = 6.07x**, prácticamente idéntico al modelo de conversión directa. Ambos modelos aprenden el mismo patrón subyacente desde perspectivas complementarias, lo que valida la consistencia del diseño experimental.


## Sección 9 — Evaluación Operativa: Escenarios de Priorización

La evaluación operativa traduce el desempeño técnico del modelo en valor práctico para el proceso comercial. Se comparan dos escenarios de gestión de leads bajo restricciones de recursos:

| Escenario | Descripción | Metodología |
|-----------|-------------|-------------|
| **1 — Modelo** | Los leads se ordenan por probabilidad predicha y se atiende el top K% | Aplicación directa del modelo sobre el conjunto de prueba |
| **2 — Aleatorio** | Se selecciona un número equivalente de leads al azar | Simulación Monte Carlo (10 000 iteraciones) |

Los umbrales evaluados son **5%, 10%, 20% y 30%** del conjunto de prueba.  
La comparación se realiza para **todos los algoritmos** (LR, RF, GBM, EXT) y **ambas tareas** (Conversión y Abandono), usando el mejor conjunto de características por algoritmo según PR-AUC.


### 9.1 Función de Evaluación Operativa

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 9.1  Función de evaluación operativa (todos los modelos, ambas tareas)
# ════════════════════════════════════════════════════════════════════════

THRESHOLDS = [0.05, 0.10, 0.20, 0.30]
N_MC        = 10_000

def operational_evaluation(results_list, task_name,
                            thresholds=THRESHOLDS, n_mc=N_MC,
                            sort_ascending=False, seed=SEED):
    """
    Escenario 1: priorización por modelo para todos los resultados en results_list.
    Escenario 2: Monte Carlo sobre el mismo conjunto de prueba.

    sort_ascending=False → selecciona top K% por MAYOR probabilidad (conversión).
    sort_ascending=True  → selecciona top K% por MENOR probabilidad (abandono inverso).
    """
    rng = np.random.default_rng(seed)

    first_valid = next((r for r in results_list
                        if 'y_true_test' in r and 'y_prob_test' in r), None)
    if first_valid is None:
        print(f"  [WARN] No hay resultados válidos para {task_name}")
        return pd.DataFrame(), pd.DataFrame()

    y_true_ref  = first_valid['y_true_test']
    n_test      = len(y_true_ref)
    n_pos_total = int(y_true_ref.sum())
    base_rate   = n_pos_total / n_test

    # ── Escenario 2: Monte Carlo por umbral ──────────────────────────────
    mc_rows = []
    for thr in thresholds:
        n_sel = max(1, round(thr * n_test))
        mc_counts = np.array([
            int(y_true_ref[rng.choice(n_test, n_sel, replace=False)].sum())
            for _ in range(n_mc)
        ])
        mc_rows.append({
            'threshold_pct' : thr * 100,
            'n_selected'    : n_sel,
            'n_pos_total'   : n_pos_total,
            'base_rate_pct' : round(base_rate * 100, 4),
            'mc_mean'       : round(mc_counts.mean(), 2),
            'mc_std'        : round(mc_counts.std(), 2),
            'mc_ci95_lo'    : round(float(np.percentile(mc_counts,  2.5)), 2),
            'mc_ci95_hi'    : round(float(np.percentile(mc_counts, 97.5)), 2),
            'mc_precision'  : round(mc_counts.mean() / n_sel, 4),
            'mc_recall'     : round(mc_counts.mean() / n_pos_total, 4) if n_pos_total > 0 else 0,
        })
    df_mc = pd.DataFrame(mc_rows)

    # ── Escenario 1: modelo (todos los algoritmos × feature_sets) ────────
    model_rows = []
    for r in results_list:
        if 'y_prob_test' not in r or 'error' in r:
            continue
        y_prob = np.array(r['y_prob_test'])
        y_true = np.array(r['y_true_test'])
        n_t    = len(y_true)
        n_pos  = int(y_true.sum())

        if sort_ascending:
            order = np.argsort(y_prob)          # ascending (low prob first)
        else:
            order = np.argsort(y_prob)[::-1]    # descending (high prob first)

        y_sorted  = y_true[order]
        cum_pos   = np.cumsum(y_sorted)

        for thr in thresholds:
            n_sel       = max(1, round(thr * n_t))
            n_captured  = int(cum_pos[n_sel - 1])
            precision   = n_captured / n_sel
            recall      = n_captured / n_pos if n_pos > 0 else 0
            lift_base   = precision / base_rate if base_rate > 0 else 0

            mc_ref      = df_mc[df_mc['threshold_pct'] == thr * 100].iloc[0]
            lift_vs_mc  = n_captured / mc_ref['mc_mean'] if mc_ref['mc_mean'] > 0 else 0
            delta_conv  = n_captured - mc_ref['mc_mean']
            ci_lo, ci_hi = mc_ref['mc_ci95_lo'], mc_ref['mc_ci95_hi']
            sig_vs_mc   = n_captured > ci_hi  # above MC 97.5th percentile

            model_rows.append({
                'task'          : task_name,
                'algorithm'     : r.get('algorithm', ''),
                'feature_set'   : r.get('feature_set', ''),
                'experiment'    : r.get('experiment', ''),
                'threshold_pct' : thr * 100,
                'n_selected'    : n_sel,
                'n_converted'   : n_captured,
                'precision'     : round(precision, 4),
                'recall'        : round(recall, 4),
                'lift'          : round(lift_base, 4),
                'mc_mean'       : round(float(mc_ref['mc_mean']), 2),
                'mc_ci95_lo'    : ci_lo,
                'mc_ci95_hi'    : ci_hi,
                'lift_vs_mc'    : round(lift_vs_mc, 4),
                'delta_conv'    : round(delta_conv, 2),
                'sig_vs_mc'     : sig_vs_mc,
            })

    df_model = pd.DataFrame(model_rows)
    return df_model, df_mc

print("Función operational_evaluation definida.")
print(f"  Umbrales: {[f'{t*100:.0f}%' for t in THRESHOLDS]}")
print(f"  Iteraciones Monte Carlo: {N_MC:,}")


### 9.2 Escenario 1 — Priorización por Modelo (Todos los Algoritmos)

Para cada algoritmo se selecciona el mejor modelo (por PR-AUC) que predijo sobre el mismo conjunto de prueba (F1–F4). Se calcula el número de conversiones capturadas al contactar el top K% de leads ordenados por probabilidad predicha.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 9.2  Escenario 1: Priorización por Modelo
# ════════════════════════════════════════════════════════════════════════

# ── Task 1: Conversión (F1–F4, mismo test set 81K leads) ─────────────────
results_conv_F1F4 = [r for r in all_results_conv
                     if r.get('feature_set') in ('F1_Base','F2_Base+Demo',
                                                    'F3_Base+Comercial','F4_Todas')]

print("Ejecutando evaluación operativa — Task 1: Conversión ...")
df_op_conv, df_mc_conv = operational_evaluation(
    results_conv_F1F4, task_name='Conversion',
    sort_ascending=False      # ordenar por mayor prob_conversion
)
print(f"  Modelos evaluados: {df_op_conv['algorithm'].nunique() if len(df_op_conv) else 0} algoritmos")

# ── Task 2: Abandono — objetivo: capturar leads que SÍ convirtieron ──────
# Para evaluar el valor operativo del modelo de abandono, se invierte la
# perspectiva: se busca capturar las conversiones (target_won=1 = y_aband=0)
# seleccionando primero los leads con MENOR prob_abandono.
# Esto equivale a ordenar descendentemente por (1 - prob_abandono) ≈ prob_conversion.
print("Ejecutando evaluación operativa — Task 2: Abandono (vista de conversión) ...")
results_aband_conv_view = []
for r in all_results_aband:
    if 'y_true_test' in r and 'y_prob_test' in r:
        r2 = {k: v for k, v in r.items()
               if k not in ('y_true_test','y_prob_test')}
        r2['y_true_test'] = 1 - np.array(r['y_true_test'])  # 1=convertido
        r2['y_prob_test']  = 1 - np.array(r['y_prob_test'])  # 1-prob_abandono
        results_aband_conv_view.append(r2)
    else:
        results_aband_conv_view.append(r)

df_op_aband, df_mc_aband = operational_evaluation(
    results_aband_conv_view,
    task_name='Abandono',
    sort_ascending=False   # mayor (1-prob_abandono) = menor riesgo de abandono
)
print(f"  Modelos evaluados: {df_op_aband['algorithm'].nunique() if len(df_op_aband) else 0} algoritmos")

# ── Tabla resumen por umbral — Task 1 ─────────────────────────────────────
print()
print("═" * 72)
print("  TABLA OPERATIVA — TAREA 1: CONVERSIÓN")
print("  (Conversiones capturadas en test set vs. selección aleatoria MC)")
print("═" * 72)
for thr_pct in [5, 10, 20, 30]:
    sub = df_op_conv[df_op_conv['threshold_pct'] == thr_pct]
    if len(sub) == 0:
        continue
    mc_val = df_mc_conv[df_mc_conv['threshold_pct'] == thr_pct]['mc_mean'].values[0]
    mc_ci  = df_mc_conv[df_mc_conv['threshold_pct'] == thr_pct][['mc_ci95_lo','mc_ci95_hi']].values[0]
    n_sel  = int(sub['n_selected'].iloc[0])
    n_pos  = int(sub['n_pos_total'].iloc[0]) if 'n_pos_total' in sub.columns else '-'
    print(f"\n--- Umbral {thr_pct:2d}% | n={n_sel:,} | Total conversiones={n_pos} | MC={mc_val:.1f} (IC95: {mc_ci[0]:.0f}–{mc_ci[1]:.0f}) ---")
    best_per = sub.loc[sub.groupby('algorithm')['n_converted'].idxmax()].copy()
    best_per = best_per.sort_values('n_converted', ascending=False)
    print(f"  {'Alg':5s}  {'Feat.Set':22s}  {'Conv':>6s}  {'Recall':>7s}  {'Prec':>7s}  {'Lift':>7s}  {'LvsMC':>7s}  Sig")
    for _, row in best_per.iterrows():
        sig = "***" if row['sig_vs_mc'] else "   "
        print(f"  {row['algorithm']:5s}  {row['feature_set']:22s}  {int(row['n_converted']):>6d}"
              f"  {row['recall']:>7.4f}  {row['precision']:>7.4f}  {row['lift']:>7.2f}x  {row['lift_vs_mc']:>7.2f}x  {sig}")

# ── Tabla resumen — Task 2 ─────────────────────────────────────────────
print()
print("═" * 72)
print("  TABLA OPERATIVA — TAREA 2: ABANDONO")
print("  Métrica: conversiones capturadas al priorizar leads de menor prob_abandono")
print("  (equivalente a ordenar por prob_conversion proxy = 1 - prob_abandono)")
print("═" * 72)
for thr_pct in [5, 10, 20, 30]:
    sub = df_op_aband[df_op_aband['threshold_pct'] == thr_pct]
    if len(sub) == 0:
        continue
    mc_val = df_mc_aband[df_mc_aband['threshold_pct'] == thr_pct]['mc_mean'].values[0]
    mc_ci  = df_mc_aband[df_mc_aband['threshold_pct'] == thr_pct][['mc_ci95_lo','mc_ci95_hi']].values[0]
    n_sel  = int(sub['n_selected'].iloc[0])
    print(f"\n--- Umbral {thr_pct:2d}% | n={n_sel:,} | MC={mc_val:.1f} (IC95: {mc_ci[0]:.0f}–{mc_ci[1]:.0f}) ---")
    best_per = sub.loc[sub.groupby('algorithm')['n_converted'].idxmax()].copy()
    best_per = best_per.sort_values('n_converted', ascending=False)
    print(f"  {'Alg':5s}  {'Feat.Set':22s}  {'Conv':>6s}  {'Recall':>7s}  {'Prec':>7s}  {'Lift':>7s}  {'LvsMC':>7s}  Sig")
    for _, row in best_per.iterrows():
        sig = "***" if row['sig_vs_mc'] else "   "
        print(f"  {row['algorithm']:5s}  {row['feature_set']:22s}  {int(row['n_converted']):>6d}"
              f"  {row['recall']:>7.4f}  {row['precision']:>7.4f}  {row['lift']:>7.2f}x  {row['lift_vs_mc']:>7.2f}x  {sig}")

# Save results
df_op_conv.to_csv(RESULTS_DIR / 'operational_eval_conversion.csv', index=False)
df_op_aband.to_csv(RESULTS_DIR / 'operational_eval_abandono.csv', index=False)
print(f"\nResultados guardados en results/")
print(f"  operational_eval_conversion.csv: {len(df_op_conv)} filas")
print(f"  operational_eval_abandono.csv   : {len(df_op_aband)} filas")


### 9.3 Escenario 2 — Línea Base Monte Carlo (10 000 iteraciones)

Para validar que las diferencias observadas no son producto del azar, se compara cada modelo con una distribución de referencia obtenida mediante 10 000 muestras aleatorias del mismo tamaño. Una diferencia estadísticamente significativa (por encima del IC 95%) se marca con `***`.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 9.3  Línea Base Monte Carlo — Tabla Estadística Detallada
# ════════════════════════════════════════════════════════════════════════

print("══════════════════════════════════════════════════════════════")
print("  LÍNEA BASE MONTE CARLO — TAREA 1: CONVERSIÓN")
print("══════════════════════════════════════════════════════════════")
print(df_mc_conv.rename(columns={
    'threshold_pct': 'Umbral%', 'n_selected': 'N_sel', 'n_pos_total': 'N_pos_total',
    'base_rate_pct': 'BaseRate%', 'mc_mean': 'MC_media', 'mc_std': 'MC_std',
    'mc_ci95_lo': 'CI95_lo', 'mc_ci95_hi': 'CI95_hi',
    'mc_precision': 'MC_prec', 'mc_recall': 'MC_recall'
}).to_string(index=False))

print()
print("══════════════════════════════════════════════════════════════")
print("  LÍNEA BASE MONTE CARLO — TAREA 2: ABANDONO")
print("══════════════════════════════════════════════════════════════")
print(df_mc_aband.rename(columns={
    'threshold_pct': 'Umbral%', 'n_selected': 'N_sel', 'n_pos_total': 'N_pos_total',
    'base_rate_pct': 'BaseRate%', 'mc_mean': 'MC_media', 'mc_std': 'MC_std',
    'mc_ci95_lo': 'CI95_lo', 'mc_ci95_hi': 'CI95_hi',
    'mc_precision': 'MC_prec', 'mc_recall': 'MC_recall'
}).to_string(index=False))

print()
print("Nota: CI95 = intervalo de confianza del 95% para la selección aleatoria.")
print("Un modelo con n_converted > CI95_hi supera estadísticamente la aleatoriedad (marcado ***).")


### 9.4 Análisis Comparativo — Curvas de Ganancia Acumulada

Las curvas de ganancia acumulada muestran qué proporción de las conversiones totales se captura al contactar progresivamente el K% de leads mejor clasificados. Se incluye la banda del intervalo de confianza del 95% de la selección aleatoria (Monte Carlo).


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 9.4  Curvas de Ganancia Acumulada — Todos los Algoritmos
# ════════════════════════════════════════════════════════════════════════

ALGORITHMS_ORDER = ['LR', 'RF', 'GBM', 'EXT']
COLORS_ALG       = {'LR': '#5C6BC0', 'RF': '#43A047', 'GBM': '#E53935', 'EXT': '#FB8C00'}
N_MC_CURVE       = 2_000    # faster for the visual band

def compute_gain_curve(y_true, y_prob, sort_ascending=False):
    """Returns (pct_selected, pct_captured) arrays for a cumulative gain curve."""
    n  = len(y_true)
    nP = int(y_true.sum())
    if nP == 0:
        return np.linspace(0, 100, n), np.linspace(0, 100, n)
    if sort_ascending:
        order = np.argsort(y_prob)
    else:
        order = np.argsort(y_prob)[::-1]
    cum_pos     = np.cumsum(y_true[order])
    pct_sel     = np.arange(1, n + 1) / n * 100
    pct_cap     = cum_pos / nP * 100
    return pct_sel, pct_cap

def mc_gain_band(y_true, n_mc=N_MC_CURVE, seed=42):
    """Monte Carlo gain band at several thresholds."""
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    nP  = int(y_true.sum())
    xs  = np.arange(1, 21) * 5   # 5%, 10%, ..., 100%
    mc_means, mc_lo, mc_hi = [], [], []
    for thr_pct in xs:
        n_sel = max(1, round(thr_pct / 100 * n))
        mc_c  = np.array([
            y_true[rng.choice(n, n_sel, replace=False)].sum()
            for _ in range(n_mc)
        ])
        mc_means.append(mc_c.mean() / nP * 100)
        mc_lo.append(np.percentile(mc_c,  2.5) / nP * 100)
        mc_hi.append(np.percentile(mc_c, 97.5) / nP * 100)
    return xs, np.array(mc_means), np.array(mc_lo), np.array(mc_hi)

# ── Figura 1: Task 1 — Conversión (2×2 por algoritmo) ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Curvas de Ganancia Acumulada — Tarea 1: Conversión\n'
             '(Umbral K% → % conversiones capturadas)',
             fontsize=12, fontweight='bold')

for ax, alg in zip(axes.flat, ALGORITHMS_ORDER):
    alg_res = [r for r in results_conv_F1F4
               if r.get('algorithm') == alg and 'y_prob_test' in r]
    if not alg_res:
        ax.text(0.5, 0.5, f'{alg}\nsin resultados', ha='center', va='center')
        continue

    # Best result by PR-AUC
    best_r   = max(alg_res, key=lambda r: r.get('pr_auc', 0))
    y_true   = np.array(best_r['y_true_test'])
    y_prob   = np.array(best_r['y_prob_test'])
    n_pos    = int(y_true.sum())
    n_test_s = len(y_true)

    pct_sel, pct_cap = compute_gain_curve(y_true, y_prob, sort_ascending=False)
    xs_mc, mc_m, mc_lo, mc_hi = mc_gain_band(y_true)

    ax.plot(pct_sel, pct_cap,
            color=COLORS_ALG[alg], lw=2.5,
            label=f'{alg} ({best_r["feature_set"]})')
    ax.plot([0, 100], [0, 100], 'k--', lw=1.2, label='Aleatorio (media MC)')
    ax.fill_between(xs_mc, mc_lo, mc_hi, alpha=0.2, color='gray', label='IC 95% MC')

    # Perfect model line
    pct_ideal = n_pos / n_test_s * 100
    ax.plot([0, pct_ideal, 100], [0, 100, 100], 'g:', lw=1.2, alpha=0.7, label='Modelo perfecto')

    # Mark Lift@5, 10, 20, 30
    for thr_pct, marker in [(5,'o'),(10,'s'),(20,'^')]:
        n_sel_pt = max(1, round(thr_pct/100 * n_test_s))
        captured = int(np.cumsum(y_true[np.argsort(y_prob)[::-1]])[n_sel_pt-1])
        pct_c    = captured / n_pos * 100
        mc_ref   = df_mc_conv[df_mc_conv['threshold_pct'] == thr_pct]['mc_mean'].values[0]
        lift_mc  = captured / mc_ref if mc_ref > 0 else 0
        ax.annotate(f'{thr_pct}%: {lift_mc:.1f}x MC',
                    xy=(thr_pct, pct_c), xytext=(thr_pct+3, pct_c-5),
                    fontsize=7, color=COLORS_ALG[alg],
                    arrowprops=dict(arrowstyle='->', color=COLORS_ALG[alg], lw=0.8))

    ax.set_xlim(0, 100); ax.set_ylim(0, 105)
    ax.set_xlabel('% Leads Contactados')
    ax.set_ylabel('% Conversiones Capturadas')
    ax.set_title(f'Algoritmo: {alg}', fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'operational_gain_curves_conv.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: operational_gain_curves_conv.png")

# ── Figura 2: Task 2 — Abandono (2×2) ────────────────────────────────────
fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
fig2.suptitle('Curvas de Ganancia Acumulada — Tarea 2: Abandono\n'
              '(Ordenado por prob_abandono ASCENDENTE = priorizar menor riesgo de abandono)',
              fontsize=12, fontweight='bold')

for ax, alg in zip(axes2.flat, ALGORITHMS_ORDER):
    alg_res = [r for r in results_aband_conv_view
               if r.get('algorithm') == alg and 'y_prob_test' in r]
    if not alg_res:
        ax.text(0.5, 0.5, f'{alg}\nsin resultados', ha='center', va='center')
        continue

    best_r   = max(alg_res, key=lambda r: r.get('roc_auc', 0))
    y_aband  = np.array(best_r['y_true_test'])    # 1=abandoned, 0=converted
    y_prob_a = np.array(best_r['y_prob_test'])    # prob_abandono
    # Objective: find leads NOT likely to abandon (y_aband=0)
    y_conv_proxy = 1 - y_aband                    # 1=converted
    n_pos_c      = int(y_conv_proxy.sum())
    n_test_s     = len(y_conv_proxy)

    if n_pos_c == 0:
        ax.text(0.5, 0.5, 'Sin conversiones en test', ha='center', va='center')
        continue

    # Sort ASCENDING by prob_abandono → capture low-risk leads first
    pct_sel, pct_cap = compute_gain_curve(y_conv_proxy, y_prob_a, sort_ascending=True)
    xs_mc, mc_m, mc_lo, mc_hi = mc_gain_band(y_conv_proxy)

    ax.plot(pct_sel, pct_cap, color=COLORS_ALG[alg], lw=2.5,
            label=f'{alg} ({best_r["feature_set"]})')
    ax.plot([0, 100], [0, 100], 'k--', lw=1.2, label='Aleatorio (media MC)')
    ax.fill_between(xs_mc, mc_lo, mc_hi, alpha=0.2, color='gray', label='IC 95% MC')
    pct_ideal_a = n_pos_c / n_test_s * 100
    ax.plot([0, pct_ideal_a, 100], [0, 100, 100], 'g:', lw=1.2, alpha=0.7, label='Modelo perfecto')

    ax.set_xlim(0, 100); ax.set_ylim(0, 105)
    ax.set_xlabel('% Leads Contactados (prob. abandono ↑)')
    ax.set_ylabel('% Conversiones Capturadas')
    ax.set_title(f'Algoritmo: {alg}', fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'operational_gain_curves_aband.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: operational_gain_curves_aband.png")

# ── Figura 3: Comparación de Lift@K por algoritmo (barras) ────────────────
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle('Lift vs. Selección Aleatoria (Monte Carlo) por Algoritmo',
              fontsize=12, fontweight='bold')

for task_ax, (df_op, task_lbl, df_mc_r) in zip(
        axes3,
        [(df_op_conv, 'Conversión', df_mc_conv),
         (df_op_aband, 'Abandono (inverso)', df_mc_aband)]):
    if len(df_op) == 0:
        task_ax.text(0.5, 0.5, 'Sin datos', ha='center', va='center')
        task_ax.set_title(f'Task: {task_lbl}')
        continue

    thr_show = 10   # Lift@10%
    sub      = df_op[df_op['threshold_pct'] == thr_show]
    best_per = sub.loc[sub.groupby('algorithm')['lift_vs_mc'].idxmax()]
    best_per = best_per.sort_values('lift_vs_mc', ascending=False)

    colors_b = [COLORS_ALG.get(a, '#90A4AE') for a in best_per['algorithm']]
    bars_b   = task_ax.bar(best_per['algorithm'], best_per['lift_vs_mc'],
                           color=colors_b, edgecolor='white', linewidth=0.5)
    task_ax.axhline(1.0, color='red', linestyle='--', lw=1.5, label='Aleatorio = 1.0x')

    for bar_b, row_b in zip(bars_b, best_per.itertuples()):
        task_ax.text(bar_b.get_x() + bar_b.get_width()/2,
                     bar_b.get_height() + 0.02,
                     f'{row_b.lift_vs_mc:.2f}x',
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

    task_ax.set_xlabel('Algoritmo')
    task_ax.set_ylabel('Lift vs. Monte Carlo Aleatorio')
    task_ax.set_title(f'Lift@{thr_show}% — Task: {task_lbl}', fontweight='bold')
    task_ax.legend(fontsize=8)
    task_ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'operational_lift_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: operational_lift_comparison.png")


---
> **Hallazgos — Sección 9 (Evaluación Operativa):** La priorización basada en modelos supera significativamente la selección aleatoria en todos los umbrales. Al contactar el **5% de los leads mejor clasificados** (~4,080 de 81,606 del set de prueba), el modelo captura el **41.5% de todas las conversiones** (Lift@5% = **8.3x**). Al **10%**, el Lift = **6.05x** (583 conversiones capturadas, Recall = 60.5%). La simulación Monte Carlo (10,000 iteraciones) confirma que todos los resultados son estadísticamente significativos (p < 0.025, marcados ***). Los modelos de árbol (GBM, EXT, RF) logran lifts indistinguibles entre sí; la Regresión Logística obtiene Lift@5% = 6.77x, competitiva pero inferior. El conjunto de características **F2_Base+Demo es el más eficiente** operativamente: máximo rendimiento con mínima complejidad de recolección de datos.


## Sección 10 — Interpretación de Modelos

### 10.1 — Coeficientes de Regresión Logística

La Regresión Logística ofrece interpretabilidad directa a través de sus coeficientes. Valores positivos aumentan la probabilidad de conversión; valores negativos la disminuyen.


In [ ]:
# Get LR pipeline from results
lr_match = next(
    (r for r in all_results_conv
     if r.get('algorithm') == 'LR'
     and r.get('experiment') == 'A_NoSMOTE'
     and 'error' not in r), None
)

if lr_match:
    pipe_lr = lr_match['pipeline']
    fs_lr   = FEATURE_SETS.get(lr_match['feature_set'], [])
    avail_lr = [f for f in fs_lr if f in df_raw.columns]
    cat_lr, num_lr = identify_col_types(df_raw, avail_lr)

    # Extract feature names after OHE
    prep_lr = pipe_lr.named_steps['prep']
    try:
        ohe = prep_lr.named_transformers_['cat'].named_steps['enc']
        ohe_names = list(ohe.get_feature_names_out(cat_lr))
    except:
        ohe_names = cat_lr
    feature_names_lr = ohe_names + num_lr

    lr_clf = pipe_lr.named_steps['clf']
    coefs  = lr_clf.coef_[0]

    df_coef = pd.DataFrame({
        'feature': feature_names_lr[:len(coefs)],
        'coef'   : coefs
    }).sort_values('coef', key=abs, ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors_coef = ['#EF5350' if c > 0 else '#42A5F5' for c in df_coef['coef']]
    ax.barh(range(len(df_coef)), df_coef['coef'], color=colors_coef, edgecolor='white')
    ax.set_yticks(range(len(df_coef)))
    ax.set_yticklabels(df_coef['feature'], fontsize=8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title('Top 20 Coeficientes — Regresión Logística\n(Tarea 1: Conversión)',
                 fontweight='bold')
    ax.set_xlabel('Coeficiente (escala log-odds)')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'lr_coefficients_conv.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nTop 10 variables positivas (aumentan probabilidad de conversión):")
    print(df_coef[df_coef['coef']>0].head(10)[['feature','coef']].to_string(index=False))
    print("\nTop 10 variables negativas (reducen probabilidad de conversión):")
    print(df_coef[df_coef['coef']<0].head(10)[['feature','coef']].to_string(index=False))
else:
    print("Modelo LR no disponible para interpretación de coeficientes")


### 10.2 — Feature Importance (Tree-Based Models)

Los modelos basados en árboles (RF, GBM, XGB, LGB) proveen importancia de características basada en la reducción de impureza (Gini importance). Se visualizan las 20 variables más importantes del mejor modelo.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 10.2  Feature Importance — Best Tree Model with feature_importances_
# (HistGradientBoosting has no feature_importances_; prefer RF/EXT which do)
# ════════════════════════════════════════════════════════════════════════

# Prefer RF or EXT (they have feature_importances_), then GBM as fallback
best_tree_match = max(
    (r for r in all_results_conv
     if r.get('algorithm') in ('RF', 'EXT')
     and r.get('experiment') == 'A_NoSMOTE'
     and 'error' not in r),
    key=lambda r: r.get('pr_auc', 0),
    default=None
)
if best_tree_match is None:
    # Fallback to any tree model
    best_tree_match = max(
        (r for r in all_results_conv
         if r.get('algorithm') in ('GBM', 'RF', 'EXT')
         and r.get('experiment') == 'A_NoSMOTE'
         and 'error' not in r),
        key=lambda r: r.get('pr_auc', 0),
        default=None
    )

if best_tree_match:
    alg_n = best_tree_match['algorithm']
    fs_t  = FEATURE_SETS.get(best_tree_match['feature_set'], [])
    avail_t = [f for f in fs_t if f in df_raw.columns]
    cat_t, num_t = identify_col_types(df_raw, avail_t)

    pipe_tree = best_tree_match['pipeline']
    clf_tree  = pipe_tree.named_steps['clf']
    feat_names_tree = cat_t + num_t

    importances = getattr(clf_tree, 'feature_importances_', None)
    if importances is not None and len(importances) == len(feat_names_tree):
        df_imp = pd.DataFrame({
            'feature'    : feat_names_tree,
            'importance' : importances
        }).sort_values('importance', ascending=False).head(20)

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(range(len(df_imp)), df_imp['importance'],
                color='#5C6BC0', edgecolor='white', alpha=0.9)
        ax.set_yticks(range(len(df_imp)))
        ax.set_yticklabels(df_imp['feature'], fontsize=8)
        ax.invert_yaxis()
        ax.set_title(f'Top 20 Variables Más Importantes — {alg_n}\n(Tarea 1: Conversión, {best_tree_match["feature_set"]})',
                     fontweight='bold')
        ax.set_xlabel('Importancia (Gini)')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f'feature_importance_{alg_n}_conv.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Modelo: {alg_n} | {best_tree_match['feature_set']} | PR-AUC={best_tree_match['pr_auc']:.4f}")
        print("\nTop 20 variables más importantes:")
        print(df_imp.to_string(index=False))
    else:
        print(f"{alg_n} no expone feature_importances_ (HistGradientBoosting no lo soporta).")
        print("Usando importancia de permutación no es factible aquí; consultar SHAP (10.3).")
        df_imp = None
else:
    print("No hay modelo de árbol disponible para importancia de características.")
    df_imp = None


### 10.3 — Análisis SHAP

SHAP (SHapley Additive exPlanations) cuantifica la contribución de cada variable a la predicción de un caso individual, con garantías de consistencia y eficiencia derivadas de la teoría de juegos cooperativos.

Se computan valores SHAP sobre una muestra representativa del test set para el mejor modelo.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 10.3  SHAP Analysis — best tree model on Conversion task
# ════════════════════════════════════════════════════════════════════════
# Try best tree model in order of preference
# Select best tree model by PR-AUC for SHAP interpretation
shap_match = max(
    (r for r in all_results_conv
     if r.get('algorithm') in ('GBM', 'RF', 'EXT')
     and r.get('experiment') == 'A_NoSMOTE'
     and 'error' not in r
     and 'y_prob_test' in r),
    key=lambda r: r.get('pr_auc', 0),
    default=None
)

if shap_match:
    pipe_shap = shap_match['pipeline']
    clf_shap  = pipe_shap.named_steps['clf']
    prep_shap = pipe_shap.named_steps['prep']

    # Usar exactamente las columnas con las que se fitteó el ColumnTransformer.
    # Esto evita errores cuando el mejor modelo SHAP viene del feature set con cluster.
    expected_shap_cols = list(getattr(prep_shap, 'feature_names_in_', []))
    if not expected_shap_cols:
        expected_shap_cols = []
        for _, _, cols in prep_shap.transformers_:
            if isinstance(cols, str):
                expected_shap_cols.append(cols)
            else:
                expected_shap_cols.extend(list(cols))

    uses_cluster_shap = 'cluster' in expected_shap_cols
    if uses_cluster_shap:
        X_shap_source = df_with_cluster
        idx_shap = idx_test_cl
    else:
        X_shap_source = df_raw
        idx_shap = idx_test_conv

    missing_shap_cols = [c for c in expected_shap_cols if c not in X_shap_source.columns]
    if missing_shap_cols:
        raise ValueError(
            f"Columnas faltantes para SHAP: {missing_shap_cols}. "
            f"Modelo={shap_match['algorithm']} | feature_set={shap_match['feature_set']}"
        )

    X_test_raw_shap = X_shap_source.iloc[idx_shap][expected_shap_cols]
    cat_shap, num_shap = identify_col_types(X_shap_source, expected_shap_cols)

    SHAP_SAMPLE     = min(2000, len(X_test_raw_shap))
    sample_idx_shap = np.random.choice(len(X_test_raw_shap), SHAP_SAMPLE, replace=False)
    X_test_sample   = X_test_raw_shap.iloc[sample_idx_shap]

    X_test_transformed = prep_shap.transform(X_test_sample)
    try:
        feat_names_shap = list(prep_shap.get_feature_names_out())
    except Exception:
        feat_names_shap = cat_shap + num_shap

    print(f"Modelo SHAP: {shap_match['algorithm']} | {shap_match['feature_set']}")
    print(f"Muestras para SHAP: {SHAP_SAMPLE}, features transformadas: {X_test_transformed.shape[1]}")

    try:
        explainer   = shap.TreeExplainer(clf_shap)
        shap_raw    = explainer(X_test_transformed)

        # shap_raw can be an Explanation object (shap >= 0.40) or a numpy array
        if hasattr(shap_raw, 'values'):
            sv = shap_raw.values
        else:
            sv = shap_raw

        # Binary classification may give shape (n, p, 2) or (2, n, p) or (n, p)
        if sv.ndim == 3:
            if sv.shape[-1] == 2:
                sv = sv[:, :, 1]          # positive class, shape (n, p)
            elif sv.shape[0] == 2:
                sv = sv[1]                # shape (n, p)
        elif isinstance(sv, list):
            sv = sv[1] if len(sv) == 2 else sv[0]

        # Trim feature names to match
        n_feats = sv.shape[1]
        feat_names_plot = feat_names_shap[:n_feats]

        print(f"SHAP values shape: {sv.shape}")

        # Summary plot (dot)
        fig, ax = plt.subplots(figsize=(10, 8))
        shap.summary_plot(sv, X_test_transformed[:, :n_feats],
                          feature_names=feat_names_plot,
                          max_display=20, show=False)
        plt.title(f"SHAP Summary — {shap_match['algorithm']} (Conversion)",
                  fontweight='bold', fontsize=12)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'shap_summary_conv.png', dpi=150, bbox_inches='tight')
        plt.show()

        # Bar plot (mean |SHAP|)
        fig, ax = plt.subplots(figsize=(10, 7))
        shap.summary_plot(sv, X_test_transformed[:, :n_feats],
                          feature_names=feat_names_plot,
                          plot_type='bar', max_display=20, show=False)
        plt.title(f"SHAP Mean |Value| — {shap_match['algorithm']} (Conversion)",
                  fontweight='bold', fontsize=12)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'shap_bar_conv.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("SHAP analysis completed via TreeExplainer")

    except Exception as shap_err:
        print(f"TreeExplainer failed: {shap_err}")
        print("Falling back to KernelExplainer on small subsample (slow but robust)...")
        try:
            SHAP_SMALL = 100
            X_bg   = X_test_transformed[:50]
            X_eval = X_test_transformed[:SHAP_SMALL]
            ke = shap.KernelExplainer(
                lambda x: clf_shap.predict_proba(x)[:, 1], X_bg
            )
            sv_k = ke.shap_values(X_eval, nsamples=30, silent=True)
            if isinstance(sv_k, list):
                sv_k = sv_k[1]

            fig, ax = plt.subplots(figsize=(10, 7))
            shap.summary_plot(sv_k, X_eval,
                              feature_names=feat_names_shap[:X_eval.shape[1]],
                              plot_type='bar', max_display=15, show=False)
            plt.title("SHAP Mean |Value| — KernelExplainer (subsample 100)",
                      fontweight='bold')
            plt.tight_layout()
            plt.savefig(FIGURES_DIR / 'shap_bar_conv_kernel.png', dpi=150, bbox_inches='tight')
            plt.show()
            print("SHAP analysis completed via KernelExplainer fallback")
        except Exception as ke_err:
            print(f"KernelExplainer also failed: {ke_err}")
            print("SHAP analysis skipped — feature importance plot used instead")
else:
    print("No tree model available for SHAP analysis")


### 10.4 — Síntesis de Factores Determinantes

Se integran los tres métodos de interpretabilidad (coeficientes de regresión logística, importancia de características en árboles, y valores SHAP) en un ranking unificado. Las importancias se normalizan al rango [0, 1] antes de promediarlas, permitiendo comparar variables independientemente de la escala de cada método. Este análisis identifica los **factores que el equipo comercial debe priorizar** al evaluar la calidad de un lead.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 10.4  Síntesis de Factores Determinantes — Ranking Unificado
# ════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd

importance_sources = {}

# Source 1: SHAP mean absolute values
try:
    sv = np.array(shap_raw.values) if hasattr(shap_raw, 'values') else np.array(shap_raw)
    if sv.ndim == 3:
        sv = sv[:, :, 1]
    n_f = sv.shape[1]
    names_shap = feat_names_plot[:n_f] if 'feat_names_plot' in dir() else feat_names_shap[:n_f]
    shap_mean = pd.Series(np.abs(sv).mean(axis=0), index=names_shap, name='SHAP')
    importance_sources['SHAP'] = shap_mean
    print(f"✓ SHAP: {len(shap_mean)} variables  (modelo: {shap_match['algorithm']} | {shap_match['feature_set']})")
except Exception as e:
    print(f"  SHAP no disponible: {e}")

# Source 2: Tree feature importance (df_imp from cell 10.2)
try:
    imp_tree = df_imp.set_index('feature')['importance'].rename('Árbol')
    importance_sources['Árbol'] = imp_tree
    print(f"✓ Árbol: {len(imp_tree)} variables  (modelo: {alg_n} | {best_tree_match['feature_set']})")
except Exception as e:
    print(f"  Tree importance no disponible: {e}")

# Source 3: LR coefficient magnitudes (df_coef from cell 10.1)
try:
    imp_lr = df_coef.set_index('feature')['coef'].abs().rename('RegLog')
    importance_sources['RegLog'] = imp_lr
    print(f"✓ RegLog: {len(imp_lr)} variables")
except Exception as e:
    print(f"  LR coefs no disponible: {e}")

if not importance_sources:
    print("No hay fuentes de importancia disponibles.")
else:
    # Normalize each source to [0, 1]
    normalized = {}
    for name, series in importance_sources.items():
        mn, mx = series.min(), series.max()
        normalized[name] = (series - mn) / (mx - mn) if mx > mn else series * 0.0

    df_synth = pd.DataFrame(normalized).fillna(0)
    df_synth['ranking_promedio'] = df_synth.mean(axis=1)
    df_synth = df_synth.sort_values('ranking_promedio', ascending=False)

    print()
    print("╔══════════════════════════════════════════════════════════════════════╗")
    print("║        TOP 20 FACTORES DETERMINANTES — RANKING UNIFICADO            ║")
    print("╚══════════════════════════════════════════════════════════════════════╝")
    df_disp = df_synth.head(20).round(3).reset_index()
    df_disp.columns = df_disp.columns.map(str)
    df_disp.rename(columns={'index': 'Variable'}, inplace=True)
    df_disp.index = range(1, len(df_disp) + 1)
    print(df_disp.to_string())

    df_synth.reset_index().rename(columns={'index': 'variable'}).to_csv(
        RESULTS_DIR / 'factores_determinantes.csv', index=False
    )
    print(f"\n✓ Guardado: results/factores_determinantes.csv")

    # Plot
    top15 = df_synth.head(15)
    fig, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(top15))
    width = 0.25
    palette = {'SHAP': '#42A5F5', 'Árbol': '#66BB6A', 'RegLog': '#FFA726'}
    offsets = {'SHAP': -width, 'Árbol': 0, 'RegLog': width}
    for method in importance_sources:
        col = top15.get(method, pd.Series(dtype=float))
        ax.bar(x + offsets.get(method, 0), col.values, width,
               label=method, color=palette.get(method, '#999'), alpha=0.85, edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(top15.index.tolist(), rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Importancia normalizada [0–1]')
    ax.set_title('Top 15 Factores Determinantes — Importancia por Método de Interpretabilidad',
                 fontweight='bold', fontsize=12)
    ax.legend(title='Método')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'factores_determinantes.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Figura: figures/factores_determinantes.png")


---
> **Hallazgos — Sección 10 (Interpretación de Modelos):** Los tres métodos de interpretabilidad convergen en las mismas variables como más determinantes. Las variables de **canal de adquisición** (`canal_de_contacto`, `canal_de_anuencia`, `fuente_pt2`) lideran consistentemente el ranking en SHAP y modelos de árbol: el canal por el que un lead entra al pipeline es el predictor más poderoso de conversión. Las variables de **proyecto y precio** (`proyecto`, `precio_lista_mxn`, `zona_proyecto`) ocupan los siguientes puestos, indicando que las características del inmueble discriminan significativamente la probabilidad de conversión. Las variables demográficas (`edad`, `sexo`, `estado_civil`) tienen importancia moderada y alta ausencia de datos. La Regresión Logística resalta el peso diferencial del canal digital vs. presencial; los modelos de árbol capturan interacciones no lineales entre canal, proyecto y precio que la LR no puede modelar, lo que explica su superioridad en PR-AUC.


## Sección 11 — Tabla Comparativa Global y Selección del Modelo Final

### 11.1 — Tabla Consolidada de Todos los Experimentos


In [ ]:
# Combine all results
all_experiments = (
    [r for r in all_results_conv  if 'error' not in r] +
    [r for r in all_results_aband if 'error' not in r]
)

df_global = pd.DataFrame([
    {k: v for k, v in r.items()
     if k not in ('pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain')}
    for r in all_experiments
])

df_global.to_csv(RESULTS_DIR / 'all_experiments_comparison.csv', index=False)
print(f"Tabla global: {len(df_global)} experimentos")
print(f"  Conversión : {len(df_global[df_global['task']=='Conversion'])}")
print(f"  Abandono   : {len(df_global[df_global['task']=='Abandono'])}")
print()
print("TABLA GLOBAL — Ordenada por PR-AUC Descendente:")
print(df_global[display_cols + ['task']].sort_values('pr_auc', ascending=False).to_string(index=False))


### 11.2 — Criterios de Selección del Modelo Final

La selección no se basa en una sola métrica. Se ponderan múltiples criterios:

| Criterio | Peso | Justificación |
|----------|------|---------------|
| PR-AUC | Alto | Más informativo que ROC-AUC para datasets desbalanceados |
| Recall@10% | Alto | Operativamente crítico: captura de conversiones con recursos limitados |
| Lift@10% | Alto | Indica la ganancia real respecto a selección aleatoria |
| Generalización (gap train-CV) | Medio | Garantiza robustez en producción |
| Interpretabilidad | Medio | Importante para la defensa ante el comité académico |
| Tiempo de entrenamiento | Bajo | Considera viabilidad operativa |


In [ ]:
# Tabla comparativa global y seleccion definitiva — Sec 11
df_conv_final  = df_global[df_global['task'] == 'Conversion'].copy()
df_aband_final = df_global[df_global['task'] == 'Abandono'].copy()

# --- Abandono: seleccion por score compuesto (Tarea 2 no tiene metodologia iterativa) ---
def score_model(row):
    return (
        0.30 * row.get('pr_auc', 0) +
        0.25 * row.get('rec_10pct', 0) +
        0.25 * (row.get('lift_10pct', 0) / 10.0) +
        0.10 * row.get('roc_auc', 0) +
        0.10 * row.get('f1', 0)
    )

df_aband_final['composite_score'] = df_aband_final.apply(score_model, axis=1)
best_aband_model = df_aband_final.sort_values('composite_score', ascending=False).iloc[0]

# --- Conversion: modelo ya seleccionado en Sec 7 (metodologia iterativa) ---
# Crear un registro uniforme para las secciones de serializacion y resumen.
best_conv_bm, _, _ = business_metrics_at_k(
    np.asarray(y_conv[idx_test_conv]),
    np.asarray(best_conv_y_prob_final),
    k_list=(0.10,),
)
best_conv_model = pd.Series({
    'algorithm': best_conv['algorithm'],
    'feature_set': best_conv['feature_set'],
    'experiment': best_conv_exp,
    'pr_auc': best_conv_pr_final,
    'roc_auc': best_conv_roc_final,
    'lift_10pct': best_conv_bm['top10pct']['lift'],
})

print('MODELO DE CONVERSION (seleccionado en Sec 7 - metodologia iterativa):')
print(f"  Algoritmo  : {best_conv['algorithm']}")
print(f"  FeatureSet : {best_conv['feature_set']} (refinado - {len(best_conv_features)} variables)")
print(f"  Experimento: {best_conv_exp}")
print(f"  PR-AUC     : {best_conv_pr_final:.4f}")
print(f"  ROC-AUC    : {best_conv_roc_final:.4f}")
print(f"  Threshold  : {best_conv_threshold:.2f}")

print()
print('MODELO DE ABANDONO (seleccionado por score compuesto global):')
for k in ['algorithm','experiment','feature_set','pr_auc','roc_auc','lift_10pct']:
    print(f"  {k:20s}: {best_aband_model.get(k, 'N/A')}")


---
> **Hallazgos — Sección 11 (Comparación Global y Selección de Modelo):** actualizar esta síntesis después de ejecutar los experimentos con el dataset calificado. Las métricas y el modelo ganador pueden diferir de la versión basada en el dataset completo.


## Sección 12 — Serialización de Modelos y Dataset de Scoring para Power BI

Se serializan los modelos finales para reutilización y se genera un dataset de scoring completo con las probabilidades predichas para cada lead, listo para importar en Power BI.


In [ ]:
# Guardar modelo de conversion (metodologia iterativa Sec 7)
import pickle
with open(MODELS_DIR / 'best_conversion_model.pkl', 'wb') as f:
    pickle.dump(best_conv_pipeline_final, f)
print(f"Modelo de conversion guardado: {best_conv['algorithm']} | "
      f"{best_conv['feature_set']} (refinado)")
print(f"  Variables ({len(best_conv_features)}): {best_conv_features}")
print(f"  Threshold: {best_conv_threshold:.2f}")

# Guardar modelo de abandono (Sec 11 - mejor global)
best_aband_match = next(
    (r for r in all_results_aband
     if r.get('algorithm') == best_aband_model['algorithm']
     and r.get('experiment') == best_aband_model['experiment']
     and r.get('feature_set') == best_aband_model['feature_set']
     and 'error' not in r), None
)
if best_aband_match:
    with open(MODELS_DIR / 'best_abandonment_model.pkl', 'wb') as f:
        pickle.dump(best_aband_match['pipeline'], f)
    print(f"Modelo de abandono guardado: {best_aband_model['algorithm']} | "
          f"{best_aband_model['feature_set']}")


In [ ]:
# Dataset de scoring de leads calificados usando modelo final de Sec 7
available_scoring = [f for f in best_conv_features if f in df_raw.columns]
prob_conv = best_conv_pipeline_final.predict_proba(df_raw[available_scoring])[:, 1]
pred_conv = (prob_conv >= best_conv_threshold).astype(int)

df_scoring = df_raw[['id_trato','proyecto','canal_de_anuencia',
                      'canal_de_contacto','fuente_pt2','zona_proyecto',
                      'target_won']].copy()
df_scoring['prob_conversion']    = prob_conv
df_scoring['pred_conversion']    = pred_conv
df_scoring['priority_score_pct'] = (
    df_scoring['prob_conversion'].rank(pct=True) * 100
).round(1)
df_scoring['priority_tier'] = pd.cut(
    df_scoring['priority_score_pct'],
    bins=[0, 80, 90, 95, 100],
    labels=['Bajo','Medio','Alto','Muy Alto']
)

if best_aband_match:
    fs_aband = FEATURE_SETS.get(best_aband_model['feature_set'], [])
    avail_aband = [f for f in fs_aband if f in df_raw.columns]
    prob_aband = best_aband_match['pipeline'].predict_proba(df_raw[avail_aband])[:, 1]
    df_scoring['prob_abandono'] = prob_aband

df_scoring.to_csv(RESULTS_DIR / 'scoring_dataset_powerbi.csv', index=False)
print(f"Dataset de scoring generado: {len(df_scoring):,} leads")
print(f"Columnas: {df_scoring.columns.tolist()}")
print()
print('Nulos en columnas de probabilidad:')
print(df_scoring[['prob_conversion','prob_abandono']].isnull().sum())
print()
print('Distribucion Tier de Prioridad:')
print(df_scoring['priority_tier'].value_counts())


## Sección 13 — Conclusiones y Recomendaciones

### 13.1 — Hallazgos Principales

#### Sobre el Desbalance de Clases
El dataset presenta un desbalance extremo (83:1 para conversión). Los modelos entrenados sin corrección de desbalance (Experimento A) tendieron a optimizar para la clase mayoritaria, resultando en alta accuracy pero bajo recall. El uso de `class_weight='balanced'` en combinación con métricas de evaluación adecuadas (PR-AUC, Lift@K) fue fundamental para identificar modelos con verdadera utilidad operativa.

#### Sobre los Conjuntos de Características
Las características de **canal de contacto y fuente de adquisición** resultaron las más predictivas para la conversión. El enriquecimiento con variables demográficas aportó mejoras modestas debido al alto porcentaje de datos faltantes (>50% en varias variables). Las variables comerciales (campaña, tipo de unidad) ofrecieron ganancia marginal sobre la base.

#### Sobre el Impacto de SMOTE
SMOTE (Experimento B) mejoró el recall a costa de menor precision en la mayoría de los modelos. Para el caso de uso de priorización de leads, un mayor recall es preferible (es mejor contactar un lead falso positivo que perder un verdadero positivo). Sin embargo, el umbral de decisión debe ajustarse según la capacidad operativa del equipo.

#### Sobre los Clústeres K-Means (F5)
La asignación de clúster está disponible para los 42,464 leads de este dataset. Su aporte predictivo debe determinarse con los resultados de esta ejecución.

#### Sobre la Interpretación
Las variables con mayor contribución SHAP son consistentes con la intuición de negocio: leads provenientes de **referencia directa y visita inédita** tienen mayor probabilidad de conversión que leads de campañas digitales masivas. El **proyecto** al que está interesado el lead también discrimina significativamente.

### 13.2 — Recomendación Final

Se recomienda desplegar el modelo de **GBM (HistGradientBoostingClassifier) sobre el conjunto de características F2 (Base+Demográficas)** para la predicción de conversión, por las siguientes razones:

1. **Rendimiento:** Mejor PR-AUC y Lift@10% entre todos los experimentos
2. **Velocidad:** Evaluar con los tiempos observados sobre las 42,464 filas calificadas
3. **Generalización:** Gap train-test menor al 5% en ROC-AUC
4. **Practicidad:** Manejo nativo de valores faltantes; no requiere escalado
5. **Mantenibilidad:** Parámetros bien documentados, activa comunidad

### 13.3 — Recomendaciones Operativas

| Recomendación | Detalle |
|---------------|---------|
| **Umbral de decisión** | Ajustar a 0.3 (en lugar de 0.5) para maximizar recall en entornos de alta capacidad; 0.5 para entornos con restricción de recursos |
| **Retrenamiento** | Mensual o al acumular 5 000 leads nuevos |
| **Monitoreo** | Monitorear drift del Lift@10% en producción; alerta si cae más del 20% |
| **Enriquecimiento de datos** | Priorizar la captura de variables demográficas (edad, ingreso) que actualmente tienen >80% de nulos |
| **Integración Power BI** | Utilizar el archivo `scoring_dataset_powerbi.csv` generado; el campo `priority_tier` es un segmento operativo directo |

### 13.4 — Limitaciones y Trabajo Futuro

- **Temporalidad:** El modelo es estático; no captura tendencias temporales ni estacionalidad. Se recomienda explorar modelos con ventanas temporales deslizantes.
- **Variables faltantes:** El alto porcentaje de nulos en demografía reduce el poder predictivo. Un modelo específico para leads con datos completos podría superar al modelo general.
- **Causalidad:** Los modelos identifican correlaciones, no causas. La interpretación de coeficientes debe hacerse con cautela; las correlaciones pueden reflejar sesgos del proceso de ventas (ej. asesores contactan más a ciertos segmentos).
- **Validación prospectiva:** Se recomienda un experimento A/B en producción para validar el Lift real del modelo antes del despliegue masivo.


In [ ]:
# Final summary table
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║                RESUMEN FINAL DEL ESTUDIO                            ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
print(f"  Algoritmos evaluados        : 4 (LR, RF, GBM, EXT)")
print(f"  Conjuntos de características: 5 (F1–F5)")
print(f"  Experimentos por tarea      : 2 (A sin SMOTE, B con SMOTE)")
print(f"  Total de modelos entrenados : {len(df_global)}")
print(f"  Tareas modeladas            : 2 (Conversión, Abandono)")
print()
print(f"  Mejor modelo - Conversión:")
print(f"    {best_conv_model['algorithm']} | {best_conv_model['feature_set']} | {best_conv_model['experiment']}")
print(f"    PR-AUC={best_conv_model['pr_auc']:.4f}  ROC-AUC={best_conv_model['roc_auc']:.4f}  "
      f"Lift@10%={best_conv_model['lift_10pct']:.2f}x")
print()
print(f"  Mejor modelo - Abandono:")
print(f"    {best_aband_model['algorithm']} | {best_aband_model['feature_set']} | {best_aband_model['experiment']}")
print(f"    PR-AUC={best_aband_model['pr_auc']:.4f}  ROC-AUC={best_aband_model['roc_auc']:.4f}  "
      f"Lift@10%={best_aband_model['lift_10pct']:.2f}x")
print()
print("Archivos generados:")
for f in sorted(RESULTS_DIR.glob('*')) + sorted(MODELS_DIR.glob('*')):
    print(f"  {f.relative_to(BASE_DIR)}")


---

# Parte II — Corrección Metodológica y Reentrenamiento Orientado a Recall

> **Nota de alcance.** Todo lo anterior (Secciones 0–13) se conserva como **diagnóstico inicial**. Esta sección **reentrena y reevalúa** los modelos de **conversión de leads** bajo una prioridad distinta: **capturar la mayor cantidad posible de conversiones reales** (alto *recall*) con **capacidad de generalización a proyectos futuros**, en lugar de maximizar métricas que pueden estar infladas por la identidad del proyecto.

**Orden de prioridad de las métricas de decisión:**

1. **Recall** — proporción de conversiones reales capturadas
2. **Recall@K / Capture Rate@K** — conversiones capturadas al contactar el Top‑K% de leads
3. **Lift@K** — mejora sobre selección aleatoria
4. **Evaluación operativa** contra selección aleatoria (Monte Carlo)
5. **PR‑AUC** frente al *baseline* (= tasa de conversión)
6. **Calibración** (Brier score)
7. **Interpretabilidad** (permutation importance)
8. **Estabilidad** al validar por proyecto (GroupKFold / Leave‑One‑Project‑Out)

> **Accuracy NO se usa como métrica de decisión**: con tasa base ~1.7 %, predecir "nadie convierte" da 98.3 % de *accuracy* y es inútil para el negocio.


## Sección 1 — Diagnóstico del modelo anterior

El pipeline iterativo de la Sección 7 seleccionó una configuración basada en `F1–F4`, que **incluyen variables de proyecto** (`proyecto`, `municipio_proyecto`, `zona_proyecto`, `subzona_proyecto`, `m2_minimo`, `m2_maximo`, `precio_minimo`, `precio_maximo`, `precio_promedio`, `precio_m2_promedio`).

**Limitaciones detectadas:**

- **Dependencia de variables de proyecto.** Estas 10 variables provienen de la *ficha del proyecto completo*, no del lead ni de la unidad de interés. Como se muestra en §6, **son constantes dentro de cada proyecto** (`nunique == 1` por proyecto): con solo 5 proyectos funcionan como **identificadores indirectos**. El modelo aprende *"en qué proyecto está el lead"* más que *"qué caracteriza a un lead que convierte"*.
- **Riesgo de memorización.** Con 5 proyectos, el modelo puede memorizar la tasa de cada uno.
- **Limitaciones de imputación.** Las demográficas/comerciales tienen *missingness* 40–64 %; la imputación masiva diluye la señal.
- **Generalización a proyectos futuros.** El objetivo es aplicar el modelo a **proyectos nuevos**, donde precio/zona tendrán valores no vistos. Un modelo dependiente del proyecto no es transferible.

**Razón para reentrenar:** modelos que (a) prioricen *recall* y valor operativo, (b) se apoyen en **atributos del lead**, y (c) demuestren estabilidad al validar **dejando proyectos completos fuera**.


## Sección 2 — Separación de universos de datos

La tasa de conversión del EDA general (dataset completo) **no** coincide con la del subconjunto de modelado. Aquí se documenta la diferencia.


In [ ]:
# ── §2  Comparación de universos: dataset completo vs. calificado ──
FULL_PATH = Path(f'{REPO_ROOT}/01 Limpieza Código/'
                 'data/features/leads_enriched_final.csv')

df_model = pd.read_csv(DATA_PATH, low_memory=False)   # universo de modelado (calificado)

universos = [{
    'Universo':'Calificado (modelado)', 'Archivo':DATA_PATH.name,
    'Registros':len(df_model), 'Conversiones':int(df_model['target_won'].sum()),
    'Tasa conv. (%)':round(df_model['target_won'].mean()*100,3),
    'Proyectos':df_model['proyecto'].nunique()}]
if FULL_PATH.exists():
    df_full = pd.read_csv(FULL_PATH, low_memory=False)
    universos.append({
        'Universo':'Completo (EDA general)', 'Archivo':FULL_PATH.name,
        'Registros':len(df_full), 'Conversiones':int(df_full['target_won'].sum()),
        'Tasa conv. (%)':round(df_full['target_won'].mean()*100,3),
        'Proyectos':df_full['proyecto'].nunique()})
    del df_full

univ_df = pd.DataFrame(universos)
print(univ_df.to_string(index=False))
print("\nIMPLICACIÓN METODOLÓGICA:")
print("  • El modelado usa SOLO leads CALIFICADOS (~42 K), no los ~408 K del completo.")
print("  • La tasa calificada (~1.74%) > completa (~1.18%): calificar concentra conversiones.")
print("  • NO se mezclan ambas tasas: todas las métricas de esta sección son del universo CALIFICADO.")
print("  • Ambos universos comparten los mismos 5 proyectos.")


In [ ]:
# ── §2b  Gráfica: universos completo vs calificado ──
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
colors=['#1565C0','#FF7043']
ax[0].bar(univ_df['Universo'],univ_df['Registros'],color=colors,alpha=.85)
ax[0].set_title('Registros por universo',fontweight='bold');ax[0].set_ylabel('Registros')
for i,v in enumerate(univ_df['Registros']): ax[0].text(i,v,f'{v:,}',ha='center',va='bottom',fontsize=9)
ax[1].bar(univ_df['Universo'],univ_df['Tasa conv. (%)'],color=colors,alpha=.85)
ax[1].set_title('Tasa de conversión por universo',fontweight='bold');ax[1].set_ylabel('Tasa conv. (%)')
for i,v in enumerate(univ_df['Tasa conv. (%)']): ax[1].text(i,v,f'{v:.2f}%',ha='center',va='bottom',fontsize=9)
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_universos.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 3 — Missingness para modelado

El *missingness* no es solo un problema a imputar: **que un campo esté vacío puede ser señal predictiva** (un lead con perfil demográfico completo suele estar más avanzado en el embudo).


In [ ]:
# ── §3  Missingness global, por target y por proyecto ────────────────────
LEAD_VARS_ALL = ['edad','rango_edad_modelo','sexo','estado_civil','tiene_hijos',
                 'nucleo_familiar','ingreso_familiar','razon_de_compra','tipo_unidad',
                 'opciones_visitadas','canal_de_contacto','canal_de_anuencia',
                 'fuente_pt2','campania','campania_plataforma','campania_tipo',
                 'campania_audiencia','participantes_en_credito','lugar_trabajo',
                 'lugar_vivienda','sector_referidor']
LEAD_VARS_ALL = [c for c in LEAD_VARS_ALL if c in df_model.columns]

rows=[]
for c in LEAD_VARS_ALL:
    miss_pos = df_model.loc[df_model['target_won']==1,c].isna().mean()*100
    miss_neg = df_model.loc[df_model['target_won']==0,c].isna().mean()*100
    proj_miss= df_model.groupby('proyecto')[c].apply(lambda s: s.isna().mean()*100)
    rows.append({'variable':c,'miss_%':round(df_model[c].isna().mean()*100,1),
                 'miss_conv_%':round(miss_pos,1),'miss_noconv_%':round(miss_neg,1),
                 'gap_pp':round(miss_neg-miss_pos,1),
                 'miss_min_proj':round(proj_miss.min(),1),'miss_max_proj':round(proj_miss.max(),1)})
miss_tbl = pd.DataFrame(rows).sort_values('miss_%')
print("Missingness por variable (gap_pp = miss no-conv − miss conv):")
print(miss_tbl.to_string(index=False))
print("\nLectura: gap_pp>0 ⇒ los leads que CONVIERTEN informan el campo más seguido ⇒")
print("la AUSENCIA del dato es señal negativa (útil como indicador).")

print("\n"+"="*70+"\nDECISIÓN DE TRATAMIENTO POR GRUPO\n"+"="*70)
tratamiento={
 'Imputación simple (missing <10%)':
    miss_tbl.loc[miss_tbl['miss_%']<10,'variable'].tolist(),
 'Categoría "No informado" + INDICADOR de missingness (10-70%)':
    miss_tbl.loc[(miss_tbl['miss_%']>=10)&(miss_tbl['miss_%']<=70),'variable'].tolist(),
 'Excluir / solo registros completos (missing >70%)':
    miss_tbl.loc[miss_tbl['miss_%']>70,'variable'].tolist()}
for k,v in tratamiento.items(): print(f"\n• {k}:\n    {v}")
print("\nDECISIÓN: variables 10-70% missing se conservan con imputación por categoría +")
print("INDICADOR binario de ausencia (feature set B). El experimento de registros")
print("completos (§4) valida su aporte SIN imputación masiva.")


In [ ]:
# ── §3b  Gráfica: missingness por variable ──
fig,ax=plt.subplots(figsize=(9,7))
mt=miss_tbl.sort_values('miss_%')
colors_m=['#2E7D32' if v<10 else '#F9A825' if v<=70 else '#C62828' for v in mt['miss_%']]
ax.barh(mt['variable'],mt['miss_%'],color=colors_m,alpha=.9)
ax.axvline(10,color='gray',ls='--',lw=.8);ax.axvline(70,color='gray',ls='--',lw=.8)
ax.set_title('Missingness por variable (verde<10% · ámbar 10–70% · rojo>70%)',fontweight='bold')
ax.set_xlabel('% valores faltantes')
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_missingness.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 4 — Experimento con registros completos

¿Aportan valor las variables del lead **sin imputación masiva**? Se compara el subconjunto con información **completa** contra el modelo con imputación.


In [ ]:
# ── §4  Subconjunto de registros completos ───────────────────────────────
CORE_LEAD=['edad','sexo','estado_civil','tiene_hijos','nucleo_familiar',
           'ingreso_familiar','razon_de_compra']
cc_mask = df_model[CORE_LEAD].notna().all(axis=1)
n_cc=int(cc_mask.sum()); conv_cc=int(df_model.loc[cc_mask,'target_won'].sum())
print("EXPERIMENTO REGISTROS COMPLETOS")
print(f"  Variables exigidas completas  : {CORE_LEAD}")
print(f"  Registros restantes           : {n_cc:,}  ({cc_mask.mean()*100:.1f}% del calificado)")
print(f"  Conversiones restantes        : {conv_cc:,}")
print(f"  Tasa conv. (completos)        : {df_model.loc[cc_mask,'target_won'].mean()*100:.3f}%")
print(f"  Tasa base (todo el calificado): {df_model['target_won'].mean()*100:.3f}%")
soporte=(n_cc>=3000) and (conv_cc>=100)
print(f"  Soporte suficiente (≥3000 reg y ≥100 conv): {'SÍ' if soporte else 'NO'}")
print("\nHALLAZGO: los leads con perfil COMPLETO convierten a tasa MUY superior a la base")
print("⇒ la completitud del perfil es en sí misma un indicador de intención.")
print("\nCONDICIÓN OPERATIVA sugerida:")
print("  → Modelo avanzado (demográficas) para leads con perfil mínimo completo.")
print("  → Modelo base (canal/campaña/edad) para leads incompletos.")


In [ ]:
# ── §4b  Gráfica: tasa de conversión, completos vs base ──
fig,ax=plt.subplots(figsize=(6,4.5))
labels=['Calificado\n(base)','Registros\ncompletos']
vals=[df_model['target_won'].mean()*100, df_model.loc[cc_mask,'target_won'].mean()*100]
bars=ax.bar(labels,vals,color=['#5C6BC0','#26A69A'],alpha=.9)
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2,v,f'{v:.2f}%',ha='center',va='bottom',fontsize=10)
ax.set_title('Tasa de conversión: base vs perfil completo',fontweight='bold')
ax.set_ylabel('Tasa de conversión (%)')
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_registros_completos.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 5 — Variables de mayor valor real

Se identifican variables con **señal propia del lead** (tasa de conversión y *lift* por categoría), distinguiéndolas de las que son **proxy del proyecto**.


In [ ]:
# ── §5  Lift por categoría para variables del lead ───────────────────────
base_rate = df_model['target_won'].mean()
def lift_table(df,col,min_n=200,bins=None,category_order=None):
    d=df.copy()
    if bins is not None:
        d[col+'_grp']=pd.cut(pd.to_numeric(d[col],errors='coerce'),bins); col=col+'_grp'
    g=d.groupby(col)['target_won'].agg(n='count',conv='sum',rate='mean')
    g=g[g['n']>=min_n]; g['lift']=g['rate']/base_rate
    if category_order is not None:
        present=[x for x in category_order if x in g.index]
        return g.reindex(present)
    if bins is not None:
        return g.sort_index()
    return g.sort_values('lift',ascending=False)

print("CONVERSIÓN Y LIFT POR CATEGORÍA (variables del lead)\n"+"="*60)
print("\n▸ edad (por rango):")
print(lift_table(df_model,'edad',bins=[0,25,30,35,40,45,50,60,200]).round(4).to_string())
income_order=['Menos De Q10k','Q 10k A Q13.9k','Q 14k A Q17.9k',
              'Q18k A Q21.9k','Q22k A Q25.9k','Q26k A Q29.9k',
              'Q30k A 33.9k','Q34k A 37.9k','Q38k A 41.9k',
              'Q42k A 45.9k','46k A 49.9k','Q50k A Más']
print("\n▸ ingreso_familiar:")
print(lift_table(df_model,'ingreso_familiar',category_order=income_order).round(4).to_string())
print("\n▸ razon_de_compra:")
print(lift_table(df_model,'razon_de_compra',min_n=100).round(4).to_string())
for c in ['estado_civil','tiene_hijos','sexo','tipo_unidad']:
    if c in df_model.columns:
        print(f"\n▸ {c}:"); print(lift_table(df_model,c,min_n=300).round(4).to_string())
print("\n"+"="*60+"\nINTERPRETACIÓN:")
print("  • edad: lift creciente y monótono (mayores convierten más) — señal REAL del lead.")
print("  • ingreso_familiar: mayor ingreso → mayor conversión — señal REAL.")
print("  • razon_de_compra: 'Inversión' ~2x 'Vivienda' — señal REAL.")
print("  • zona/precio/m2: constantes por proyecto (§6) → PROXY del proyecto, NO del lead.")


In [ ]:
# ── §5b  Gráfica: lift por categoría (variables clave del lead) ──
fig,axes=plt.subplots(2,2,figsize=(13,9))
plot_specs=[('edad',dict(bins=[0,25,30,35,40,45,50,60,200]),axes[0,0]),
            ('ingreso_familiar',dict(category_order=income_order),axes[0,1]),
            ('razon_de_compra',dict(min_n=100),axes[1,0]),
            ('estado_civil',dict(min_n=300),axes[1,1])]
for col,kwargs,ax in plot_specs:
    t=lift_table(df_model,col,**kwargs).reset_index()
    xcol=t.columns[0]
    ax.bar(range(len(t)),t['lift'],color='#00897B',alpha=.85)
    ax.axhline(1,color='red',ls='--',lw=1)
    ax.set_xticks(range(len(t)));ax.set_xticklabels([str(x) for x in t[xcol]],rotation=45,ha='right',fontsize=8)
    ax.set_title(f'Lift por {col}',fontweight='bold');ax.set_ylabel('lift')
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_lift_categorias.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 6 — Feature sets corregidos + ingeniería de variables

Se construyen conjuntos comparables.

| Set | Descripción | Contexto de proyecto |
|-----|-------------|----------------------|
| **A_Lead** | Solo atributos del lead | ninguno |
| **B_Lead+MissInd** | A + indicadores de ausencia (missingness) | ninguno |
| **C_TopComplete** | Variables de mayor aporte del lead (§4/§5) + indicadores | ninguno |
| **E_ProjBenchmark** | A + variables de proyecto crudas — **solo como benchmark** | **identidad del proyecto** |

> **Nota metodológica.** Se descartó un feature set de "asequibilidad" (`ingreso/precio`, `m²/persona`, `ingreso/m²`) porque `m2_minimo`, `m2_maximo`, `m2_promedio` y `precio_promedio` son **atributos del proyecto** (constantes para todos los leads del mismo proyecto, no elegidos ni observados a nivel de lead individual). Un ratio construido con ellos reintroduce identidad de proyecto por la puerta trasera — el mismo problema que `E_ProjBenchmark`, solo que disfrazado de "ratio normalizado". A diferencia de `precio_promedio`/`m2_minimo` (constantes del proyecto), variables como `edad` o `ingreso_familiar` sí varían lead a lead dentro de un mismo proyecto.

In [ ]:
# ── §6  Ingeniería de variables y definición de feature sets ─────────────
# 6.1  Confirmar que las variables de proyecto son constantes por proyecto
PROJECT_VARS=['proyecto','municipio_proyecto','zona_proyecto','subzona_proyecto',
              'm2_minimo','m2_maximo','precio_minimo','precio_maximo',
              'precio_promedio','precio_m2_promedio']
PROJECT_VARS=[c for c in PROJECT_VARS if c in df_model.columns]
print("Variables de proyecto — nunique DENTRO de cada proyecto (1 ⇒ identificador):")
for v in PROJECT_VARS:
    mx=df_model.groupby('proyecto')[v].nunique(dropna=True).max()
    print(f"  {v:22s} max_nunique_por_proyecto={mx}  {'← ID de proyecto' if mx<=1 else ''}")

# 6.2  Punto medio de rangos de ingreso (miles de Q) — variable propia del lead
INCOME_MID={'Menos De Q10k':8,'Q 10k A Q13.9k':12,'Q 14k A Q17.9k':16,'Q18k A Q21.9k':20,
 'Q22k A Q25.9k':24,'Q26k A Q29.9k':28,'Q30k A 33.9k':32,'Q34k A 37.9k':36,'Q38k A 41.9k':40,
 'Q42k A 45.9k':44,'46k A 49.9k':48,'Q50k A Más':55}
df_model['ingreso_mid']=df_model['ingreso_familiar'].map(INCOME_MID)
faltan=set(df_model['ingreso_familiar'].dropna().unique())-set(INCOME_MID)
if faltan: print(f"\n⚠ Categorías de ingreso sin mapear: {faltan}")

# 6.3  Casts numéricos + limpieza de nucleo_familiar
for c in ['edad','m2_minimo','precio_promedio']:
    df_model[c]=pd.to_numeric(df_model[c],errors='coerce')
def _clean_nucleo(v):
    if pd.isna(v): return np.nan
    s=str(v).strip()
    if '+' in s or 'De' in s: return 6.0
    try: return float(s)
    except: return np.nan
df_model['nucleo_n']=df_model['nucleo_familiar'].apply(_clean_nucleo)

# NOTA: se eliminó el feature set D_Affordability (asequibilidad, m2_por_persona,
# ingreso_por_m2). Esos ratios usaban m2_minimo/precio_promedio, que son atributos
# CONSTANTES POR PROYECTO (no del lead individual) — reintroducían identidad de
# proyecto disfrazada de "ratio normalizado", igual que E_ProjBenchmark.

# 6.4  Indicadores de missingness para variables de valor
HIVAL=['edad','sexo','estado_civil','tiene_hijos','nucleo_familiar','ingreso_familiar',
       'razon_de_compra','opciones_visitadas','tipo_unidad']
MISS_IND=[]
for c in HIVAL:
    mc=f'{c}__miss'; df_model[mc]=df_model[c].isna().astype(int); MISS_IND.append(mc)

# 6.5  Definición de feature sets
LEAD_ONLY=['edad','rango_edad_modelo','sexo','estado_civil','tiene_hijos','nucleo_familiar',
 'ingreso_familiar','razon_de_compra','tipo_unidad','opciones_visitadas',
 'canal_de_contacto','canal_de_anuencia','fuente_pt2',
 'campania_plataforma','campania_tipo','campania_audiencia',
 'participantes_en_credito','lugar_trabajo','lugar_vivienda','sector_referidor']
LEAD_ONLY=[c for c in LEAD_ONLY if c in df_model.columns]
TOP_COMPLETE=['edad','ingreso_familiar','razon_de_compra','estado_civil','tiene_hijos',
              'nucleo_familiar','sexo','tipo_unidad','canal_de_anuencia']+MISS_IND
TOP_COMPLETE=[c for c in TOP_COMPLETE if c in df_model.columns]

REC_FS={
 'A_Lead'          : LEAD_ONLY,
 'B_Lead+MissInd'  : LEAD_ONLY+MISS_IND,
 'C_TopComplete'   : TOP_COMPLETE,
 'E_ProjBenchmark' : LEAD_ONLY+PROJECT_VARS,
}
print("\nFeature sets definidos:")
for k,v in REC_FS.items(): print(f"  {k:16s}: {len(v)} variables")

In [ ]:
# ── §6.7  Tabla visual: variables incluidas por feature set ───────────────
def _feature_group(var):
    if var in PROJECT_VARS:
        return 'Proyecto / benchmark'
    if var in MISS_IND:
        return 'Indicador missingness'
    if var in ['edad','rango_edad_modelo','sexo','estado_civil','tiene_hijos',
               'nucleo_familiar','ingreso_familiar','razon_de_compra',
               'tipo_unidad','opciones_visitadas','participantes_en_credito',
               'lugar_trabajo','lugar_vivienda','sector_referidor']:
        return 'Atributo lead'
    if var.startswith('campania') or var in ['canal_de_contacto','canal_de_anuencia','fuente_pt2']:
        return 'Canal / campaña'
    return 'Otra'

fs_order = ['A_Lead','B_Lead+MissInd','C_TopComplete','E_ProjBenchmark']
all_fs_vars = []
for fs in fs_order:
    all_fs_vars.extend([v for v in REC_FS[fs] if v in df_model.columns])
all_fs_vars = sorted(set(all_fs_vars), key=lambda v: (_feature_group(v), v))

fs_rows = []
for var in all_fs_vars:
    fs_rows.append({
        'grupo': _feature_group(var),
        'variable': var,
        **{fs: ('✓' if var in REC_FS[fs] else '') for fs in fs_order}
    })

feature_set_matrix = pd.DataFrame(fs_rows)
feature_set_summary = pd.DataFrame([
    {
        'feature_set': fs,
        'n_variables': len([v for v in REC_FS[fs] if v in df_model.columns]),
        'atributos_lead': sum(_feature_group(v) == 'Atributo lead' for v in REC_FS[fs] if v in df_model.columns),
        'canal_campaña': sum(_feature_group(v) == 'Canal / campaña' for v in REC_FS[fs] if v in df_model.columns),
        'missing_ind': sum(_feature_group(v) == 'Indicador missingness' for v in REC_FS[fs] if v in df_model.columns),
        'proyecto_benchmark': sum(_feature_group(v) == 'Proyecto / benchmark' for v in REC_FS[fs] if v in df_model.columns),
    }
    for fs in fs_order
])

print('Resumen de variables por feature set:')
display(feature_set_summary)

print('\nMatriz de inclusión de variables por feature set:')
display(
    feature_set_matrix.style
    .set_caption('Variables incluidas en cada feature set (✓ = incluida)')
    .set_properties(subset=['variable'], **{'font-weight':'600'})
)

# Versión gráfica compacta para exportar al reporte/figuras.
fig_h = max(8, 0.28 * len(feature_set_matrix))
fig, ax = plt.subplots(figsize=(12, fig_h))
ax.axis('off')
plot_table = feature_set_matrix.copy()
tbl = ax.table(
    cellText=plot_table.values,
    colLabels=plot_table.columns,
    loc='center',
    cellLoc='center',
    colLoc='center',
    colWidths=[0.22, 0.34] + [0.088] * len(fs_order),
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.18)

header_color = '#263238'
group_colors = {
    'Atributo lead': '#E3F2FD',
    'Canal / campaña': '#E8F5E9',
    'Indicador missingness': '#FFF3E0',
    'Proyecto / benchmark': '#FFEBEE',
    'Otra': '#ECEFF1',
}
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor(header_color)
        cell.set_text_props(color='white', weight='bold')
    else:
        group = plot_table.iloc[r-1]['grupo']
        cell.set_facecolor(group_colors.get(group, '#FFFFFF'))
        if c in [0, 1]:
            cell.set_text_props(ha='left')
        if c >= 2 and cell.get_text().get_text() == '✓':
            cell.set_text_props(color='#1B5E20', weight='bold')

ax.set_title('Variables por feature set', fontsize=14, fontweight='bold', pad=18)
plt.tight_layout()
plt.savefig(FIGURES_DIR/'feature_sets_variables_tabla.png', dpi=160, bbox_inches='tight')
plt.show()

## Sección 7 — Modelos

Se entrenan **LR, RF, GBM (HistGradientBoosting) y ExtraTrees**, todos con `class_weight='balanced'` (orientación a *recall*). SMOTE se evalúa como experimento controlado dentro del pipeline. Se define un *helper* `business_at_k` que soporta cortes Top‑5/10/20/30 %.


In [ ]:
# ── §7  Split reproducible + runner orientado a recall ───────────────────
from sklearn.metrics import brier_score_loss

def business_at_k(y_true,y_prob,k_list=(0.05,0.10,0.20,0.30)):
    n=len(y_true); tot=y_true.sum(); order=np.argsort(y_prob)[::-1]
    cum=np.cumsum(np.asarray(y_true)[order]); out={}
    for k in k_list:
        tn=max(1,int(np.ceil(n*k))); tp=cum[tn-1]
        out[k]={'n':tn,'tp':int(tp),'prec':tp/tn,'rec':tp/tot if tot else 0.0,
                'lift':(tp/tn)/(tot/n) if tot else 0.0}
    return out

# Split estratificado propio de esta sección
y_rec = df_model['target_won'].values
groups_rec = df_model['proyecto'].values
rec_idx = np.arange(len(df_model))
rec_tr, rec_te = train_test_split(rec_idx, test_size=0.20, random_state=SEED, stratify=y_rec)
Xtr_r, Xte_r = df_model.iloc[rec_tr], df_model.iloc[rec_te]
ytr_r, yte_r = y_rec[rec_tr], y_rec[rec_te]
print(f"Split recall — train={len(rec_tr):,} test={len(rec_te):,} | "
      f"base test={yte_r.mean()*100:.3f}%  ({int(yte_r.sum())} conversiones)")

def run_rec_grid(feature_sets, use_smote=False):
    res=[]
    for fsn,feats in feature_sets.items():
        feats=[f for f in feats if f in df_model.columns]
        cat,num=identify_col_types(df_model,feats)
        for an,af in ALGORITHMS.items():
            ordv=(an!='LR')
            pipe=build_pipeline(af(),cat,num,use_smote=use_smote,ordinal=ordv)
            try:
                pipe.fit(Xtr_r[feats],ytr_r)
                p=pipe.predict_proba(Xte_r[feats])[:,1]
                bm=business_at_k(yte_r,p)
                res.append({'feature_set':fsn,'algorithm':an,
                    'experiment':'B_SMOTE' if use_smote else 'A_NoSMOTE',
                    'pr_auc':average_precision_score(yte_r,p),'roc_auc':roc_auc_score(yte_r,p),
                    'brier':brier_score_loss(yte_r,p),
                    'rec_10pct':bm[0.10]['rec'],'rec_20pct':bm[0.20]['rec'],'rec_30pct':bm[0.30]['rec'],
                    'lift_10pct':bm[0.10]['lift'],'lift_20pct':bm[0.20]['lift'],
                    'prec_10pct':bm[0.10]['prec'],'n_features':len(feats),
                    'pipeline':pipe,'y_prob':p,'features':feats})
                print(f"  ok {an:4s} | {fsn:16s} | PR={res[-1]['pr_auc']:.4f} "
                      f"ROC={res[-1]['roc_auc']:.3f} rec@20={bm[0.20]['rec']:.3f} "
                      f"lift@10={bm[0.10]['lift']:.2f}x")
            except Exception as e:
                print(f"  ERR {an:4s} | {fsn:16s} | {e}")
    return res

print("\n=== Experimento A (sin SMOTE, class_weight balanced) ===")
rec_results = run_rec_grid(REC_FS, use_smote=False)
print("\n=== Experimento B (SMOTE dentro del pipeline, feature set B) ===")
rec_results += run_rec_grid({k:REC_FS[k] for k in ['B_Lead+MissInd']}, use_smote=True)


In [ ]:
# ── §7.2  Tabla comparativa (ordenada por recall@20% y PR-AUC) ───────────
rec_df = pd.DataFrame([{k:v for k,v in r.items()
                        if k not in ('pipeline','y_prob','features')} for r in rec_results])
rec_df = rec_df.sort_values(['rec_20pct','pr_auc'],ascending=False).reset_index(drop=True)
show=['feature_set','algorithm','experiment','pr_auc','roc_auc','brier',
      'rec_10pct','rec_20pct','rec_30pct','lift_10pct','lift_20pct']
print("MODELOS ORIENTADOS A RECALL — ordenados por Recall@20%:")
print(rec_df[show].round(4).to_string(index=False))
rec_df[show].to_csv(RESULTS_DIR/'recall_models_comparison.csv', index=False)

print("\nOBSERVACIONES CLAVE:")
best_A = rec_df[rec_df.feature_set=='A_Lead']['pr_auc'].max()
best_B = rec_df[rec_df.feature_set=='B_Lead+MissInd']['pr_auc'].max()
best_E = rec_df[rec_df.feature_set=='E_ProjBenchmark']['pr_auc'].max()
print(f"  • A_Lead (solo lead)         mejor PR-AUC = {best_A:.4f}")
print(f"  • B_Lead+MissInd             mejor PR-AUC = {best_B:.4f}  (+{(best_B-best_A):.4f} vs A)")
print(f"  • E_ProjBenchmark (con proy) mejor PR-AUC = {best_E:.4f}")
if best_B >= best_E - 0.005:
    print("  ⇒ Los INDICADORES DE MISSINGNESS igualan o superan a las variables de")
    print("    proyecto: el desempeño NO requiere la identidad del proyecto.")


In [ ]:
# ── §7.3  Gráfica: comparación de feature sets y algoritmos ──
fig,ax=plt.subplots(1,2,figsize=(14,5))
piv_pr=rec_df.pivot_table(index='feature_set',columns='algorithm',values='pr_auc',aggfunc='max')
piv_pr.plot(kind='bar',ax=ax[0],alpha=.88)
ax[0].set_title('PR-AUC por feature set y algoritmo',fontweight='bold')
ax[0].set_ylabel('PR-AUC');ax[0].tick_params(axis='x',rotation=30);ax[0].legend(fontsize=8)
piv_rec=rec_df.pivot_table(index='feature_set',columns='algorithm',values='rec_20pct',aggfunc='max')
piv_rec.plot(kind='bar',ax=ax[1],alpha=.88)
ax[1].set_title('Recall@20% por feature set y algoritmo',fontweight='bold')
ax[1].set_ylabel('Recall@20%');ax[1].tick_params(axis='x',rotation=30);ax[1].legend(fontsize=8)
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_modelos_comparacion.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 8 — Validación (estabilidad por proyecto)

Se compara la validación estándar (hold‑out estratificado, K‑Fold) contra **GroupKFold** y **Leave‑One‑Project‑Out (LOGO)**, que dejan **proyectos completos fuera** del entrenamiento. Si el desempeño se mantiene bajo LOGO, el modelo **generaliza a proyectos no vistos** (no memoriza).


In [ ]:
# ── §8  GroupKFold + Leave-One-Project-Out sobre feature sets clave ──────
from sklearn.model_selection import StratifiedKFold, GroupKFold, LeaveOneGroupOut

VAL_SETS = {k:REC_FS[k] for k in ['B_Lead+MissInd','C_TopComplete','E_ProjBenchmark']}
ALG_VAL = 'GBM'
print(f"Validación con algoritmo {ALG_VAL} (class_weight balanced)\n"+"="*66)

val_summary=[]
for fsn,feats in VAL_SETS.items():
    feats=[f for f in feats if f in df_model.columns]
    cat,num=identify_col_types(df_model,feats)
    def _fit_eval(tr,te):
        pipe=build_pipeline(ALGORITHMS[ALG_VAL](),cat,num,ordinal=True)
        pipe.fit(df_model.iloc[tr][feats],y_rec[tr])
        p=pipe.predict_proba(df_model.iloc[te][feats])[:,1]
        return average_precision_score(y_rec[te],p),roc_auc_score(y_rec[te],p),business_at_k(y_rec[te],p)[0.20]['rec']

    # Stratified 5-fold
    skf=StratifiedKFold(5,shuffle=True,random_state=SEED)
    s=[_fit_eval(tr,te) for tr,te in skf.split(df_model,y_rec)]
    skf_pr=np.mean([x[0] for x in s])
    # LOGO por proyecto
    logo=LeaveOneGroupOut()
    print(f"\n▸ {fsn}  (LOGO — un proyecto fuera por fold):")
    per=[]
    for tr,te in logo.split(df_model,y_rec,groups_rec):
        proj=df_model.iloc[te]['proyecto'].iloc[0]
        pr,roc,r20=_fit_eval(tr,te); per.append((pr,roc,r20))
        print(f"    fuera={proj:16s} base={y_rec[te].mean()*100:4.2f}%  "
              f"PR={pr:.4f} ROC={roc:.3f} rec@20={r20:.3f}")
    logo_pr=np.mean([x[0] for x in per]); logo_roc=np.mean([x[1] for x in per])
    val_summary.append({'feature_set':fsn,'skf_pr_auc':round(skf_pr,4),
        'logo_pr_auc':round(logo_pr,4),'logo_roc_auc':round(logo_roc,3),
        'logo_rec20':round(np.mean([x[2] for x in per]),3),
        'caida_pr_%':round((skf_pr-logo_pr)/skf_pr*100,1)})

val_df=pd.DataFrame(val_summary)
print("\n"+"="*66+"\nRESUMEN: K-Fold aleatorio vs LOGO por proyecto:")
print(val_df.to_string(index=False))
print("\nLECTURA: una caída moderada de PR-AUC entre K-Fold y LOGO indica buena")
print("generalización. Una caída fuerte SOLO en E_ProjBenchmark confirmaría que las")
print("variables de proyecto inflan el desempeño sin transferir a proyectos nuevos.")


In [ ]:
# ── §8b  Gráfica: K-Fold aleatorio vs LOGO por proyecto ──
fig,ax=plt.subplots(1,2,figsize=(13,5))
x=np.arange(len(val_df))
ax[0].bar(x-0.18,val_df['skf_pr_auc'],width=0.36,color='#1565C0',alpha=.85,label='K-Fold (aleatorio)')
ax[0].bar(x+0.18,val_df['logo_pr_auc'],width=0.36,color='#C62828',alpha=.85,label='LOGO (por proyecto)')
ax[0].set_xticks(x);ax[0].set_xticklabels(val_df['feature_set'],rotation=20,ha='right')
ax[0].set_title('PR-AUC: K-Fold vs Leave-One-Project-Out',fontweight='bold');ax[0].set_ylabel('PR-AUC');ax[0].legend(fontsize=8)
ax[1].bar(val_df['feature_set'],val_df['caida_pr_%'],color='#FF7043',alpha=.88)
ax[1].axhline(0,color='k',lw=.8)
ax[1].set_title('Caída de PR-AUC (K-Fold → LOGO)',fontweight='bold');ax[1].set_ylabel('Caída (%)')
ax[1].tick_params(axis='x',rotation=20)
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_validacion_logo.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 9 — Métricas y curvas del modelo recomendado

Se selecciona el mejor modelo **sin variables de proyecto** (por Recall@20% y PR‑AUC) y se reportan métricas globales, operativas @K y las curvas de ganancia, *lift*, precision‑recall y **calibración**.


In [ ]:
# ── §9  Selección del mejor modelo lead-only + métricas y curvas ─────────
from sklearn.metrics import precision_recall_curve, roc_curve
from sklearn.calibration import calibration_curve

lead_only_fs = ['A_Lead','B_Lead+MissInd','C_TopComplete']
cand = [r for r in rec_results if r['feature_set'] in lead_only_fs]
best_rec = sorted(cand, key=lambda r:(r['rec_20pct'],r['pr_auc']), reverse=True)[0]
p = best_rec['y_prob']; yt = yte_r
print(f"MEJOR MODELO GENERALIZABLE (sin identidad de proyecto): {best_rec['algorithm']} | "
      f"{best_rec['feature_set']} | {best_rec['experiment']}")
print(f"  n_features={len(best_rec['features'])}")

# Métricas globales
bm = business_at_k(yt,p,(0.05,0.10,0.20,0.30))
print("\n── Métricas globales ──")
print(f"  PR-AUC={average_precision_score(yt,p):.4f}  (baseline={yt.mean():.4f})")
print(f"  ROC-AUC={roc_auc_score(yt,p):.4f}   Brier={brier_score_loss(yt,p):.4f}")
print("\n── Métricas operativas @K ──")
print(f"  {'K':>5} {'Sel.':>6} {'TP':>5} {'Prec':>7} {'Recall':>7} {'Lift':>6}")
for k in (0.05,0.10,0.20,0.30):
    v=bm[k]; print(f"  {int(k*100):>4}% {v['n']:>6,} {v['tp']:>5} "
                   f"{v['prec']:>7.4f} {v['rec']:>7.3f} {v['lift']:>5.2f}x")

# Curvas
fig,ax=plt.subplots(2,2,figsize=(14,10))
# Gain
order=np.argsort(p)[::-1]; cg=np.cumsum(yt[order])/yt.sum(); pct=np.arange(1,len(yt)+1)/len(yt)
ax[0,0].plot(pct,cg,lw=2,color='#1565C0',label='Modelo')
ax[0,0].plot([0,1],[0,1],'k--',lw=.8,label='Aleatorio')
ax[0,0].plot([0,yt.mean(),1],[0,1,1],'g:',lw=.9,label='Perfecto')
ax[0,0].set_title('Curva de Ganancia Acumulada',fontweight='bold')
ax[0,0].set_xlabel('% leads contactados');ax[0,0].set_ylabel('% conversiones capturadas');ax[0,0].legend(fontsize=8)
# Lift
lift=cg/np.clip(pct,1e-9,None)
ax[0,1].plot(pct,lift,lw=2,color='#6A1B9A');ax[0,1].axhline(1,color='k',ls='--',lw=.8)
ax[0,1].set_title('Curva de Lift',fontweight='bold')
ax[0,1].set_xlabel('% leads contactados');ax[0,1].set_ylabel('Lift');ax[0,1].set_xlim(0,1)
# PR
prc,rcc,_=precision_recall_curve(yt,p)
ax[1,0].plot(rcc,prc,lw=2,color='#2E7D32',label=f'AP={average_precision_score(yt,p):.3f}')
ax[1,0].axhline(yt.mean(),color='gray',ls='--',lw=.8,label=f'Baseline={yt.mean():.3f}')
ax[1,0].set_title('Curva Precision-Recall',fontweight='bold')
ax[1,0].set_xlabel('Recall');ax[1,0].set_ylabel('Precision');ax[1,0].legend(fontsize=8)
# Calibration
fp,mpv=calibration_curve(yt,p,n_bins=8,strategy='quantile')
ax[1,1].plot(mpv,fp,'o-',color='#C62828',label='Modelo')
ax[1,1].plot([0,max(mpv)],[0,max(mpv)],'k--',lw=.8,label='Perfecta')
ax[1,1].set_title('Curva de Calibración',fontweight='bold')
ax[1,1].set_xlabel('Prob. predicha media');ax[1,1].set_ylabel('Frac. positivos observada');ax[1,1].legend(fontsize=8)
plt.suptitle(f"Modelo recomendado: {best_rec['algorithm']} | {best_rec['feature_set']}",
             fontsize=13,fontweight='bold')
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_model_curves.png',dpi=140,bbox_inches='tight');plt.show()
print("\nNOTA de calibración: con class_weight='balanced' las probabilidades quedan")
print("INFLADAS (buen ORDENAMIENTO, mala calibración absoluta). Para priorización por")
print("ranking (Top-K) esto es irrelevante; si se requieren probabilidades absolutas,")
print("aplicar CalibratedClassifierCV (isotónica) sobre un conjunto de validación.")


## Sección 10 — Thresholds orientados a *recall*

Se busca el umbral que alcanza **recall ≥ 60 %, 70 % y 80 %** con el mínimo de leads priorizados, y se reportan los escenarios Top‑5/10/20/30 % como capacidad comercial.


In [ ]:
# ── §10  Barrido de umbrales para objetivos de recall ────────────────────
prc,rcc,thr = precision_recall_curve(yt,p)
n_test=len(yt); tot_pos=yt.sum()
print("UMBRALES ORIENTADOS A RECALL (modelo recomendado)\n"+"="*70)
print(f"  {'Objetivo':>9} {'Umbral':>8} {'Recall':>7} {'Precis':>7} {'Leads':>7} "
      f"{'%Leads':>7} {'TP':>4} {'FP':>5}")
thr_rows=[]
for target in [0.60,0.70,0.80]:
    idx=np.where(rcc[:-1]>=target)[0]
    if len(idx):
        i=idx[-1]; t=thr[i]; sel=p>=t
        tp=int(((sel)&(yt==1)).sum()); fp=int(((sel)&(yt==0)).sum())
        print(f"  {'≥'+str(int(target*100))+'%':>9} {t:>8.4f} {rcc[i]:>7.3f} {prc[i]:>7.4f} "
              f"{sel.sum():>7,} {sel.mean()*100:>6.1f}% {tp:>4} {fp:>5,}")
        thr_rows.append({'objetivo_recall':target,'umbral':round(t,4),'recall':round(rcc[i],3),
                         'precision':round(prc[i],4),'leads':int(sel.sum()),
                         'pct_leads':round(sel.mean()*100,1),'TP':tp,'FP':fp})
    else:
        print(f"  {'≥'+str(int(target*100))+'%':>9}  no alcanzable con el modelo actual")

print("\nESCENARIOS Top-K (capacidad comercial):")
print(f"  {'Corte':>6} {'Leads':>7} {'TP':>4} {'Precis':>7} {'Recall':>7} {'Lift':>6}")
for k in (0.05,0.10,0.20,0.30):
    v=business_at_k(yt,p)[k]
    print(f"  {int(k*100):>5}% {v['n']:>7,} {v['tp']:>4} {v['prec']:>7.4f} {v['rec']:>7.3f} {v['lift']:>5.2f}x")
pd.DataFrame(thr_rows).to_csv(RESULTS_DIR/'recall_thresholds.csv',index=False)
print("\nDECISIÓN OPERATIVA: si ventas puede atender X% de leads, los escenarios Top-K")
print("indican cuántas conversiones captura. El umbral de recall define el punto de corte.")


In [ ]:
# ── §10b  Gráfica: barrido de umbrales y escenarios Top-K ──
fig,ax=plt.subplots(1,2,figsize=(13,5))
thr_df=pd.DataFrame(thr_rows)
if len(thr_df):
    ax[0].bar(thr_df['objetivo_recall'].astype(str),thr_df['pct_leads'],color='#5C6BC0',alpha=.88)
    ax[0].set_title('% de leads necesarios por objetivo de recall',fontweight='bold')
    ax[0].set_xlabel('Recall objetivo');ax[0].set_ylabel('% leads priorizados')
    for i,v in enumerate(thr_df['pct_leads']): ax[0].text(i,v,f'{v:.1f}%',ha='center',va='bottom',fontsize=9)
bk=[business_at_k(yt,p)[k] for k in (0.05,0.10,0.20,0.30)]
ks=['5%','10%','20%','30%']
ax[1].bar(ks,[v['rec'] for v in bk],color='#26A69A',alpha=.85,label='Recall')
ax[1].bar(ks,[v['prec'] for v in bk],color='#FF7043',alpha=.6,label='Precision')
ax[1].set_title('Recall y precisión por corte Top-K',fontweight='bold');ax[1].legend(fontsize=8)
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_umbrales.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 11 — Evaluación operativa con simulación histórica (Monte Carlo)

Se traduce el desempeño técnico en valor comercial: **priorización por modelo** vs **selección aleatoria** (Monte Carlo, 1 000 iteraciones) sobre el conjunto de prueba histórico.


In [ ]:
# ── §11  Modelo vs selección aleatoria (Monte Carlo) ─────────────────────
N_MC=1000; rng=np.random.default_rng(SEED)
order=np.argsort(p)[::-1]; cum=np.cumsum(yt[order]); n=len(yt); base=yt.mean()
op_rows=[]
for k in (0.05,0.10,0.20,0.30):
    tn=int(np.ceil(n*k)); tp_m=int(cum[tn-1])
    rand=np.array([yt[rng.choice(n,tn,replace=False)].sum() for _ in range(N_MC)])
    lo,hi=np.percentile(rand,[2.5,97.5])
    op_rows.append({'corte':f'Top {int(k*100)}%','leads':tn,'conv_modelo':tp_m,
        'conv_aleatorio':round(rand.mean(),1),'dif_abs':int(round(tp_m-rand.mean())),
        'mejora_%':round((tp_m-rand.mean())/rand.mean()*100,1),
        'tasa_modelo_%':round(tp_m/tn*100,2),'tasa_aleatoria_%':round(base*100,2),
        'capture_rate':round(tp_m/yt.sum(),3),'lift':round(tp_m/rand.mean(),2),
        'IC95_aleatorio':f'[{lo:.0f}, {hi:.0f}]'})
op_df=pd.DataFrame(op_rows)
print("EVALUACIÓN OPERATIVA — Modelo vs Selección Aleatoria (Monte Carlo, "
      f"{N_MC} iter)\n"+"="*70)
print(op_df.to_string(index=False))
op_df.to_csv(RESULTS_DIR/'recall_operational_eval.csv',index=False)

fig,ax=plt.subplots(1,2,figsize=(14,5))
pct=np.arange(1,n+1)/n; cg=cum/yt.sum()
ax[0].plot(pct,cg,lw=2,color='#1565C0',label='Modelo')
ax[0].plot([0,1],[0,1],'k--',lw=.8,label='Aleatorio')
for k in (0.10,0.20,0.30): ax[0].axvline(k,color='gray',ls=':',lw=.7)
ax[0].set_title('Ganancia acumulada: Modelo vs Aleatorio',fontweight='bold')
ax[0].set_xlabel('% leads contactados');ax[0].set_ylabel('% conversiones capturadas');ax[0].legend()
ax[1].bar([r['corte'] for r in op_rows],[r['conv_modelo'] for r in op_rows],
          alpha=.85,color='#1565C0',label='Modelo')
ax[1].bar([r['corte'] for r in op_rows],[r['conv_aleatorio'] for r in op_rows],
          alpha=.7,color='#FF7043',label='Aleatorio (media)')
ax[1].set_title('Conversiones capturadas por corte',fontweight='bold')
ax[1].set_ylabel('Conversiones (test)');ax[1].legend()
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_operational.png',dpi=140,bbox_inches='tight');plt.show()

k20=op_df[op_df.corte=='Top 20%'].iloc[0]
print(f"\nINTERPRETACIÓN: si ventas atiende el Top 20% ({k20['leads']:,} leads), el modelo")
print(f"captura {k20['conv_modelo']} conversiones vs {k20['conv_aleatorio']} esperadas al azar")
print(f"({k20['dif_abs']} adicionales, +{k20['mejora_%']:.0f}%, lift {k20['lift']}x).")


## Sección 12 — Análisis específico de la edad

`edad` es la variable **numérica del lead** con señal más consistente. Se analiza *missingness*, distribución, conversión y *lift* por rango, e importancia sin variables de proyecto.


In [ ]:
# ── §12  Análisis de edad ────────────────────────────────────────────────
edad=pd.to_numeric(df_model['edad'],errors='coerce')
print(f"edad — missing={edad.isna().mean()*100:.1f}%  "
      f"media={edad.mean():.1f}  mediana={edad.median():.0f}  "
      f"rango=[{edad.min():.0f}, {edad.max():.0f}]")
print(f"  missing en conv={df_model.loc[df_model.target_won==1,'edad'].isna().mean()*100:.1f}%  "
      f"vs no-conv={df_model.loc[df_model.target_won==0,'edad'].isna().mean()*100:.1f}%")

bins=[0,25,30,35,40,45,50,60,200]
g=df_model.assign(eb=pd.cut(edad,bins)).groupby('eb')['target_won'].agg(n='count',conv='sum',rate='mean')
g['lift']=g['rate']/df_model['target_won'].mean()
print("\nConversión y lift por rango de edad:")
print(g.round(4).to_string())

fig,ax=plt.subplots(1,2,figsize=(13,4.5))
edad.dropna().hist(bins=40,ax=ax[0],color='#5C6BC0',alpha=.85);ax[0].set_title('Distribución de edad',fontweight='bold')
ax[0].set_xlabel('edad');ax[0].set_ylabel('frecuencia')
gr=g.reset_index();ax[1].bar(range(len(gr)),gr['lift'],color='#26A69A',alpha=.85)
ax[1].axhline(1,color='red',ls='--',lw=1);ax[1].set_xticks(range(len(gr)))
ax[1].set_xticklabels([str(x) for x in gr['eb']],rotation=45,ha='right',fontsize=8)
ax[1].set_title('Lift de conversión por rango de edad',fontweight='bold');ax[1].set_ylabel('lift')
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_edad.png',dpi=140,bbox_inches='tight');plt.show()
print("\nINTERPRETACIÓN: la conversión CRECE de forma casi monótona con la edad")
print("(los mayores de 50-60 convierten ~2-3x la tasa base). Es señal REAL del lead,")
print("estable e independiente del proyecto, y no requiere contexto de precio.")


## Sección 13 — Interpretabilidad

*Permutation importance* del modelo recomendado, comparación de importancia **con y sin** variables de proyecto, y análisis de falsos negativos / positivos para detectar posibles *proxies* del proyecto.


In [ ]:
# ── §13  Permutation importance + comparación proyecto vs lead ───────────
from sklearn.inspection import permutation_importance
feats_best=best_rec['features']
pi=permutation_importance(best_rec['pipeline'],Xte_r[feats_best],yt,
                          scoring='average_precision',n_repeats=5,random_state=SEED,n_jobs=-1)
imp=pd.Series(pi.importances_mean,index=feats_best).sort_values(ascending=False)
print(f"PERMUTATION IMPORTANCE — {best_rec['algorithm']} | {best_rec['feature_set']}")
print("(caída de PR-AUC al permutar cada variable; top 15)\n")
print(imp.head(15).round(5).to_string())

fig,ax=plt.subplots(figsize=(9,6));top=imp.head(15)[::-1]
ax.barh(range(len(top)),top.values,color='#00897B',alpha=.85)
ax.set_yticks(range(len(top)));ax.set_yticklabels(top.index,fontsize=9)
ax.set_title(f'Permutation importance — {best_rec["feature_set"]}',fontweight='bold')
ax.set_xlabel('Caída de PR-AUC')
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_importance.png',dpi=140,bbox_inches='tight');plt.show()

# Importancia de variables de proyecto en E_ProjBenchmark
e_res=[r for r in rec_results if r['feature_set']=='E_ProjBenchmark' and r['algorithm']==best_rec['algorithm']]
if e_res:
    er=e_res[0]
    pe=permutation_importance(er['pipeline'],Xte_r[er['features']],yt,
                              scoring='average_precision',n_repeats=5,random_state=SEED,n_jobs=-1)
    impe=pd.Series(pe.importances_mean,index=er['features'])
    proj_imp=impe[[c for c in er['features'] if c in PROJECT_VARS]].sort_values(ascending=False)
    print("\nImportancia de las VARIABLES DE PROYECTO (dentro de E_ProjBenchmark):")
    print(proj_imp.round(5).to_string())
    print("\n⇒ Si su importancia es baja frente a los indicadores de missingness y edad,")
    print("  se confirma que el modelo NO depende de la identidad del proyecto.")

# Falsos negativos / positivos en el corte Top-20%
k=0.20; tn=int(np.ceil(len(yt)*k)); thr_k=np.sort(p)[::-1][tn-1]
sel=p>=thr_k
fn=(~sel)&(yt==1); fp=(sel)&(yt==0)
print(f"\nAnálisis de errores en corte Top-20% (umbral={thr_k:.4f}):")
print(f"  Falsos negativos (conv. no priorizadas): {int(fn.sum())}")
print(f"  Falsos positivos (priorizados sin conv): {int(fp.sum())}")
if 'edad' in df_model.columns:
    print(f"  Edad media — FN={pd.to_numeric(Xte_r.loc[fn,'edad'],errors='coerce').mean():.1f}  "
          f"FP={pd.to_numeric(Xte_r.loc[fp,'edad'],errors='coerce').mean():.1f}")


## Sección 14 — Selección final y recomendaciones

Tabla final de candidatos y recomendación de modelo(s), variables mínimas y estrategia para leads incompletos.


In [ ]:
# ── §14  Tabla final de selección + recomendaciones ──────────────────────
CTX={'A_Lead':'ninguno','B_Lead+MissInd':'ninguno','C_TopComplete':'ninguno',
     'E_ProjBenchmark':'identidad de proyecto'}
final_rows=[]
for fsn in ['A_Lead','B_Lead+MissInd','C_TopComplete','E_ProjBenchmark']:
    sub=[r for r in rec_results if r['feature_set']==fsn]
    if not sub: continue
    b=sorted(sub,key=lambda r:(r['rec_20pct'],r['pr_auc']),reverse=True)[0]
    final_rows.append({'feature_set':fsn,'algoritmo':b['algorithm'],
        'n_vars':len(b['features']),'recall@10':round(b['rec_10pct'],3),
        'recall@20':round(b['rec_20pct'],3),'recall@30':round(b['rec_30pct'],3),
        'lift@10':round(b['lift_10pct'],2),'lift@20':round(b['lift_20pct'],2),
        'pr_auc':round(b['pr_auc'],4),'brier':round(b['brier'],4),
        'contexto_proyecto':CTX[fsn],
        'generalizable':'no (benchmark)' if fsn=='E_ProjBenchmark' else 'SÍ'})
final_df=pd.DataFrame(final_rows).sort_values('recall@20',ascending=False)
print("TABLA FINAL DE SELECCIÓN\n"+"="*95)
print(final_df.to_string(index=False))
final_df.to_csv(RESULTS_DIR/'recall_final_selection.csv',index=False)

# candidatos por nivel de generalización — todos sin ningún contexto de proyecto
pure_sets=['A_Lead','B_Lead+MissInd','C_TopComplete']
pure = final_df[final_df.feature_set.isin(pure_sets)].sort_values('recall@20',ascending=False)
rec_pure = pure.iloc[0]        # máxima generalización (cero contexto de proyecto)
rec_interpret = final_df[final_df.feature_set=='C_TopComplete']
min_vars = CORE_LEAD + ['canal_de_contacto','canal_de_anuencia']

print("\n"+"="*95+"\nRECOMENDACIONES FINALES\n"+"="*95)
print("1. MODELO PRIMARIO — máxima generalización (SIN ningún contexto de proyecto):")
print(f"   → {rec_pure['algoritmo']} | {rec_pure['feature_set']}  "
      f"(recall@20={rec_pure['recall@20']}, recall@30={rec_pure['recall@30']}, "
      f"lift@10={rec_pure['lift@10']}x, PR-AUC={rec_pure['pr_auc']})")
print("   Solo atributos del lead + indicadores de missingness ⇒ transferible sin supuestos.")
if len(rec_interpret):
    ri=rec_interpret.iloc[0]
    print(f"\n2. MODELO INTERPRETABLE ALTERNATIVO:")
    print(f"   → {ri['algoritmo']} | C_TopComplete ({ri['n_vars']} vars, recall@20={ri['recall@20']})")
print(f"\n3. VARIABLES MÍNIMAS PARA APLICAR EL MODELO:")
print(f"   {min_vars}")
print(f"   + indicadores de missingness derivados automáticamente de cada variable.")
print(f"\n4. ESTRATEGIA PARA LEADS INCOMPLETOS (modelo en dos niveles):")
print(f"   • Perfil mínimo completo  → modelo avanzado (B_Lead+MissInd).")
print(f"   • Perfil incompleto       → modelo base (edad + canal + campaña + indicadores).")
print(f"   Los indicadores de missingness permiten que AMBOS niveles usen el mismo pipeline.")
print(f"\n5. TRADE-OFF: E_ProjBenchmark NO se recomienda. Bajo validación LOGO (§8) generaliza")
print(f"   PEOR a proyectos no vistos que los modelos lead-only, confirmando que sus métricas")
print(f"   se apoyan en la identidad de los 5 proyectos actuales y no transfieren.")
print(f"\n6. NOTA sobre el feature set D_Affordability (descartado): usaba m2_minimo/precio_promedio")
print(f"   como base de ratios de 'asequibilidad'. Como esos atributos son constantes por proyecto")
print(f"   (no varían lead a lead dentro del mismo proyecto), el ratio reintroducía identidad de")
print(f"   proyecto de forma indirecta — el mismo riesgo de fuga que E_ProjBenchmark.")

In [ ]:
# ── §14b  Gráfica: tabla final de selección ──
fig,ax=plt.subplots(1,2,figsize=(14,5))
fd=final_df.sort_values('recall@20')
colors_f=['#C62828' if fs=='E_ProjBenchmark' else '#1565C0' for fs in fd['feature_set']]
ax[0].barh(fd['feature_set'],fd['recall@20'],color=colors_f,alpha=.88)
ax[0].set_title('Recall@20% por feature set (candidatos finales)',fontweight='bold')
ax[0].set_xlabel('Recall@20%')
ax[1].barh(fd['feature_set'],fd['lift@10'],color=colors_f,alpha=.88)
ax[1].axvline(1,color='gray',ls='--',lw=.8)
ax[1].set_title('Lift@10% por feature set',fontweight='bold');ax[1].set_xlabel('Lift@10%')
plt.tight_layout();plt.savefig(FIGURES_DIR/'recall_seleccion_final.png',dpi=140,bbox_inches='tight');plt.show()

## Conclusión de la corrección metodológica

**Qué se hizo.** Se reentrenaron los modelos de conversión priorizando *recall* y valor operativo, separando explícitamente **atributos del lead** de **variables de proyecto**, y validando la **generalización a proyectos no vistos** (Leave‑One‑Project‑Out).

**Hallazgos principales:**

1. **La completitud del perfil es la señal más fuerte.** Los indicadores de *missingness* (feature set B) elevan sustancialmente el desempeño frente a las variables del lead crudas, y la *permutation importance* está dominada por ellos junto con `edad`. Un lead que entrega su información demográfica/comercial está más avanzado en el embudo.
2. **El desempeño NO requiere identidad del proyecto.** El *benchmark* con variables de proyecto (E) **no supera de forma decisiva** al modelo lead‑only, y bajo LOGO el modelo lead‑only **mantiene** su desempeño en proyectos no vistos → generaliza, no memoriza.
3. **Señales de negocio reales y estables:** `edad` (creciente), `ingreso_familiar` (creciente), `razon_de_compra` (Inversión ≫ Vivienda). Estas sí varían lead a lead dentro de un mismo proyecto, a diferencia de `m2_minimo`/`precio_promedio`.
4. **Valor operativo demostrado:** el modelo recomendado captura una fracción muy superior de conversiones frente a la selección aleatoria en los cortes Top‑10/20/30 % (Monte Carlo, IC 95 %).
5. **Se descartó el feature set D_Affordability** (ratios ingreso/precio, m²/persona, ingreso/m²): usaba `m2_minimo`/`m2_maximo`/`m2_promedio`/`precio_promedio`, que son **atributos constantes del proyecto**, no del lead individual. Un ratio construido con una constante de proyecto reintroduce identidad de proyecto de forma indirecta — el mismo riesgo de fuga que `E_ProjBenchmark`, solo que disfrazado de "normalizador".

**Recomendación:** desplegar el modelo **lead‑only + indicadores de missingness** para priorización Top‑K, complementado con un **esquema de dos niveles** según la completitud del perfil. Este modelo prioriza la **capacidad de generalización a proyectos futuros** sobre métricas potencialmente infladas por la identidad de los 5 proyectos actuales. Si se requieren **probabilidades calibradas** (no solo ranking), aplicar `CalibratedClassifierCV` (isotónica).

---

## Sección 15 — Modelo entrenado solo con leads de perfil de atributos completo

> **Motivación.** Las secciones anteriores usan imputación e indicadores de *missingness* para aprovechar leads con datos incompletos. Esta sección explora el enfoque complementario: restringir el entrenamiento a los leads cuyo **perfil de atributos del lead está 100% completo** — excepto `opciones_visitadas` (56.5% missing) y `sector_referidor` (90.1% missing), que se permiten vacíos por su altísima tasa de ausencia intrínseca. Se prueban distintas combinaciones de variables y los 4 algoritmos (LR, RF, GBM, EXT), buscando el umbral que alcanza **recall ≥ 0.80** en la identificación de conversiones.

### Sección 15.1 — Subconjunto: leads con perfil de atributos completo

In [ ]:
# ── §15.1  Subconjunto: leads con perfil de atributos completo ───────────
LEAD_EXCEPT = {'opciones_visitadas','sector_referidor'}
LEAD_REQUIRED = [c for c in LEAD_ONLY if c not in LEAD_EXCEPT]
cpl_mask = df_model[LEAD_REQUIRED].notna().all(axis=1)
df_complete = df_model.loc[cpl_mask].copy()

# sector_referidor no tenia indicador de missingness propio (no esta en HIVAL) - se crea aqui
df_complete['sector_referidor__miss'] = df_complete['sector_referidor'].isna().astype(int)
MISS_IND_CPL = MISS_IND + ['sector_referidor__miss']

print("SUBCONJUNTO: PERFIL DE ATRIBUTOS DEL LEAD 100% COMPLETO")
print("="*70)
print(f"  Variables exigidas completas ({len(LEAD_REQUIRED)}): {LEAD_REQUIRED}")
print(f"  Excepciones permitidas (pueden estar vacias): {sorted(LEAD_EXCEPT)}")
print(f"\n  Registros                   : {len(df_complete):,}  ({cpl_mask.mean()*100:.1f}% del calificado)")
print(f"  Conversiones                : {int(df_complete['target_won'].sum()):,}")
print(f"  Tasa de conversion          : {df_complete['target_won'].mean()*100:.3f}%")
print(f"  Tasa base (todo calificado) : {df_model['target_won'].mean()*100:.3f}%")
print(f"  opciones_visitadas - % vacio en este subconjunto: {df_complete['opciones_visitadas'].isna().mean()*100:.1f}%")
print(f"  sector_referidor   - % vacio en este subconjunto: {df_complete['sector_referidor'].isna().mean()*100:.1f}%")

fig,ax=plt.subplots(1,2,figsize=(12,4.5))
labels=['Calificado\n(base)','Perfil lead\ncompleto']
regs=[len(df_model), len(df_complete)]
tasas=[df_model['target_won'].mean()*100, df_complete['target_won'].mean()*100]
ax[0].bar(labels,regs,color=['#5C6BC0','#26A69A'],alpha=.9)
for i,v in enumerate(regs): ax[0].text(i,v,f'{v:,}',ha='center',va='bottom',fontsize=9)
ax[0].set_title('Registros disponibles',fontweight='bold');ax[0].set_ylabel('Registros')
ax[1].bar(labels,tasas,color=['#5C6BC0','#26A69A'],alpha=.9)
for i,v in enumerate(tasas): ax[1].text(i,v,f'{v:.2f}%',ha='center',va='bottom',fontsize=9)
ax[1].set_title('Tasa de conversion',fontweight='bold');ax[1].set_ylabel('Tasa conv. (%)')
plt.tight_layout();plt.savefig(FIGURES_DIR/'completo_universo.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 15.2 — Combinaciones de variables

Sobre este subconjunto de perfil completo se construyen distintas combinaciones de variables, de menor a mayor cobertura:

| Set | Variables incluidas | n |
|---|---|---|
| **I1_CoreDemo** | Demográficas básicas (edad, sexo, estado civil, hijos, núcleo familiar, ingreso, razón de compra) | 7 |
| **I2_CoreDemo+Comercial** | I1 + tipo de unidad, opciones visitadas, participantes en crédito | 10 |
| **I3_CoreDemo+Canal** | I1 + canal de contacto/anuencia, fuente, campaña (plataforma/tipo/audiencia) | 13 |
| **I4_LeadFull** | Todas las variables de atributo lead (`LEAD_ONLY` completo) | 20 |
| **I5_LeadFull+MissInd** | I4 + indicadores de ausencia (incl. `opciones_visitadas` y `sector_referidor`) | 29 |

In [ ]:
# ── §15.2  Definición de combinaciones de variables ───────────────────────
I1 = ['edad','sexo','estado_civil','tiene_hijos','nucleo_familiar','ingreso_familiar','razon_de_compra']
I2 = I1 + ['tipo_unidad','opciones_visitadas','participantes_en_credito']
I3 = I1 + ['canal_de_contacto','canal_de_anuencia','fuente_pt2',
           'campania_plataforma','campania_tipo','campania_audiencia']
I4 = list(LEAD_ONLY)
I5 = I4 + MISS_IND_CPL

COMPLETE_FS = {
 'I1_CoreDemo'          : [c for c in I1 if c in df_complete.columns],
 'I2_CoreDemo+Comercial': [c for c in I2 if c in df_complete.columns],
 'I3_CoreDemo+Canal'    : [c for c in I3 if c in df_complete.columns],
 'I4_LeadFull'          : [c for c in I4 if c in df_complete.columns],
 'I5_LeadFull+MissInd'  : [c for c in I5 if c in df_complete.columns],
}
print("Combinaciones de variables definidas:")
for k,v in COMPLETE_FS.items(): print(f"  {k:24s}: {len(v)} variables")

## Sección 15.3 — Entrenamiento: modelos × combinaciones de variables

Se entrenan los 4 algoritmos (LR, RF, GBM, EXT) sobre cada combinación de variables, con `class_weight='balanced'`. Dado el tamaño reducido del subconjunto (pocas conversiones en términos absolutos), se agrega un experimento adicional con SMOTE sobre la combinación más completa (I5).

In [ ]:
# ── §15.3  Split + entrenamiento de todas las combinaciones ──────────────
y_cpl = df_complete['target_won'].values
cpl_idx = np.arange(len(df_complete))
cpl_tr, cpl_te = train_test_split(cpl_idx, test_size=0.20, random_state=SEED, stratify=y_cpl)
Xtr_cpl, Xte_cpl = df_complete.iloc[cpl_tr], df_complete.iloc[cpl_te]
ytr_cpl, yte_cpl = y_cpl[cpl_tr], y_cpl[cpl_te]
print(f"Split perfil completo — train={len(cpl_tr):,} test={len(cpl_te):,} | "
      f"base test={yte_cpl.mean()*100:.3f}%  ({int(yte_cpl.sum())} conversiones)")

def run_cpl_grid(feature_sets, use_smote=False):
    res=[]
    for fsn,feats in feature_sets.items():
        feats=[f for f in feats if f in df_complete.columns]
        cat,num=identify_col_types(df_complete,feats)
        for an,af in ALGORITHMS.items():
            ordv=(an!='LR')
            pipe=build_pipeline(af(),cat,num,use_smote=use_smote,ordinal=ordv)
            try:
                pipe.fit(Xtr_cpl[feats],ytr_cpl)
                p=pipe.predict_proba(Xte_cpl[feats])[:,1]
                bm=business_at_k(yte_cpl,p)
                res.append({'feature_set':fsn,'algorithm':an,
                    'experiment':'B_SMOTE' if use_smote else 'A_NoSMOTE',
                    'pr_auc':average_precision_score(yte_cpl,p),'roc_auc':roc_auc_score(yte_cpl,p),
                    'brier':brier_score_loss(yte_cpl,p),
                    'rec_10pct':bm[0.10]['rec'],'rec_20pct':bm[0.20]['rec'],'rec_30pct':bm[0.30]['rec'],
                    'lift_10pct':bm[0.10]['lift'],'lift_20pct':bm[0.20]['lift'],
                    'prec_10pct':bm[0.10]['prec'],'n_features':len(feats),
                    'pipeline':pipe,'y_prob':p,'features':feats})
                print(f"  ok {an:4s} | {fsn:22s} | PR={res[-1]['pr_auc']:.4f} "
                      f"ROC={res[-1]['roc_auc']:.3f} rec@20={bm[0.20]['rec']:.3f} "
                      f"lift@10={bm[0.10]['lift']:.2f}x")
            except Exception as e:
                print(f"  ERR {an:4s} | {fsn:22s} | {e}")
    return res

print("\n=== Experimento A (sin SMOTE, class_weight balanced) ===")
cpl_results = run_cpl_grid(COMPLETE_FS, use_smote=False)
print("\n=== Experimento B (SMOTE, combinación I5_LeadFull+MissInd) ===")
cpl_results += run_cpl_grid({'I5_LeadFull+MissInd': COMPLETE_FS['I5_LeadFull+MissInd']}, use_smote=True)

In [ ]:
# ── §15.4  Tabla comparativa + gráfica ────────────────────────────────────
cpl_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ('pipeline','y_prob','features')} for r in cpl_results])
cpl_df = cpl_df.sort_values(['rec_20pct','pr_auc'],ascending=False).reset_index(drop=True)
show=['feature_set','algorithm','experiment','pr_auc','roc_auc','brier',
      'rec_10pct','rec_20pct','rec_30pct','lift_10pct','lift_20pct']
print("MODELOS — PERFIL DE LEAD COMPLETO — ordenados por Recall@20%:")
print(cpl_df[show].round(4).to_string(index=False))
cpl_df[show].to_csv(RESULTS_DIR/'completo_models_comparison.csv', index=False)

fig,ax=plt.subplots(1,2,figsize=(14,5))
piv_pr=cpl_df.pivot_table(index='feature_set',columns='algorithm',values='pr_auc',aggfunc='max')
piv_pr.plot(kind='bar',ax=ax[0],alpha=.88)
ax[0].set_title('PR-AUC por combinación y algoritmo',fontweight='bold')
ax[0].set_ylabel('PR-AUC');ax[0].tick_params(axis='x',rotation=30);ax[0].legend(fontsize=8)
piv_rec=cpl_df.pivot_table(index='feature_set',columns='algorithm',values='rec_20pct',aggfunc='max')
piv_rec.plot(kind='bar',ax=ax[1],alpha=.88)
ax[1].set_title('Recall@20% por combinación y algoritmo',fontweight='bold')
ax[1].set_ylabel('Recall@20%');ax[1].tick_params(axis='x',rotation=30);ax[1].legend(fontsize=8)
plt.tight_layout();plt.savefig(FIGURES_DIR/'completo_modelos_comparacion.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 15.5 — Barrido de umbral hasta Recall ≥ 0.80

Para cada combinación de modelo/variables se busca el umbral que alcanza **recall ≥ 0.80**, y se reporta qué porcentaje de leads es necesario priorizar y con qué precisión. Un recall de 0.80 es matemáticamente alcanzable por cualquier modelo con solo seleccionar suficientes leads (en el límite, 100%); lo que diferencia a los modelos es el **costo operativo** — el % de leads que hay que contactar para llegar ahí.

In [ ]:
# ── §15.5  Umbral para recall ≥ 0.80 por combinación ──────────────────────
TARGET_RECALL = 0.80
thr80_rows=[]
for r in cpl_results:
    yt_r, p_r = yte_cpl, r['y_prob']
    prc,rcc,thr = precision_recall_curve(yt_r,p_r)
    idx=np.where(rcc[:-1]>=TARGET_RECALL)[0]
    if len(idx):
        i=idx[-1]; t=thr[i]; sel=p_r>=t
        tp=int(((sel)&(yt_r==1)).sum()); fp=int(((sel)&(yt_r==0)).sum())
        thr80_rows.append({'feature_set':r['feature_set'],'algorithm':r['algorithm'],
            'experiment':r['experiment'],'umbral':round(t,4),'recall':round(rcc[i],3),
            'precision':round(prc[i],4),'leads':int(sel.sum()),
            'pct_leads':round(sel.mean()*100,1),'TP':tp,'FP':fp,'alcanzable':True})
    else:
        thr80_rows.append({'feature_set':r['feature_set'],'algorithm':r['algorithm'],
            'experiment':r['experiment'],'umbral':np.nan,'recall':np.nan,
            'precision':np.nan,'leads':np.nan,'pct_leads':np.nan,'TP':np.nan,'FP':np.nan,
            'alcanzable':False})

thr80_df = pd.DataFrame(thr80_rows).sort_values('pct_leads')
print(f"UMBRAL PARA RECALL >= {TARGET_RECALL:.0%} - por modelo y combinación de variables")
print("="*90)
print(thr80_df.to_string(index=False))
thr80_df.to_csv(RESULTS_DIR/'completo_recall80_umbrales.csv', index=False)

n_ok = int(thr80_df['alcanzable'].sum())
print(f"\n{n_ok}/{len(thr80_df)} combinaciones alcanzan recall >= {TARGET_RECALL:.0%} "
      f"dentro del rango de umbrales evaluado.")

In [ ]:
# ── §15.5b  Gráfica: % de leads necesarios para recall ≥ 0.80 ────────────
plot_df = thr80_df.dropna(subset=['pct_leads']).copy()
plot_df['label'] = plot_df['feature_set'] + ' | ' + plot_df['algorithm'] + ' | ' + plot_df['experiment']
plot_df = plot_df.sort_values('pct_leads')
fig,ax=plt.subplots(1,2,figsize=(14,max(5,0.35*len(plot_df))))
ax[0].barh(plot_df['label'],plot_df['pct_leads'],color='#5C6BC0',alpha=.88)
ax[0].set_title(f'% de leads necesarios para recall ≥ {TARGET_RECALL:.0%}',fontweight='bold')
ax[0].set_xlabel('% leads priorizados');ax[0].invert_yaxis()
ax[1].barh(plot_df['label'],plot_df['precision'],color='#FF7043',alpha=.88)
ax[1].set_title(f'Precisión al alcanzar recall ≥ {TARGET_RECALL:.0%}',fontweight='bold')
ax[1].set_xlabel('Precisión');ax[1].invert_yaxis()
plt.tight_layout();plt.savefig(FIGURES_DIR/'completo_recall80_umbrales.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 15.6 — Selección del mejor modelo (Recall ≥ 0.80)

Entre las combinaciones que alcanzan recall ≥ 0.80, se elige la que requiere **menor % de leads** (mayor eficiencia operativa); en caso de empate, la de mayor precisión.

In [ ]:
# ── §15.6  Mejor modelo con recall ≥ 0.80 + curvas ────────────────────────
reach80 = thr80_df[thr80_df['alcanzable']].sort_values(['pct_leads','precision'],ascending=[True,False])
best80 = reach80.iloc[0]
print(f"MEJOR MODELO PARA RECALL >= {TARGET_RECALL:.0%}:")
print(f"  → {best80['algorithm']} | {best80['feature_set']} | {best80['experiment']}")
print(f"  Umbral={best80['umbral']}  Recall={best80['recall']}  Precisión={best80['precision']}")
print(f"  Leads a contactar={int(best80['leads']):,} ({best80['pct_leads']}% del subconjunto de prueba)")
print(f"  TP={int(best80['TP'])}  FP={int(best80['FP'])}")

best80_res = [r for r in cpl_results if r['feature_set']==best80['feature_set']
              and r['algorithm']==best80['algorithm'] and r['experiment']==best80['experiment']][0]
p80 = best80_res['y_prob']; yt80 = yte_cpl

fig,ax=plt.subplots(1,2,figsize=(13,5))
order=np.argsort(p80)[::-1]; cg=np.cumsum(yt80[order])/yt80.sum(); pct=np.arange(1,len(yt80)+1)/len(yt80)
ax[0].plot(pct,cg,lw=2,color='#1565C0',label='Modelo')
ax[0].plot([0,1],[0,1],'k--',lw=.8,label='Aleatorio')
ax[0].axhline(TARGET_RECALL,color='red',ls=':',lw=1,label=f'Recall {TARGET_RECALL:.0%}')
ax[0].axvline(best80['pct_leads']/100,color='red',ls=':',lw=1)
ax[0].set_title('Curva de ganancia — mejor modelo',fontweight='bold')
ax[0].set_xlabel('% leads contactados');ax[0].set_ylabel('% conversiones capturadas');ax[0].legend(fontsize=8)

prc,rcc,_=precision_recall_curve(yt80,p80)
ax[1].plot(rcc,prc,lw=2,color='#2E7D32',label=f'AP={average_precision_score(yt80,p80):.3f}')
ax[1].axvline(TARGET_RECALL,color='red',ls=':',lw=1,label=f'Recall {TARGET_RECALL:.0%}')
ax[1].set_title('Curva Precision-Recall — mejor modelo',fontweight='bold')
ax[1].set_xlabel('Recall');ax[1].set_ylabel('Precision');ax[1].legend(fontsize=8)
plt.suptitle(f"Modelo recomendado (recall>={TARGET_RECALL:.0%}): {best80['algorithm']} | {best80['feature_set']} | {best80['experiment']}",
             fontsize=12,fontweight='bold')
plt.tight_layout();plt.savefig(FIGURES_DIR/'completo_mejor_modelo_curvas.png',dpi=140,bbox_inches='tight');plt.show()

## Conclusión — Sección 15

**Qué se hizo.** Se entrenaron LR, RF, GBM y EXT sobre 5 combinaciones de variables de atributo lead, restringiendo el universo a leads con perfil **100% completo** (excepto `opciones_visitadas` y `sector_referidor`), y se buscó el umbral que alcanza **recall ≥ 0.80** para cada combinación.

**Cómo leer los resultados:** la tabla de la Sección 15.5 (`completo_recall80_umbrales.csv`) está ordenada por % de leads necesario para alcanzar el objetivo de recall — el modelo recomendado (Sección 15.6) es el de menor % de leads entre los que sí alcanzan el objetivo.

**Trade-off frente a las secciones anteriores.** Este enfoque usa un universo considerablemente más pequeño (perfil 100% completo) que el modelo con indicadores de missingness de la Sección 9/14, lo que reduce el volumen de leads elegibles pero elimina por completo la necesidad de imputación. Debe entenderse como un **modelo especializado** para el subconjunto de leads que sí completan su perfil, complementario — no un reemplazo — del modelo primario recomendado en la Sección 14.

---

## Sección 16 — Perfil de leads que convierten: ¿qué columnas están casi siempre completas?

> **Objetivo.** Para los leads que sí convierten (`target_won=1`), ¿qué variables tienden a estar informadas? Esto permite distinguir variables **confiables** (poco missing entre quienes convierten, útiles para perfilar al comprador típico) de variables que son **artefactos posteriores a la venta** (solo se llenan porque el lead ya compró — riesgo de fuga de información si se usan como predictoras).

### Sección 16.1 — Missingness por columna: conversión vs. no conversión

In [ ]:
# ── §16.1  Missingness por columna: conversión vs no conversión ──────────
META_EXCL = {'id_trato','person_id','user_id','fecha_creacion_trato','fuente_dataset',
             'target_won','proyecto_fuera_alcance'}
POST_SALE_VARS = sorted([c for c in df_model.columns if c.endswith('_label')])
ALL_COLS = [c for c in df_model.columns if c not in META_EXCL]

def _col_group(c):
    if c in PROJECT_VARS: return 'Proyecto (constante)'
    if c in POST_SALE_VARS: return 'Post-venta (posible fuga)'
    if c in LEAD_ONLY or c == 'campania': return 'Atributo lead'
    return 'Otro'

conv_mask = df_model['target_won']==1
prof_rows=[]
for c in ALL_COLS:
    prof_rows.append({
        'variable': c,
        'grupo': _col_group(c),
        'miss_conv_%': round(df_model.loc[conv_mask,c].isna().mean()*100,1),
        'miss_noconv_%': round(df_model.loc[~conv_mask,c].isna().mean()*100,1),
    })
prof_df = pd.DataFrame(prof_rows).sort_values('miss_conv_%').reset_index(drop=True)
prof_df['gap_pp'] = (prof_df['miss_noconv_%']-prof_df['miss_conv_%']).round(1)

print(f"MISSINGNESS POR COLUMNA - conversión (n={int(conv_mask.sum())}) vs no conversión (n={int((~conv_mask).sum())})")
print("="*90)
print(prof_df.to_string(index=False))
prof_df.to_csv(RESULTS_DIR/'perfil_convierten_missingness.csv', index=False)

In [ ]:
# ── §16.1b  Gráfica: missingness por columna entre conversiones ──────────
fig,ax=plt.subplots(figsize=(10,15))
gcolors={'Proyecto (constante)':'#78909C','Atributo lead':'#1565C0',
         'Post-venta (posible fuga)':'#C62828','Otro':'#9E9E9E'}
colors=[gcolors[g] for g in prof_df['grupo']]
ax.barh(prof_df['variable'],prof_df['miss_conv_%'],color=colors,alpha=.88)
ax.axvline(15,color='k',ls='--',lw=.8,label='Umbral 15%')
ax.invert_yaxis()
ax.set_title('% de valores faltantes entre leads que SÍ convierten, por variable',fontweight='bold')
ax.set_xlabel('% missing (target_won=1)')
handles=[plt.Rectangle((0,0),1,1,color=c) for c in gcolors.values()]
ax.legend(handles,gcolors.keys(),fontsize=8,loc='lower right')
plt.tight_layout();plt.savefig(FIGURES_DIR/'perfil_convierten_missingness.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 16.2 — Columnas casi siempre completas entre quienes convierten

Se consideran "casi siempre completas" las variables con **menos de 15% de valores faltantes** entre los leads que convierten.

> **Advertencia de fuga de información.** Varias columnas `*_label` (asignación de apartamento, parqueo, bodega, encuestas CSI/NPS post-venta) aparecen mucho más completas entre quienes convierten simplemente porque **se generan después de la venta** — no son atributos del lead antes de decidir, sino consecuencia de haber comprado. Usarlas como predictoras sería fuga de información (*target leakage*). Estas columnas YA están excluidas de todos los feature sets de modelado (Secciones 6–15); aquí se documentan solo para dejar constancia explícita del motivo.

In [ ]:
# ── §16.2  Columnas casi siempre completas (<15% missing entre conversiones) ─
THRESH = 15.0
almost_complete = prof_df[prof_df['miss_conv_%']<THRESH].copy()
print(f"COLUMNAS CASI SIEMPRE COMPLETAS ENTRE CONVERSIONES (<{THRESH:.0f}% missing)")
print("="*80)
for grp in ['Proyecto (constante)','Atributo lead','Post-venta (posible fuga)','Otro']:
    sub=almost_complete[almost_complete['grupo']==grp]
    if len(sub):
        print(f"\n▸ {grp} ({len(sub)} columnas):")
        print(sub[['variable','miss_conv_%','miss_noconv_%','gap_pp']].to_string(index=False))

usable = almost_complete[almost_complete['grupo']=='Atributo lead']['variable'].tolist()
print(f"\n{'='*80}\nCOLUMNAS DE ATRIBUTO LEAD UTILIZABLES SIN FUGA (casi completas): {usable}")

## Sección 16.3 — Perfil descriptivo: ¿cómo son los leads que convierten?

Para las variables de atributo lead identificadas en §16.2 (sin riesgo de fuga), se compara la distribución de categorías entre convertidos y no convertidos.

In [ ]:
# ── §16.3  Distribución de categorías: convierte vs no convierte ─────────
profile_tables = {}
for c in usable:
    t = pd.crosstab(df_model[c].fillna('(vacío)'), df_model['target_won'], normalize='columns')*100
    t.columns=['no_convierte_%','convierte_%']
    t['dif_pp']=(t['convierte_%']-t['no_convierte_%']).round(1)
    t=t.round(1).sort_values('convierte_%',ascending=False)
    profile_tables[c]=t
    print(f"\n▸ {c}")
    print(t.to_string())

In [ ]:
# ── §16.3b  Gráfica: distribución de categorías por variable ─────────────
n=len(usable)
ncols=2; nrows=int(np.ceil(n/ncols)) if n else 1
fig,axes=plt.subplots(nrows,ncols,figsize=(13,4.2*nrows))
axes=np.array(axes).reshape(-1)
for ax,c in zip(axes,usable):
    t=profile_tables[c].head(6)
    x=np.arange(len(t))
    ax.bar(x-0.2,t['no_convierte_%'],width=0.4,color='#B0BEC5',alpha=.9,label='No convierte')
    ax.bar(x+0.2,t['convierte_%'],width=0.4,color='#2E7D32',alpha=.9,label='Convierte')
    ax.set_xticks(x);ax.set_xticklabels(t.index,rotation=40,ha='right',fontsize=8)
    ax.set_title(c,fontweight='bold',fontsize=10);ax.set_ylabel('%')
    ax.legend(fontsize=7)
for ax in axes[n:]: ax.axis('off')
plt.tight_layout();plt.savefig(FIGURES_DIR/'perfil_convierten_distribuciones.png',dpi=140,bbox_inches='tight');plt.show()

## Conclusión — Sección 16

**Columnas confiables (sin fuga) casi siempre completas entre quienes convierten (<15% missing):** `fuente_pt2`, `canal_de_anuencia`, `canal_de_contacto`, `sexo`, `tipo_unidad`, `opciones_visitadas`, `razon_de_compra`, `estado_civil`, `lugar_vivienda`, `lugar_trabajo`, `ingreso_familiar`, `rango_edad_modelo`, `participantes_en_credito`, `edad`, `nucleo_familiar` (15 variables — ver §16.2). Las variables de proyecto (`proyecto`, `zona_proyecto`, precios, m²) también aparecen "siempre completas", pero son **constantes por proyecto** — no aportan información para distinguir entre leads del mismo proyecto (ver Sección 6).

**Perfil del lead que convierte (§16.3):** frente a quienes no convierten, el lead que sí convierte tiende a:
- Tener su **perfil informado** — la diferencia más marcada de todas: quienes convierten casi nunca dejan campos vacíos (p. ej. `opciones_visitadas` vacío: 10.0% en conversión vs 57.3% en no conversión; `edad` vacío: 13.4% vs 51.2%).
- Ser **hombre** con más frecuencia relativa (60.5% de conversiones vs 43.2% de no conversiones).
- Comprar por **inversión** en mayor proporción (41.2% de conversiones vs 15.8% de no conversiones — más del doble), aunque "Vivienda" sigue siendo mayoritaria en ambos grupos.
- Estar **casado/a** o **soltero/a** casi por igual (42.9% y 42.2% de conversiones respectivamente), muy por encima de la proporción de "Futuros Cónyuges" (2.6%).
- Provenir en mayor proporción de canales de **referencia** (14.2% de conversiones vs 2.4% de no conversiones) y menos de **Facebook** (34.3% vs 62.8%) — el canal más masivo no es el más efectivo relativo.

**Columnas a evitar como predictoras:** todas las `*_label` (asignación de apartamento/parqueo/bodega, encuestas CSI/NPS) son artefactos posteriores a la venta — su mayor completitud entre convertidos refleja la **consecuencia** de convertir, no una causa observable antes de la decisión. Ninguna de estas columnas fue incluida en los feature sets de modelado de las Secciones 6–15.

**Utilidad práctica.** Las variables identificadas en §16.2/§16.3 son las más confiables para describir el perfil del comprador típico en reportes o dashboards (poco missing, sin fuga), y ya se usan como features de modelado en las secciones anteriores (p. ej. `sexo`, `canal_de_contacto`, `edad`, `razon_de_compra`).

---

## Sección 17 — Universo de leads con fuente, canal, y perfil demográfico completo

> **Pregunta.** ¿Cuántos leads tienen informadas simultáneamente `fuente_pt2`, `canal_de_anuencia`, `canal_de_contacto`, `sexo`, `razon_de_compra`, `estado_civil`, `ingreso_familiar`, `edad`, `tiene_hijos` y `nucleo_familiar`? Este es un subconjunto de 10 variables (más liviano que el perfil 100% completo de la Sección 15, que exige 18) — útil para entender cuánto se gana en tamaño de muestra al relajar las exigencias de completitud.

### Sección 17.1 — Definición del universo y comparación de tamaños

In [ ]:
# ── §17.1  Definición del universo y comparación de tamaños ──────────────
VARS_17 = ['fuente_pt2','canal_de_anuencia','canal_de_contacto','sexo','razon_de_compra',
           'estado_civil','ingreso_familiar','edad','tiene_hijos','nucleo_familiar']
VARS_17 = [c for c in VARS_17 if c in df_model.columns]
mask_17 = df_model[VARS_17].notna().all(axis=1)
df_u17 = df_model.loc[mask_17].copy()

print("UNIVERSO: FUENTE + CANAL + PERFIL DEMOGRÁFICO COMPLETO")
print("="*70)
print(f"  Variables exigidas completas ({len(VARS_17)}): {VARS_17}")
print(f"\n  Registros                   : {len(df_u17):,}  ({mask_17.mean()*100:.1f}% del calificado)")
print(f"  Conversiones                : {int(df_u17['target_won'].sum()):,}")
print(f"  Tasa de conversión          : {df_u17['target_won'].mean()*100:.3f}%")
print(f"  Tasa base (todo calificado) : {df_model['target_won'].mean()*100:.3f}%")
print(f"  vs. universo Sección 15 (perfil 100% completo, 18 vars): {len(df_complete):,} registros, "
      f"{df_complete['target_won'].mean()*100:.3f}% conversión")

fig,ax=plt.subplots(1,2,figsize=(13,4.5))
labels=['Calificado\n(base)','Fuente+canal+\ndemográfico (17)','Perfil lead\ncompleto (Sec.15)']
regs=[len(df_model), len(df_u17), len(df_complete)]
tasas=[df_model['target_won'].mean()*100, df_u17['target_won'].mean()*100, df_complete['target_won'].mean()*100]
cols17=['#5C6BC0','#26A69A','#FF7043']
ax[0].bar(labels,regs,color=cols17,alpha=.9)
for i,v in enumerate(regs): ax[0].text(i,v,f'{v:,}',ha='center',va='bottom',fontsize=9)
ax[0].set_title('Registros disponibles',fontweight='bold');ax[0].set_ylabel('Registros')
ax[1].bar(labels,tasas,color=cols17,alpha=.9)
for i,v in enumerate(tasas): ax[1].text(i,v,f'{v:.2f}%',ha='center',va='bottom',fontsize=9)
ax[1].set_title('Tasa de conversión',fontweight='bold');ax[1].set_ylabel('Tasa conv. (%)')
plt.tight_layout();plt.savefig(FIGURES_DIR/'universo17_comparacion.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 17.2 — Efecto marginal de cada variable exigida

¿Qué tanto se reduce el universo al agregar cada variable, una por una, en orden de menor a mayor *missingness*? Esto identifica **cuál variable es el verdadero cuello de botella**.

In [ ]:
# ── §17.2  Embudo: reducción acumulada del universo por variable ─────────
miss_order = df_model[VARS_17].isna().mean().sort_values().index.tolist()
funnel_rows=[]
acc_cols=[]
for v in miss_order:
    acc_cols.append(v)
    m = df_model[acc_cols].notna().all(axis=1)
    funnel_rows.append({'variable_agregada':v,'miss_propia_%':round(df_model[v].isna().mean()*100,1),
                        'n_acumulado':int(m.sum()),'pct_acumulado':round(m.mean()*100,1),
                        'conversiones':int(df_model.loc[m,'target_won'].sum())})
funnel_df = pd.DataFrame(funnel_rows)
print("EMBUDO DE COMPLETITUD (variables agregadas en orden de menor a mayor missingness)")
print("="*80)
print(funnel_df.to_string(index=False))
funnel_df.to_csv(RESULTS_DIR/'universo17_embudo.csv', index=False)

fig,ax=plt.subplots(figsize=(10,5))
ax.bar(funnel_df['variable_agregada'],funnel_df['n_acumulado'],color='#1565C0',alpha=.85)
for i,v in enumerate(funnel_df['n_acumulado']): ax.text(i,v,f'{v:,}',ha='center',va='bottom',fontsize=8)
ax.set_title('Registros restantes al exigir cada variable adicional (acumulado)',fontweight='bold')
ax.set_ylabel('Registros restantes');ax.set_xticklabels(funnel_df['variable_agregada'],rotation=35,ha='right')
plt.tight_layout();plt.savefig(FIGURES_DIR/'universo17_embudo.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 17.3 — Perfil y lift de conversión dentro de este universo

Para las variables categóricas de este universo, se calcula la tasa de conversión y el *lift* por categoría (mismo criterio de la Sección 5), ahora **dentro** del universo de 13 mil leads con fuente/canal/demográficas completas.

In [ ]:
# ── §17.3  Lift de conversión por categoría dentro del universo ──────────
base_rate_17 = df_u17['target_won'].mean()
cat_vars_17 = ['fuente_pt2','canal_de_anuencia','canal_de_contacto','sexo',
               'razon_de_compra','estado_civil']
for c in cat_vars_17:
    t = df_u17.groupby(c)['target_won'].agg(n='count',conv='sum',rate='mean')
    t = t[t['n']>=50]; t['lift']=(t['rate']/base_rate_17).round(3)
    t = t.sort_values('lift',ascending=False).round(4)
    print(f"\n▸ {c}")
    print(t.to_string())

edad_17 = pd.to_numeric(df_u17['edad'],errors='coerce')
g_edad = df_u17.assign(eb=pd.cut(edad_17,[0,25,30,35,40,45,50,60,200])).groupby('eb')['target_won'].agg(n='count',conv='sum',rate='mean')
g_edad['lift']=(g_edad['rate']/base_rate_17).round(3)
print("\n▸ edad (por rango)")
print(g_edad.round(4).to_string())

In [ ]:
# ── §17.3b  Gráfica: lift por categoría dentro del universo ──────────────
fig,axes=plt.subplots(2,2,figsize=(13,9))
specs = [('razon_de_compra',50,axes[0,0]),('estado_civil',50,axes[0,1]),
         ('sexo',50,axes[1,0]),('fuente_pt2',50,axes[1,1])]
for c,minn,ax in specs:
    t = df_u17.groupby(c)['target_won'].agg(n='count',rate='mean')
    t = t[t['n']>=minn]; t['lift']=t['rate']/base_rate_17
    t = t.sort_values('lift',ascending=False)
    ax.bar(range(len(t)),t['lift'],color='#00897B',alpha=.85)
    ax.axhline(1,color='red',ls='--',lw=1)
    ax.set_xticks(range(len(t)));ax.set_xticklabels(t.index,rotation=40,ha='right',fontsize=8)
    ax.set_title(f'Lift por {c} (universo Sec.17)',fontweight='bold');ax.set_ylabel('lift')
plt.tight_layout();plt.savefig(FIGURES_DIR/'universo17_lift_categorias.png',dpi=140,bbox_inches='tight');plt.show()

## Conclusión — Sección 17

**Tamaño del universo.** Exigir completas solo estas 10 variables (fuente, canal de contacto/anuencia, y perfil demográfico básico) deja **13,038 leads (30.7% del calificado)**, con **563 conversiones** y una tasa de conversión de **4.32%** — más del doble de la tasa base (1.74%). Es un universo **~1.5× más grande** que el de la Sección 15 (perfil 100% completo con 18 variables, 8,609 leads), a costa de no exigir completitud en variables de canal/campaña adicionales (`campania_*`, `tipo_unidad`, `opciones_visitadas`, `participantes_en_credito`, `lugar_trabajo`, `lugar_vivienda`, `sector_referidor`).

**Cuello de botella.** El embudo de §17.2 muestra qué variable reduce más el universo al agregarla — típicamente `tiene_hijos` y `nucleo_familiar` son las más restrictivas dentro de este grupo de 10, consistentes con su alto *missingness* global (63.9% y 55.6% respectivamente, ver Sección 3).

**Utilidad.** Este universo es un punto intermedio razonable entre el modelo primario de la Sección 14 (indicadores de missingness, sin exigir completitud) y el modelo de perfil 100% completo de la Sección 15 (muy restrictivo, 8,609 leads). Puede usarse como base para un experimento adicional de recall si se busca más volumen que en la Sección 15 sin perder la garantía de datos observados (sin imputación).

---

## Sección 18 — Modelo sobre el universo fuente+canal+demográfico (recall ≥ 0.70)

> **Motivación.** La Sección 15 exigía el perfil de lead 100% completo (18 variables → 8,609 leads, 262 conversiones) y no logró un punto de corte eficiente para recall ≥ 0.80. Aquí se repite el mismo experimento — distintas combinaciones de variables × 4 algoritmos — pero sobre el universo más amplio de la Sección 17 (10 variables: fuente, canal de contacto/anuencia, sexo, razón de compra, estado civil, ingreso familiar, edad, tiene hijos, núcleo familiar → **13,038 leads, 563 conversiones**), y con un objetivo de recall más realista: **≥ 0.70**.

### Sección 18.1 — Universo y preparación de indicadores de missingness

In [ ]:
# ── §18.1  Universo (reutiliza df_u17 de la Sección 17) + indicadores ────
df_u18 = df_u17.copy()
df_u18['campania_plataforma__miss'] = df_u18['campania_plataforma'].isna().astype(int)
df_u18['sector_referidor__miss']    = df_u18['sector_referidor'].isna().astype(int)
MISS_IND_18 = MISS_IND + ['campania_plataforma__miss','sector_referidor__miss']

print("UNIVERSO SECCIÓN 18 (= Sección 17: fuente + canal + demográfico completo)")
print("="*70)
print(f"  Registros    : {len(df_u18):,}")
print(f"  Conversiones : {int(df_u18['target_won'].sum()):,}")
print(f"  Tasa conv.   : {df_u18['target_won'].mean()*100:.3f}%")
print("\n  Missingness residual de otras variables de atributo lead dentro de este universo:")
for c in ['tipo_unidad','opciones_visitadas','campania_plataforma','campania_tipo',
          'campania_audiencia','participantes_en_credito','lugar_trabajo',
          'lugar_vivienda','sector_referidor']:
    if c in df_u18.columns:
        print(f"    {c:24s} {df_u18[c].isna().mean()*100:5.1f}% missing")

## Sección 18.2 — Combinaciones de variables

| Set | Variables incluidas | n |
|---|---|---|
| **J1_CoreDemo** | Demográficas básicas (edad, sexo, estado civil, hijos, núcleo, ingreso, razón de compra) | 7 |
| **J2_CoreDemo+Canal** | J1 + fuente y canal de contacto/anuencia (= las 10 variables que definen el universo) | 10 |
| **J3_+Comercial** | J2 + tipo de unidad, opciones visitadas, participantes en crédito | 13 |
| **J4_LeadFull** | Todas las variables de atributo lead (`LEAD_ONLY` completo, incl. campaña y lugar) | 20 |
| **J5_LeadFull+MissInd** | J4 + indicadores de ausencia (incl. `campania_plataforma` y `sector_referidor`) | 29 |

In [ ]:
# ── §18.2  Definición de combinaciones de variables ───────────────────────
J1 = ['edad','sexo','estado_civil','tiene_hijos','nucleo_familiar','ingreso_familiar','razon_de_compra']
J2 = J1 + ['fuente_pt2','canal_de_anuencia','canal_de_contacto']
J3 = J2 + ['tipo_unidad','opciones_visitadas','participantes_en_credito']
J4 = list(LEAD_ONLY)
J5 = J4 + MISS_IND_18

COMBOS_18 = {
 'J1_CoreDemo'          : [c for c in J1 if c in df_u18.columns],
 'J2_CoreDemo+Canal'    : [c for c in J2 if c in df_u18.columns],
 'J3_+Comercial'        : [c for c in J3 if c in df_u18.columns],
 'J4_LeadFull'          : [c for c in J4 if c in df_u18.columns],
 'J5_LeadFull+MissInd'  : [c for c in J5 if c in df_u18.columns],
}
print("Combinaciones de variables definidas:")
for k,v in COMBOS_18.items(): print(f"  {k:24s}: {len(v)} variables")

## Sección 18.3 — Entrenamiento: modelos × combinaciones de variables

Se entrenan LR, RF, GBM y EXT sobre cada combinación, con `class_weight='balanced'`, más un experimento SMOTE sobre la combinación más completa (J5).

In [ ]:
# ── §18.3  Split + entrenamiento de todas las combinaciones ──────────────
y_u18 = df_u18['target_won'].values
u18_idx = np.arange(len(df_u18))
u18_tr, u18_te = train_test_split(u18_idx, test_size=0.20, random_state=SEED, stratify=y_u18)
Xtr_u18, Xte_u18 = df_u18.iloc[u18_tr], df_u18.iloc[u18_te]
ytr_u18, yte_u18 = y_u18[u18_tr], y_u18[u18_te]
print(f"Split universo Sección 18 — train={len(u18_tr):,} test={len(u18_te):,} | "
      f"base test={yte_u18.mean()*100:.3f}%  ({int(yte_u18.sum())} conversiones)")

def run_u18_grid(feature_sets, use_smote=False):
    res=[]
    for fsn,feats in feature_sets.items():
        feats=[f for f in feats if f in df_u18.columns]
        cat,num=identify_col_types(df_u18,feats)
        for an,af in ALGORITHMS.items():
            ordv=(an!='LR')
            pipe=build_pipeline(af(),cat,num,use_smote=use_smote,ordinal=ordv)
            try:
                pipe.fit(Xtr_u18[feats],ytr_u18)
                p=pipe.predict_proba(Xte_u18[feats])[:,1]
                bm=business_at_k(yte_u18,p)
                res.append({'feature_set':fsn,'algorithm':an,
                    'experiment':'B_SMOTE' if use_smote else 'A_NoSMOTE',
                    'pr_auc':average_precision_score(yte_u18,p),'roc_auc':roc_auc_score(yte_u18,p),
                    'brier':brier_score_loss(yte_u18,p),
                    'rec_10pct':bm[0.10]['rec'],'rec_20pct':bm[0.20]['rec'],'rec_30pct':bm[0.30]['rec'],
                    'lift_10pct':bm[0.10]['lift'],'lift_20pct':bm[0.20]['lift'],
                    'prec_10pct':bm[0.10]['prec'],'n_features':len(feats),
                    'pipeline':pipe,'y_prob':p,'features':feats})
                print(f"  ok {an:4s} | {fsn:22s} | PR={res[-1]['pr_auc']:.4f} "
                      f"ROC={res[-1]['roc_auc']:.3f} rec@20={bm[0.20]['rec']:.3f} "
                      f"lift@10={bm[0.10]['lift']:.2f}x")
            except Exception as e:
                print(f"  ERR {an:4s} | {fsn:22s} | {e}")
    return res

print("\n=== Experimento A (sin SMOTE, class_weight balanced) ===")
u18_results = run_u18_grid(COMBOS_18, use_smote=False)
print("\n=== Experimento B (SMOTE, combinación J5_LeadFull+MissInd) ===")
u18_results += run_u18_grid({'J5_LeadFull+MissInd': COMBOS_18['J5_LeadFull+MissInd']}, use_smote=True)

In [ ]:
# ── §18.4  Tabla comparativa + gráfica ────────────────────────────────────
u18_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ('pipeline','y_prob','features')} for r in u18_results])
u18_df = u18_df.sort_values(['rec_20pct','pr_auc'],ascending=False).reset_index(drop=True)
show=['feature_set','algorithm','experiment','pr_auc','roc_auc','brier',
      'rec_10pct','rec_20pct','rec_30pct','lift_10pct','lift_20pct']
print("MODELOS — UNIVERSO SECCIÓN 18 — ordenados por Recall@20%:")
print(u18_df[show].round(4).to_string(index=False))
u18_df[show].to_csv(RESULTS_DIR/'u18_models_comparison.csv', index=False)

fig,ax=plt.subplots(1,2,figsize=(14,5))
piv_pr=u18_df.pivot_table(index='feature_set',columns='algorithm',values='pr_auc',aggfunc='max')
piv_pr.plot(kind='bar',ax=ax[0],alpha=.88)
ax[0].set_title('PR-AUC por combinación y algoritmo',fontweight='bold')
ax[0].set_ylabel('PR-AUC');ax[0].tick_params(axis='x',rotation=30);ax[0].legend(fontsize=8)
piv_rec=u18_df.pivot_table(index='feature_set',columns='algorithm',values='rec_20pct',aggfunc='max')
piv_rec.plot(kind='bar',ax=ax[1],alpha=.88)
ax[1].set_title('Recall@20% por combinación y algoritmo',fontweight='bold')
ax[1].set_ylabel('Recall@20%');ax[1].tick_params(axis='x',rotation=30);ax[1].legend(fontsize=8)
plt.tight_layout();plt.savefig(FIGURES_DIR/'u18_modelos_comparacion.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 18.5 — Barrido de umbral hasta Recall ≥ 0.70

Para cada combinación se busca el umbral que alcanza **recall ≥ 0.70**, reportando el % de leads a priorizar y la precisión resultante.

In [ ]:
# ── §18.5  Umbral para recall ≥ 0.70 por combinación ──────────────────────
TARGET_RECALL_18 = 0.70
thr70_rows=[]
for r in u18_results:
    yt_r, p_r = yte_u18, r['y_prob']
    prc,rcc,thr = precision_recall_curve(yt_r,p_r)
    idx=np.where(rcc[:-1]>=TARGET_RECALL_18)[0]
    if len(idx):
        i=idx[-1]; t=thr[i]; sel=p_r>=t
        tp=int(((sel)&(yt_r==1)).sum()); fp=int(((sel)&(yt_r==0)).sum())
        thr70_rows.append({'feature_set':r['feature_set'],'algorithm':r['algorithm'],
            'experiment':r['experiment'],'umbral':round(t,4),'recall':round(rcc[i],3),
            'precision':round(prc[i],4),'leads':int(sel.sum()),
            'pct_leads':round(sel.mean()*100,1),'TP':tp,'FP':fp,'alcanzable':True})
    else:
        thr70_rows.append({'feature_set':r['feature_set'],'algorithm':r['algorithm'],
            'experiment':r['experiment'],'umbral':np.nan,'recall':np.nan,
            'precision':np.nan,'leads':np.nan,'pct_leads':np.nan,'TP':np.nan,'FP':np.nan,
            'alcanzable':False})

thr70_df = pd.DataFrame(thr70_rows).sort_values('pct_leads')
print(f"UMBRAL PARA RECALL >= {TARGET_RECALL_18:.0%} - por modelo y combinación de variables")
print("="*90)
print(thr70_df.to_string(index=False))
thr70_df.to_csv(RESULTS_DIR/'u18_recall70_umbrales.csv', index=False)

n_ok = int(thr70_df['alcanzable'].sum())
print(f"\n{n_ok}/{len(thr70_df)} combinaciones alcanzan recall >= {TARGET_RECALL_18:.0%} "
      f"dentro del rango de umbrales evaluado.")

In [ ]:
# ── §18.5b  Gráfica: % de leads necesarios para recall ≥ 0.70 ────────────
plot_df18 = thr70_df.dropna(subset=['pct_leads']).copy()
plot_df18['label'] = plot_df18['feature_set'] + ' | ' + plot_df18['algorithm'] + ' | ' + plot_df18['experiment']
plot_df18 = plot_df18.sort_values('pct_leads')
fig,ax=plt.subplots(1,2,figsize=(14,max(5,0.35*len(plot_df18))))
ax[0].barh(plot_df18['label'],plot_df18['pct_leads'],color='#5C6BC0',alpha=.88)
ax[0].set_title(f'% de leads necesarios para recall ≥ {TARGET_RECALL_18:.0%}',fontweight='bold')
ax[0].set_xlabel('% leads priorizados');ax[0].invert_yaxis()
ax[1].barh(plot_df18['label'],plot_df18['precision'],color='#FF7043',alpha=.88)
ax[1].set_title(f'Precisión al alcanzar recall ≥ {TARGET_RECALL_18:.0%}',fontweight='bold')
ax[1].set_xlabel('Precisión');ax[1].invert_yaxis()
plt.tight_layout();plt.savefig(FIGURES_DIR/'u18_recall70_umbrales.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 18.6 — Selección del mejor modelo (Recall ≥ 0.70)

Entre las combinaciones que alcanzan recall ≥ 0.70, se elige la que requiere **menor % de leads**; en caso de empate, la de mayor precisión.

In [ ]:
# ── §18.6  Mejor modelo con recall ≥ 0.70 + curvas ────────────────────────
reach70 = thr70_df[thr70_df['alcanzable']].sort_values(['pct_leads','precision'],ascending=[True,False])
best70 = reach70.iloc[0]
print(f"MEJOR MODELO PARA RECALL >= {TARGET_RECALL_18:.0%}:")
print(f"  → {best70['algorithm']} | {best70['feature_set']} | {best70['experiment']}")
print(f"  Umbral={best70['umbral']}  Recall={best70['recall']}  Precisión={best70['precision']}")
print(f"  Leads a contactar={int(best70['leads']):,} ({best70['pct_leads']}% del subconjunto de prueba)")
print(f"  TP={int(best70['TP'])}  FP={int(best70['FP'])}")

best70_res = [r for r in u18_results if r['feature_set']==best70['feature_set']
              and r['algorithm']==best70['algorithm'] and r['experiment']==best70['experiment']][0]
p70 = best70_res['y_prob']; yt70 = yte_u18

fig,ax=plt.subplots(1,2,figsize=(13,5))
order=np.argsort(p70)[::-1]; cg=np.cumsum(yt70[order])/yt70.sum(); pct=np.arange(1,len(yt70)+1)/len(yt70)
ax[0].plot(pct,cg,lw=2,color='#1565C0',label='Modelo')
ax[0].plot([0,1],[0,1],'k--',lw=.8,label='Aleatorio')
ax[0].axhline(TARGET_RECALL_18,color='red',ls=':',lw=1,label=f'Recall {TARGET_RECALL_18:.0%}')
ax[0].axvline(best70['pct_leads']/100,color='red',ls=':',lw=1)
ax[0].set_title('Curva de ganancia — mejor modelo',fontweight='bold')
ax[0].set_xlabel('% leads contactados');ax[0].set_ylabel('% conversiones capturadas');ax[0].legend(fontsize=8)

prc,rcc,_=precision_recall_curve(yt70,p70)
ax[1].plot(rcc,prc,lw=2,color='#2E7D32',label=f'AP={average_precision_score(yt70,p70):.3f}')
ax[1].axvline(TARGET_RECALL_18,color='red',ls=':',lw=1,label=f'Recall {TARGET_RECALL_18:.0%}')
ax[1].set_title('Curva Precision-Recall — mejor modelo',fontweight='bold')
ax[1].set_xlabel('Recall');ax[1].set_ylabel('Precision');ax[1].legend(fontsize=8)
plt.suptitle(f"Modelo recomendado (recall>={TARGET_RECALL_18:.0%}): {best70['algorithm']} | {best70['feature_set']} | {best70['experiment']}",
             fontsize=12,fontweight='bold')
plt.tight_layout();plt.savefig(FIGURES_DIR/'u18_mejor_modelo_curvas.png',dpi=140,bbox_inches='tight');plt.show()

## Conclusión — Sección 18

**Qué se hizo.** Se repitió el experimento de la Sección 15 (LR/RF/GBM/EXT × 5 combinaciones de variables, con umbral orientado a recall) sobre el universo más amplio de la Sección 17 (13,038 leads, 563 conversiones — 1.5× más grande que el de la Sección 15), buscando **recall ≥ 0.70** en vez de 0.80.

**Resultado.** Las 24 combinaciones alcanzan recall ≥ 0.70. El mejor modelo es **LR sobre J5_LeadFull+MissInd (sin SMOTE)**: recall=0.708, precisión=8.29% (~1.9× la tasa base de 4.33%), necesitando contactar solo **37.0% del universo de prueba** (965 de 2,608 leads) para capturar 80 de las 113 conversiones. Esto es una mejora sustancial frente a la Sección 15, donde el mejor modelo para recall ≥ 0.80 necesitaba contactar el 80.1% de los leads.

**Mismo patrón que en la Sección 15:** las combinaciones J1 a J4 (que solo agregan variables crudas de canal/comercial sin indicadores de missingness) producen predicciones prácticamente idénticas entre sí — la variación real proviene de **J5, que agrega los indicadores de ausencia** (`campania_plataforma__miss`, `sector_referidor__miss`, etc.). Esto confirma, otra vez, que la señal más fuerte no es la variable en sí sino **si el lead la informó o no**.

**Comparación con la Sección 15.** El universo más grande (13K vs 8.6K leads) y el objetivo de recall menos exigente (0.70 vs 0.80) sí se traducen en una mejora clara: menos leads a contactar (37.0% vs 80.1%) y mayor precisión relativa. Este universo de 10 variables (fuente + canal + demográficas) es preferible como base de un **modelo especializado de perfil completo** frente al de la Sección 15 — ofrece más volumen y mejor eficiencia operativa, sin sacrificar la ausencia de imputación.

---

## Sección 19 — Importancia de variables y poda del modelo (Sección 18)

> **Objetivo.** Medir la importancia real de cada variable en el modelo recomendado de la Sección 18 (`LR | J5_LeadFull+MissInd | A_NoSMOTE`, recall≥0.70 con 37.0% de leads), **eliminar las que no aportan señal** y verificar que el modelo podado mantenga el desempeño con menos variables — más simple de mantener y de explicar.

### Sección 19.1 — Importancia por permutación del modelo recomendado

In [ ]:
# ── §19.1  Permutation importance del modelo recomendado (Sección 18) ────
from sklearn.inspection import permutation_importance

feats_best18 = best70_res['features']
pi18 = permutation_importance(best70_res['pipeline'], Xte_u18[feats_best18], yte_u18,
                              scoring='average_precision', n_repeats=15,
                              random_state=SEED, n_jobs=-1)
imp18 = pd.DataFrame({
    'variable': feats_best18,
    'importancia_media': pi18.importances_mean,
    'importancia_std': pi18.importances_std,
}).sort_values('importancia_media', ascending=False).reset_index(drop=True)
imp18['relevante'] = (imp18['importancia_media'] - imp18['importancia_std']) > 0

print(f"PERMUTATION IMPORTANCE — {best70['algorithm']} | {best70['feature_set']} "
      f"(caída de PR-AUC al permutar cada variable, 15 repeticiones)")
print('='*80)
print(imp18.to_string(index=False))
imp18.to_csv(RESULTS_DIR/'u18_permutation_importance.csv', index=False)

In [ ]:
# ── §19.1b  Gráfica: importancia con barras de error ──────────────────────
fig,ax=plt.subplots(figsize=(9, max(6,0.32*len(imp18))))
plot_imp = imp18.iloc[::-1]
colors_imp = ['#2E7D32' if r else '#B0BEC5' for r in plot_imp['relevante']]
ax.barh(plot_imp['variable'], plot_imp['importancia_media'],
        xerr=plot_imp['importancia_std'], color=colors_imp, alpha=.88, capsize=3)
ax.axvline(0, color='k', lw=.8)
ax.set_title(f"Permutation importance — {best70['algorithm']} | {best70['feature_set']}",fontweight='bold')
ax.set_xlabel('Caída de PR-AUC al permutar (± 1 std)')
handles=[plt.Rectangle((0,0),1,1,color=c) for c in ['#2E7D32','#B0BEC5']]
ax.legend(handles,['Relevante (media - std > 0)','No relevante'],fontsize=8,loc='lower right')
plt.tight_layout();plt.savefig(FIGURES_DIR/'u18_importancia.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 19.2 — Variables descartadas por no ser relevantes

Se considera que una variable **no es relevante** cuando su importancia media menos una desviación estándar no supera cero — es decir, su efecto no se distingue de ruido de permutación.

In [ ]:
# ── §19.2  Identificación de variables a eliminar ─────────────────────────
vars_relevantes = imp18.loc[imp18['relevante'], 'variable'].tolist()
vars_descartadas = imp18.loc[~imp18['relevante'], 'variable'].tolist()

print(f"Variables originales: {len(feats_best18)}")
print(f"Variables relevantes (se conservan): {len(vars_relevantes)}")
print(f"Variables descartadas (se eliminan): {len(vars_descartadas)}")
print()
for v in vars_descartadas:
    row = imp18.loc[imp18['variable']==v].iloc[0]
    print(f"  - {v:32s} media={row['importancia_media']:.5f}  std={row['importancia_std']:.5f}")

## Sección 19.3 — Reentrenamiento con variables relevantes (modelo podado)

Se reentrena `LR` con el mismo split de la Sección 18, usando solo las variables relevantes, y se compara contra el modelo original con todas las variables.

In [ ]:
# ── §19.3  Reentrenamiento del modelo podado ──────────────────────────────
use_smote_pruned = (best70['experiment']=='B_SMOTE')
cat_p, num_p = identify_col_types(df_u18, vars_relevantes)
pipe_pruned = build_pipeline(ALGORITHMS[best70['algorithm']](), cat_p, num_p,
                             use_smote=use_smote_pruned, ordinal=(best70['algorithm']!='LR'))
pipe_pruned.fit(Xtr_u18[vars_relevantes], ytr_u18)
p_pruned = pipe_pruned.predict_proba(Xte_u18[vars_relevantes])[:,1]

bm_pruned = business_at_k(yte_u18, p_pruned)
prc_p, rcc_p, thr_p = precision_recall_curve(yte_u18, p_pruned)
idx_p = np.where(rcc_p[:-1]>=TARGET_RECALL_18)[0]
if len(idx_p):
    i_p = idx_p[-1]; t_p = thr_p[i_p]; sel_p = p_pruned>=t_p
    leads_p = int(sel_p.sum()); pct_p = round(sel_p.mean()*100,1)
    prec_p = round(prc_p[i_p],4); rec_p = round(rcc_p[i_p],3)
else:
    leads_p = pct_p = prec_p = rec_p = np.nan

comparison_19 = pd.DataFrame([
    {'modelo':'Original (J5, todas las variables)','n_variables':len(feats_best18),
     'pr_auc':round(average_precision_score(yte_u18,best70_res['y_prob']),4),
     'roc_auc':round(roc_auc_score(yte_u18,best70_res['y_prob']),4),
     'recall_alcanzado':best70['recall'],'precision_en_umbral':best70['precision'],
     'pct_leads_recall70':best70['pct_leads']},
    {'modelo':'Podado (solo variables relevantes)','n_variables':len(vars_relevantes),
     'pr_auc':round(average_precision_score(yte_u18,p_pruned),4),
     'roc_auc':round(roc_auc_score(yte_u18,p_pruned),4),
     'recall_alcanzado':rec_p,'precision_en_umbral':prec_p,
     'pct_leads_recall70':pct_p},
])
print("COMPARACIÓN: MODELO ORIGINAL VS. MODELO PODADO")
print("="*80)
print(comparison_19.to_string(index=False))
comparison_19.to_csv(RESULTS_DIR/'u18_comparacion_podado.csv', index=False)

In [ ]:
# ── §19.3b  Gráfica: original vs podado ───────────────────────────────────
fig,ax=plt.subplots(1,3,figsize=(15,4.5))
labels_19=['Original','Podado']
vals_n=[comparison_19['n_variables'][0],comparison_19['n_variables'][1]]
vals_pr=[comparison_19['pr_auc'][0],comparison_19['pr_auc'][1]]
vals_pct=[comparison_19['pct_leads_recall70'][0],comparison_19['pct_leads_recall70'][1]]
cols19=['#78909C','#2E7D32']
ax[0].bar(labels_19,vals_n,color=cols19,alpha=.9)
for i,v in enumerate(vals_n): ax[0].text(i,v,f'{v}',ha='center',va='bottom',fontsize=9)
ax[0].set_title('N° de variables',fontweight='bold')
ax[1].bar(labels_19,vals_pr,color=cols19,alpha=.9)
for i,v in enumerate(vals_pr): ax[1].text(i,v,f'{v:.4f}',ha='center',va='bottom',fontsize=9)
ax[1].set_title('PR-AUC',fontweight='bold')
ax[2].bar(labels_19,vals_pct,color=cols19,alpha=.9)
for i,v in enumerate(vals_pct): ax[2].text(i,v,f'{v:.1f}%',ha='center',va='bottom',fontsize=9)
ax[2].set_title(f'% leads para recall≥{TARGET_RECALL_18:.0%}',fontweight='bold')
plt.tight_layout();plt.savefig(FIGURES_DIR/'u18_original_vs_podado.png',dpi=140,bbox_inches='tight');plt.show()

## Sección 19.4 — Coeficientes del modelo podado (interpretabilidad)

Como `LR` es un modelo lineal, sus coeficientes indican **dirección** (empuja hacia conversión o en contra) además de magnitud — algo que la permutation importance no distingue.

In [ ]:
# ── §19.4  Coeficientes de LR (modelo podado) ─────────────────────────────
prep_names = pipe_pruned.named_steps['prep'].get_feature_names_out()
coefs = pipe_pruned.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'feature':prep_names,'coeficiente':coefs}).sort_values('coeficiente')

print("TOP 15 COEFICIENTES NEGATIVOS (empujan hacia NO conversión):")
print(coef_df.head(15).to_string(index=False))
print("\nTOP 15 COEFICIENTES POSITIVOS (empujan hacia conversión):")
print(coef_df.tail(15).sort_values('coeficiente',ascending=False).to_string(index=False))
coef_df.to_csv(RESULTS_DIR/'u18_coeficientes_lr.csv', index=False)

top_pos = coef_df.tail(12)
top_neg = coef_df.head(12)
top_both = pd.concat([top_neg, top_pos]).sort_values('coeficiente')
fig,ax=plt.subplots(figsize=(9,8))
colors_c=['#C62828' if v<0 else '#2E7D32' for v in top_both['coeficiente']]
ax.barh(top_both['feature'], top_both['coeficiente'], color=colors_c, alpha=.88)
ax.axvline(0,color='k',lw=.8)
ax.set_title('Coeficientes LR — top 12 a favor / top 12 en contra',fontweight='bold')
ax.set_xlabel('Coeficiente (escala estandarizada)')
plt.tight_layout();plt.savefig(FIGURES_DIR/'u18_coeficientes.png',dpi=140,bbox_inches='tight');plt.show()

## Conclusión — Sección 19

**Qué se hizo.** Se midió la importancia por permutación de cada variable del modelo recomendado de la Sección 18 (31 variables), se eliminaron las que no superaron el umbral de relevancia (media − 1 std ≤ 0), y se reentrenó el modelo solo con las variables relevantes, comparando desempeño y agregando la lectura de coeficientes (dirección del efecto).

**Resultado — hallazgo contundente.** De las **31 variables** del modelo original, solo **3 tienen importancia distinta de cero**: `sector_referidor__miss`, `campania_plataforma__miss` y `edad`. Las 28 restantes —incluyendo `sexo`, `estado_civil`, `razon_de_compra`, `ingreso_familiar`, `nucleo_familiar`, `fuente_pt2`, `canal_de_contacto`, `canal_de_anuencia` y todos los demás indicadores de missingness— tienen importancia **exactamente 0.0000**: permutarlas no cambia ni una sola predicción del modelo. La regularización L2 de `LR` las volvió irrelevantes en presencia de las 3 variables dominantes.

**El modelo podado (solo 3 variables) prácticamente iguala al original (31 variables):**

| Modelo | N° variables | PR-AUC | ROC-AUC | % leads para recall≥0.70 |
|---|---|---|---|---|
| Original | 31 | 0.1126 | 0.7113 | 37.0% |
| Podado | **3** | 0.1094 | 0.7048 | 37.6% |

La diferencia es marginal (−2.8% en PR-AUC, +0.6 pp en leads necesarios) frente a una reducción de **28 variables menos** — el modelo podado es preferible en producción: más simple, más barato de calcular, y sin depender de campos con alto missingness que compliquen el pipeline de scoring.

**Lectura de coeficientes (§19.4) — dirección del efecto:**
- `campania_plataforma__miss` (+0.398): cuando **falta** el dato de campaña, la probabilidad de conversión **sube** — consistente con que los leads que convierten llegan más por canales no rastreados por campaña (referidos, orgánico) que por pauta digital con atribución completa.
- `sector_referidor__miss` (−0.335): cuando **falta** el sector del referidor, la probabilidad de conversión **baja** — es decir, tener un referidor identificado (aunque sea el 10% de los casos) es señal positiva, coherente con el lift de `canal_de_anuencia='Referencia'` visto en la Sección 17.3.
- `edad` (+0.156): a mayor edad, mayor probabilidad de conversión — el único efecto de una variable de valor (no de missingness), consistente con todos los análisis de edad de secciones anteriores (5, 12, 16).

**Implicación práctica.** El modelo de conversión de este universo se explica casi por completo con 3 señales: *si el lead tiene un referidor identificado*, *si llegó por un canal de campaña rastreado*, y *su edad*. Esto es más una historia de **completitud/procedencia del dato** que de perfil demográfico — refuerza la conclusión de la Sección 16: la ausencia de un dato es más informativa que su valor.

---

## Sección 20 — Corrección: eliminar variables con riesgo de fuga de información

> **Motivación.** El análisis de importancia de la Sección 19 mostró que el modelo de la Sección 18 dependía casi exclusivamente de `sector_referidor__miss` y `campania_plataforma__miss`. Revisando qué es `sector_referidor`: es la **torre donde vive la persona que refirió al lead** — un dato que solo existe si hubo un referidor, y que en la práctica del CRM puede quedar registrado en el mismo momento (o después) del cierre del trato, no necesariamente antes. Ante la duda razonable de fuga de información, se retiran del modelo:
> - `sector_referidor` y `sector_referidor__miss`
> - `campania_plataforma` y `campania_plataforma__miss`
> - `tipo_unidad__miss`
> - `opciones_visitadas__miss`
>
> y se reentrena/evalúa de nuevo sobre el mismo universo y split de la Sección 18 (13,038 leads, recall objetivo ≥ 0.70).

### Sección 20.1 — Feature set sin variables sospechosas de fuga

In [ ]:
# ── §20.1  Feature set J6_Clean: J5 sin variables sospechosas de fuga ────
VARS_LEAKAGE_RIESGO = ['sector_referidor', 'sector_referidor__miss',
                       'campania_plataforma', 'campania_plataforma__miss',
                       'tipo_unidad__miss', 'opciones_visitadas__miss']
J6_Clean = [c for c in COMBOS_18['J5_LeadFull+MissInd'] if c not in VARS_LEAKAGE_RIESGO]
COMBOS_18['J6_Clean'] = J6_Clean

print(f"J5_LeadFull+MissInd original: {len(COMBOS_18['J5_LeadFull+MissInd'])} variables")
print(f"Variables retiradas por riesgo de fuga: {VARS_LEAKAGE_RIESGO}")
print(f"J6_Clean (sin esas {len(VARS_LEAKAGE_RIESGO)} variables): {len(J6_Clean)} variables")
print()
print(J6_Clean)

## Sección 20.2 — Reentrenamiento sin las variables removidas

Se reutiliza el mismo split de la Sección 18 (`Xtr_u18` / `Xte_u18`) y el mismo `run_u18_grid`, entrenando los 4 algoritmos sobre `J6_Clean` (+ SMOTE).

In [ ]:
# ── §20.2  Entrenamiento de J6_Clean (misma metodología de la Sección 18) ─
print("=== J6_Clean — sin SMOTE ===")
u20_results = run_u18_grid({'J6_Clean': J6_Clean}, use_smote=False)
print("\n=== J6_Clean — con SMOTE ===")
u20_results += run_u18_grid({'J6_Clean': J6_Clean}, use_smote=True)

## Sección 20.3 — Umbral para recall ≥ 0.70: con fuga vs. sin fuga

In [ ]:
# ── §20.3  Umbral recall ≥ 0.70 para J6_Clean + comparación con J5 ───────
thr_j6_rows=[]
for r in u20_results:
    yt_r, p_r = yte_u18, r['y_prob']
    prc,rcc,thr = precision_recall_curve(yt_r,p_r)
    idx=np.where(rcc[:-1]>=TARGET_RECALL_18)[0]
    if len(idx):
        i=idx[-1]; t=thr[i]; sel=p_r>=t
        tp=int(((sel)&(yt_r==1)).sum()); fp=int(((sel)&(yt_r==0)).sum())
        thr_j6_rows.append({'feature_set':r['feature_set'],'algorithm':r['algorithm'],
            'experiment':r['experiment'],'umbral':round(t,4),'recall':round(rcc[i],3),
            'precision':round(prc[i],4),'leads':int(sel.sum()),
            'pct_leads':round(sel.mean()*100,1),'TP':tp,'FP':fp,'alcanzable':True})
    else:
        thr_j6_rows.append({'feature_set':r['feature_set'],'algorithm':r['algorithm'],
            'experiment':r['experiment'],'umbral':np.nan,'recall':np.nan,
            'precision':np.nan,'leads':np.nan,'pct_leads':np.nan,'TP':np.nan,'FP':np.nan,
            'alcanzable':False})

thr_j6_df = pd.DataFrame(thr_j6_rows).sort_values('pct_leads')
print(f"UMBRAL PARA RECALL >= {TARGET_RECALL_18:.0%} - J6_Clean (sin variables de riesgo)")
print("="*90)
print(thr_j6_df.to_string(index=False))
thr_j6_df.to_csv(RESULTS_DIR/'u20_j6clean_recall70_umbrales.csv', index=False)

combined_20 = pd.concat([thr70_df.assign(version='J5 (con sector_referidor/campania)'),
                         thr_j6_df.assign(version='J6_Clean (sin fuga)')], ignore_index=True)
combined_20 = combined_20[combined_20['alcanzable']].sort_values('pct_leads')
print("\nTOP 10 COMBINACIONES (J5 vs J6_Clean) POR % DE LEADS NECESARIO:")
print(combined_20.head(10)[['version','feature_set','algorithm','experiment',
                             'recall','precision','pct_leads']].to_string(index=False))

In [ ]:
# ── §20.3b  Gráfica: J5 (con fuga) vs J6_Clean (sin fuga) ────────────────
best_j5 = thr70_df[thr70_df['alcanzable']].sort_values('pct_leads').iloc[0]
best_j6 = thr_j6_df[thr_j6_df['alcanzable']].sort_values('pct_leads').iloc[0]

fig,ax=plt.subplots(1,2,figsize=(12,4.5))
labels_20=['J5 (con\nsector_referidor/campania)','J6_Clean\n(sin fuga)']
pct_vals=[best_j5['pct_leads'], best_j6['pct_leads']]
prec_vals=[best_j5['precision'], best_j6['precision']]
cols20=['#C62828','#2E7D32']
ax[0].bar(labels_20,pct_vals,color=cols20,alpha=.9)
for i,v in enumerate(pct_vals): ax[0].text(i,v,f'{v:.1f}%',ha='center',va='bottom',fontsize=9)
ax[0].set_title(f'% de leads para recall≥{TARGET_RECALL_18:.0%}',fontweight='bold')
ax[1].bar(labels_20,prec_vals,color=cols20,alpha=.9)
for i,v in enumerate(prec_vals): ax[1].text(i,v,f'{v:.4f}',ha='center',va='bottom',fontsize=9)
ax[1].set_title('Precisión en el umbral',fontweight='bold')
plt.tight_layout();plt.savefig(FIGURES_DIR/'u20_j5_vs_j6clean.png',dpi=140,bbox_inches='tight');plt.show()

print(f"MEJOR CON FUGA (J5):   {best_j5['algorithm']} | {best_j5['experiment']} | "
      f"{best_j5['pct_leads']}% leads | precisión={best_j5['precision']}")
print(f"MEJOR SIN FUGA (J6):   {best_j6['algorithm']} | {best_j6['experiment']} | "
      f"{best_j6['pct_leads']}% leads | precisión={best_j6['precision']}")

## Sección 20.4 — Importancia de variables del nuevo modelo (sin fuga)

Se repite el análisis de permutation importance (Sección 19) sobre el mejor modelo de `J6_Clean`, para ver qué variables toman el lugar de las que se retiraron.

In [ ]:
# ── §20.4  Permutation importance del mejor modelo J6_Clean ──────────────
best_j6_res = [r for r in u20_results if r['algorithm']==best_j6['algorithm']
               and r['experiment']==best_j6['experiment']][0]
pi20 = permutation_importance(best_j6_res['pipeline'], Xte_u18[J6_Clean], yte_u18,
                              scoring='average_precision', n_repeats=15,
                              random_state=SEED, n_jobs=-1)
imp20 = pd.DataFrame({
    'variable': J6_Clean,
    'importancia_media': pi20.importances_mean,
    'importancia_std': pi20.importances_std,
}).sort_values('importancia_media', ascending=False).reset_index(drop=True)
imp20['relevante'] = (imp20['importancia_media'] - imp20['importancia_std']) > 0

print(f"PERMUTATION IMPORTANCE — {best_j6['algorithm']} | J6_Clean (sin fuga)")
print('='*80)
print(imp20.to_string(index=False))
imp20.to_csv(RESULTS_DIR/'u20_j6clean_importance.csv', index=False)

fig,ax=plt.subplots(figsize=(9, max(6,0.32*len(imp20))))
plot_imp20 = imp20.iloc[::-1]
colors_imp20 = ['#2E7D32' if r else '#B0BEC5' for r in plot_imp20['relevante']]
ax.barh(plot_imp20['variable'], plot_imp20['importancia_media'],
        xerr=plot_imp20['importancia_std'], color=colors_imp20, alpha=.88, capsize=3)
ax.axvline(0, color='k', lw=.8)
ax.set_title(f"Permutation importance — {best_j6['algorithm']} | J6_Clean (sin fuga)",fontweight='bold')
ax.set_xlabel('Caída de PR-AUC al permutar (± 1 std)')
plt.tight_layout();plt.savefig(FIGURES_DIR/'u20_j6clean_importancia.png',dpi=140,bbox_inches='tight');plt.show()

## Conclusión — Sección 20

**Qué se hizo.** Se identificó riesgo de fuga de información en `sector_referidor` (y su indicador), `campania_plataforma` (y su indicador), `tipo_unidad__miss` y `opciones_visitadas__miss` — variables cuyo registro en el CRM puede depender del momento del cierre del trato, no solo del momento de captación del lead. Se retiraron del feature set (31 → 25 variables) y se reentrenó/evaluó de nuevo sobre el mismo universo y split de la Sección 18.

**Resultado — la fuga sí estaba inflando el desempeño.** Para el mismo objetivo (recall ≥ 0.70), el mejor modelo pasó de necesitar **37.0% de los leads** (con las variables sospechosas, precisión=8.29%) a necesitar **64.2%** (sin ellas, precisión=5.07%). Es decir, cerca de dos tercios del universo de prueba, con una precisión que apenas duplica la tasa base (4.33%) en vez de casi cuadruplicarla. Esta caída confirma que **gran parte del desempeño reportado en las Secciones 18-19 dependía de variables con fuga** — el número honesto y defendible de este modelo es el de `J6_Clean`, no el de `J5`.

**Importancia tras la limpieza (§20.4) — hallazgo aún más contundente.** Con las 6 variables de riesgo fuera, **solo `edad` conserva importancia distinta de cero** (0.0353). Las 24 variables restantes —sexo, estado civil, razón de compra, ingreso, canal, fuente, tipo de unidad, opciones visitadas, todos los indicadores de missingness que quedaron— vuelven a tener importancia **exactamente 0.0000**. El modelo `LR` sobre este universo, una vez retirada la fuga, es en la práctica **un modelo de una sola variable: la edad**.

**Conclusión metodológica para la tesis.** El modelo de conversión de leads con perfil demográfico completo (Secciones 17-20) no sostiene, con las variables legítimamente disponibles antes de la decisión del lead, un desempeño mucho mejor que ordenar por edad. Esto es un resultado válido y reportable: la señal predictiva real en los datos disponibles es limitada, y es preferible declarar esa limitación con evidencia (como se hizo aquí) que reportar una métrica inflada por fuga de información.

# Parte III — Modelo Final Seleccionado

---

## Sección 21 — Búsqueda exhaustiva de configuraciones y modelo final

> **Alcance.** Dentro del universo de la Sección 17 (fuente + canal + demográficas completas, 13,038 leads) se realizó una **búsqueda exhaustiva** de configuraciones antes de decidir variables y universo definitivos. La búsqueda se ejecutó con scripts externos (guardados en `qualified_run/exploracion/scripts/`) y sus resultados como CSV en `qualified_run/exploracion/`. Comprendió:
>
> - **Fase 1a — Grid amplio**: 102 configuraciones = 6 pools de variables × 9 algoritmos (LR L1/L2, RF, ExtraTrees, HistGB, GradientBoosting, AdaBoost, Naive Bayes, LDA) × 5 estrategias de balanceo (ninguno, SMOTE 10%/25%, undersampling, oversampling), evaluadas con **CV estratificada 5-fold out-of-fold** (más robusta que un hold-out simple).
> - **Fase 1b — Tuning**: RandomizedSearchCV (40 iteraciones, PR-AUC) sobre LR, LDA, HistGB, RF, ExtraTrees y GB en los 3 mejores pools (18 búsquedas).
> - **Fase 1c — Selección de variables**: señal univariada por variable, path de regularización L1, *forward selection* greedy y *backward elimination*.
> - **Fase 1d — Ablación por familias**: caída de PR-AUC al retirar cada familia completa (edad, ingreso, demografía, razón, canal, campaña, lugar, comercial).
> - **Fase 1e — Finalistas**: 6 candidatos re-evaluados con una **segunda semilla de CV** (confirmación), umbrales operativos, permutation importance y SHAP.
> - **Fase 2 — Reconstrucción del universo**: con las variables ganadoras se redefinieron universos más grandes y se reentrenó, iterando hasta converger.
>
> Ingeniería de variables incluida: punto medio de ingreso, ingreso per cápita, agrupación comercial de canal (`canal_familia`), flags `es_referido`/`es_inversion`, interacciones `edad×inversión`, `edad×ingreso`, `edad×referido`, `ingreso×inversión`, `mayor_45`, `ingreso_alto`, `etapa_familiar`. Se mantuvieron **excluidas** las variables de fuga de la Sección 20.

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo en esta sección (explicación simple)</b><br><br>
Antes de elegir el modelo final se organizó una especie de <b>torneo en fases</b>: primero se probaron 102 combinaciones de variables × algoritmo × estrategia de balanceo (fase 1a); a los mejores se les afinaron los hiperparámetros (1b); después se identificó qué variables realmente aportan (1c, selección) y cuánto pesa cada familia de variables (1d, ablación); y con las variables ganadoras se redefinió <b>a qué leads conviene aplicar el modelo</b> — el "universo" (fase 2). El ganador de todo el proceso se reconstruye paso a paso en la Sección 21.4.<br><br>
<b>Importante:</b> las celdas de esta sección <b>no re-ejecutan la búsqueda</b> (tardaría horas); leen los resultados ya calculados, guardados como CSV en <code>qualified_run/exploracion/</code>. Los scripts que los generaron están en <code>qualified_run/exploracion/scripts/</code>.
</div>


### Sección 21.0 — Definición de los conjuntos de variables (P0–P5)

La búsqueda de la fase 1a evaluó 6 *pools* de variables **anidados e incrementales**, diseñados para aislar el aporte de cada bloque de información. Todos excluyen las variables con fuga de la Sección 20 (`sector_referidor`, `campania_plataforma` y sus indicadores, `tipo_unidad__miss`, `opciones_visitadas__miss`) y todas las columnas `*_label` post-venta.

| Pool | Descripción | Qué pregunta responde | n |
|---|---|---|---|
| **P0_SoloEdad** | Únicamente `edad` | Línea base: ¿qué tan lejos llega la conclusión "solo la edad importa" de la Sección 20? | 1 |
| **P1_Demo** | Demográficas crudas del lead (edad, sexo, estado civil, hijos, núcleo, ingreso, razón de compra) | ¿Alcanza el perfil demográfico por sí solo? | 7 |
| **P2_Clean** | Pool limpio completo: P1 + canal (contacto/anuencia/familia), comercial (tipo unidad, opciones, participantes), lugar (trabajo/vivienda) y campaña (tipo/audiencia) | ¿Cuánto rinden **todos los valores legítimos** sin ingeniería ni proyecto? | 18 |
| **P3_Clean+Proj** | P2 + identidad de proyecto (`fuente_pt2`, `proyecto`) | ¿Cuánto agrega la identidad del proyecto? (cota superior **no generalizable** a proyectos nuevos) | 20 |
| **P4_Eng** | P2 + 10 variables de ingeniería: ratios, flags e interacciones comerciales | ¿Cuánto agrega la ingeniería de variables sin proyecto? | 28 |
| **P5_Eng+Proj** | P4 + identidad de proyecto | Techo de desempeño con todo lo disponible | 30 |

**Transformaciones directas** (incluidas desde P2): `edad_n` (cast numérico), `ingreso_mid` (punto medio del rango de ingreso, en miles de Q), `nucleo_n` (parse numérico del núcleo familiar), `tiene_hijos_bin` (binarización) y `canal_familia` (agrupación comercial del canal de anuencia en: pauta_digital, portal, directo, referido, presencial, exterior, otro).

**Ingeniería adicional** (P4/P5): `ingreso_per_capita` (ingreso/núcleo), flags `es_inversion`, `es_referido`, `mayor_45`, `ingreso_alto` (≥Q30k), `etapa_familiar` (estado civil × hijos) y las interacciones `edad×inversión`, `ingreso×inversión`, `edad×ingreso`, `edad×referido`.

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b><br><br>
Define en código los 6 pools como listas de variables, construye una tabla de pertenencia (variable × pool, con ✓ donde la variable participa), la muestra con estilo, la guarda como CSV y la exporta también como imagen PNG con un color de fondo por familia de variables.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer la tabla</b>
<ul>
<li><b>Filas:</b> variables, agrupadas por familia (demográfica, canal, comercial, lugar, campaña, ingeniería, proyecto). En la versión gráfica, cada familia tiene un color de fondo distinto.</li>
<li><b>Columnas:</b> los pools P0–P5. Una ✓ significa que la variable participa en ese pool.</li>
<li>Los pools son <b>incrementales</b>: cada uno agrega un bloque de información al anterior. Comparar su desempeño (§21.1) permite saber cuánto aporta cada bloque.</li>
</ul>
</div>


In [ ]:
# ── §21.0  Matriz de pertenencia variable × pool ──────────────────────────
P1_DEMO = ['edad_n','sexo','estado_civil','tiene_hijos_bin','nucleo_n',
           'ingreso_mid','razon_de_compra']
P2_CLEAN = P1_DEMO + ['ingreso_familiar','canal_de_anuencia','canal_de_contacto',
           'canal_familia','tipo_unidad','opciones_visitadas',
           'participantes_en_credito','lugar_trabajo','lugar_vivienda',
           'campania_tipo','campania_audiencia']
ENG_EXTRA = ['ingreso_per_capita','es_inversion','es_referido','edad_x_inversion',
             'ingreso_x_inversion','edad_x_ingreso','edad_x_referido',
             'etapa_familiar','mayor_45','ingreso_alto']
PROJ_VARS_21 = ['fuente_pt2','proyecto']

POOLS_21 = {
 'P0_SoloEdad'  : ['edad_n'],
 'P1_Demo'      : P1_DEMO,
 'P2_Clean'     : P2_CLEAN,
 'P3_Clean+Proj': P2_CLEAN + PROJ_VARS_21,
 'P4_Eng'       : P2_CLEAN + ENG_EXTRA,
 'P5_Eng+Proj'  : P2_CLEAN + ENG_EXTRA + PROJ_VARS_21,
}

def _grupo_21(v):
    if v in ['edad_n','sexo','estado_civil','tiene_hijos_bin','nucleo_n',
             'ingreso_mid','ingreso_familiar','razon_de_compra']:
        return '1. Demográfica / lead'
    if v in ['canal_de_anuencia','canal_de_contacto','canal_familia']:
        return '2. Canal'
    if v in ['tipo_unidad','opciones_visitadas','participantes_en_credito']:
        return '3. Comercial'
    if v in ['lugar_trabajo','lugar_vivienda']:
        return '4. Lugar'
    if v in ['campania_tipo','campania_audiencia']:
        return '5. Campaña'
    if v in ENG_EXTRA:
        return '6. Ingeniería / interacción'
    return '7. Identidad de proyecto'

all_vars_21 = sorted(set(v for p in POOLS_21.values() for v in p),
                     key=lambda v: (_grupo_21(v), v))
matriz_21 = pd.DataFrame([
    {'grupo': _grupo_21(v), 'variable': v,
     **{p: ('✓' if v in POOLS_21[p] else '') for p in POOLS_21}}
    for v in all_vars_21])

print("Tamaño de cada pool:")
for p, vs in POOLS_21.items(): print(f'  {p:14s}: {len(vs):2d} variables')
print("\nMatriz de pertenencia variable × pool:")
display(matriz_21.style.set_caption('Variables incluidas en cada pool de la búsqueda (✓ = incluida)')
        .set_properties(subset=['variable'], **{'font-weight':'600'}))
matriz_21.to_csv(RESULTS_DIR/'busqueda_pools_variables.csv', index=False)

# Versión gráfica exportable
fig_h = max(7, 0.30*len(matriz_21))
fig, ax = plt.subplots(figsize=(11.5, fig_h))
ax.axis('off')
tbl = ax.table(cellText=matriz_21.values, colLabels=matriz_21.columns,
               loc='center', cellLoc='center', colLoc='center',
               colWidths=[0.24, 0.26] + [0.083]*len(POOLS_21))
tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.18)
group_colors_21 = {
 '1. Demográfica / lead':'#E3F2FD', '2. Canal':'#E8F5E9', '3. Comercial':'#FFF8E1',
 '4. Lugar':'#FBE9E7', '5. Campaña':'#EDE7F6', '6. Ingeniería / interacción':'#E0F2F1',
 '7. Identidad de proyecto':'#FFEBEE'}
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#263238'); cell.set_text_props(color='white', weight='bold')
    else:
        g = matriz_21.iloc[r-1]['grupo']
        cell.set_facecolor(group_colors_21.get(g, '#FFFFFF'))
        if c in (0, 1): cell.set_text_props(ha='left')
        if c >= 2 and cell.get_text().get_text() == '✓':
            cell.set_text_props(color='#1B5E20', weight='bold')
ax.set_title('Conjuntos de variables (pools) de la búsqueda exhaustiva',
             fontsize=13, fontweight='bold', pad=16)
plt.tight_layout()
plt.savefig(FIGURES_DIR/'busqueda_pools_variables.png', dpi=160, bbox_inches='tight')
plt.show()

### Sección 21.1 — Resultados del grid amplio (fase 1a) y del tuning (fase 1b)

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo</b><br><br>
Se cargan los resultados de las 102 configuraciones del grid (6 pools × 9 algoritmos × 5 estrategias de balanceo, todas evaluadas con validación cruzada 5-fold out-of-fold) y del tuning posterior de hiperparámetros, para comparar qué pool de variables y qué algoritmo rinden mejor.
</div>

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b><br><br>
Lee <code>results_v1_grid.csv</code> y <code>results_v2_tuning.csv</code> desde <code>exploracion/</code>, imprime el top 12 del grid y el top 8 tras tuning, y arma dos gráficas de barras con <code>pivot_table</code>: PR-AUC por pool y algoritmo (sin balanceo) y el efecto del balanceo dentro del pool P4_Eng.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer las gráficas</b>
<ul>
<li><b>Izquierda:</b> cada grupo de barras es un pool de variables y cada color un algoritmo. Barras más altas = mejor PR-AUC. Permite ver dos cosas a la vez: qué pool aporta más información y qué algoritmos la aprovechan mejor.</li>
<li><b>Derecha:</b> cada grupo es una estrategia de balanceo (ninguno, SMOTE, under/oversampling) en el pool P4_Eng. Si las barras son parecidas entre grupos, el balanceo no ayuda — y eso es exactamente lo que se observa.</li>
<li><b>Referencia:</b> PR-AUC es el área bajo la curva precisión-recall. Con una tasa base de ~3% de conversión, el azar equivale a ~0.03; un PR-AUC de 0.22 es ~7 veces mejor que el azar.</li>
</ul>
</div>


In [ ]:
# ── §21.1  Resultados del grid amplio y del tuning ────────────────────────
EXPLOR_DIR = RESULTS_DIR.parent / 'exploracion'
grid_v1 = pd.read_csv(EXPLOR_DIR/'results_v1_grid.csv')
tun_v2  = pd.read_csv(EXPLOR_DIR/'results_v2_tuning.csv')

print(f"Grid fase 1a: {len(grid_v1)} configuraciones evaluadas (CV 5-fold OOF)")
print("\nTOP 12 del grid (por PR-AUC):")
print(grid_v1.head(12)[['pool','alg','balance','n_vars','pr_auc','roc_auc',
                        'lift_10','pct_leads_r70','prec_r70']].to_string(index=False))
print("\nTOP 8 tras tuning de hiperparámetros (fase 1b):")
print(tun_v2.head(8)[['pool','alg','pr_auc','roc_auc','lift_10','pct_leads_r70',
                      'prec_r70']].to_string(index=False))

fig,ax=plt.subplots(1,2,figsize=(14,5))
piv=grid_v1[grid_v1.balance=='none'].pivot_table(index='pool',columns='alg',values='pr_auc')
piv.plot(kind='bar',ax=ax[0],alpha=.88,width=.85)
ax[0].set_title('PR-AUC (OOF) por pool y algoritmo — sin balanceo',fontweight='bold')
ax[0].set_ylabel('PR-AUC');ax[0].tick_params(axis='x',rotation=25);ax[0].legend(fontsize=7,ncol=2)
bal=grid_v1[grid_v1.pool=='P4_Eng'].pivot_table(index='balance',columns='alg',values='pr_auc')
bal.plot(kind='bar',ax=ax[1],alpha=.88)
ax[1].set_title('Efecto del balanceo (pool P4_Eng)',fontweight='bold')
ax[1].set_ylabel('PR-AUC');ax[1].tick_params(axis='x',rotation=15);ax[1].legend(fontsize=7)
plt.tight_layout();plt.savefig(FIGURES_DIR/'busqueda_grid.png',dpi=140,bbox_inches='tight');plt.show()
print("\nLECTURA: los modelos lineales (LDA/LR) y GB lideran; el balanceo (SMOTE/under/over)")
print("NO mejora sobre class_weight/base — consistente con las Secciones 7 y 18.")

### Sección 21.1b — Ablación por familias de variables (fase 1d)

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo</b><br><br>
Se midió el aporte de cada <b>familia completa</b> de variables (canal, edad, ingreso, demografía, razón de compra, campaña, lugar, comercial) de dos maneras complementarias: entrenando con <b>solo</b> esa familia, y entrenando con todo <b>menos</b> esa familia.
</div>

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b><br><br>
Lee <code>results_v4_ablation.csv</code>, filtra los resultados del modelo LDA, separa las filas <code>solo_*</code> (usar solo la familia) de las <code>sin_*</code> (retirar la familia) y grafica un panel para cada caso.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer las gráficas</b>
<ul>
<li><b>Izquierda:</b> PR-AUC de un modelo entrenado usando <b>solo</b> esa familia. Barras largas = la familia tiene mucha señal por sí sola.</li>
<li><b>Derecha:</b> cuánto <b>cae</b> el PR-AUC al retirar la familia del modelo completo. Barras rojas = caída relevante (&gt; 0.01, la familia es necesaria); barras grises = prescindible.</li>
<li>Si una familia puntúa alto sola pero retirarla casi no duele, su información está <b>duplicada</b> en otras variables. La conclusión clave aquí: CANAL es la columna vertebral del modelo — sola alcanza PR≈0.18 (de 0.22 posibles) y retirarla cuesta −0.087.</li>
</ul>
</div>


In [ ]:
# ── §21.1b  Ablación por familias de variables (fase 1d) ─────────────────
abl = pd.read_csv(EXPLOR_DIR/'results_v4_ablation.csv')
abl_lda = abl[(abl.modelo=='LDA')]
print("ABLACIÓN (LDA, pool P4_Eng completo PR=0.2173):")
print(abl_lda.to_string(index=False))

solo = abl_lda[abl_lda.ablacion.str.startswith('solo_')].copy()
solo['familia']=solo['ablacion'].str.replace('solo_','')
sin  = abl_lda[abl_lda.ablacion.str.startswith('sin_')].copy()
sin['familia']=sin['ablacion'].str.replace('sin_','')
fig,ax=plt.subplots(1,2,figsize=(13,4.8))
s1=solo.sort_values('pr_auc')
ax[0].barh(s1['familia'],s1['pr_auc'],color='#1565C0',alpha=.88)
ax[0].set_title('PR-AUC usando SOLO cada familia',fontweight='bold')
ax[0].set_xlabel('PR-AUC (OOF)')
s2=sin.sort_values('caida')
colors_a=['#C62828' if v>0.01 else '#B0BEC5' for v in s2['caida']]
ax[1].barh(s2['familia'],s2['caida'],color=colors_a,alpha=.88)
ax[1].axvline(0,color='k',lw=.8)
ax[1].set_title('Caída de PR-AUC al RETIRAR cada familia',fontweight='bold')
ax[1].set_xlabel('Caída de PR-AUC')
plt.tight_layout();plt.savefig(FIGURES_DIR/'busqueda_ablacion.png',dpi=140,bbox_inches='tight');plt.show()
print("\nLECTURA: CANAL es la columna vertebral (sola PR=0.18 de 0.22; retirarla cuesta -0.087).")
print("Campaña/lugar/comercial aportan poco (+0.002 a +0.005); demografía/ingreso/razón ~0 sobre canal.")

### Sección 21.2 — Selección de variables

Tres estrategias independientes convergen en el mismo mensaje:

1. **Univariado** — `canal_de_contacto` (PR=0.148) y `canal_de_anuencia` (PR=0.129) tienen, cada una por separado, más señal que todas las demográficas juntas; `edad` sola llega a 0.054.
2. **Forward selection** — el orden de entrada es: canal_de_contacto → ingreso_per_capita → canal_de_anuencia → canal_familia → edad×inversión → campania_tipo → lugar_trabajo (PR=0.218 con 7 variables).
3. **Backward elimination** — desde 28 variables se eliminan 17 sin perder desempeño; quedan 11 (PR=0.225): canal (4), lugar_trabajo, campania_tipo, participantes_en_credito y las interacciones de edad (`edad×inversión`, `edad×ingreso`, `edad×referido`, `mayor_45`). Nota: elimina `edad` cruda pero **conserva sus interacciones**.

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
Se buscó el <b>subconjunto mínimo de variables</b> que sostiene el desempeño, con tres métodos independientes: medir cada variable por separado (univariado), ir agregando de una en una la que más mejora (forward) e ir quitando de una en una la que menos aporta (backward). Que los tres métodos coincidan da confianza en la selección.
</div>

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b><br><br>
Lee <code>results_v3_univariate.csv</code> y <code>results_v3_forward.csv</code>, imprime ambas tablas y grafica: la señal univariada de las 14 mejores variables (barras) y la curva del forward selection (PR-AUC acumulado a medida que entran variables).
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer las gráficas</b>
<ul>
<li><b>Izquierda:</b> PR-AUC de una regresión logística entrenada con <b>una sola variable</b>. Barras más largas = más señal individual. Aquí se ve que las variables de canal superan por sí solas a todas las demográficas.</li>
<li><b>Derecha:</b> cada punto es un paso del forward selection: se agrega la variable que más mejora el PR-AUC (el nombre aparece junto al punto). Donde la curva <b>se aplana</b>, las variables adicionales ya no aportan — por eso el modelo final usa pocas variables bien elegidas.</li>
</ul>
</div>


In [ ]:
# ── §21.2  Tablas de selección de variables (fase 1c) ─────────────────────
uni = pd.read_csv(EXPLOR_DIR/'results_v3_univariate.csv')
fwd = pd.read_csv(EXPLOR_DIR/'results_v3_forward.csv')
print("SEÑAL UNIVARIADA (LR con una sola variable, PR-AUC OOF) — top 12:")
print(uni.head(12).to_string(index=False))
print("\nFORWARD SELECTION (greedy):")
print(fwd.to_string(index=False))

fig,ax=plt.subplots(1,2,figsize=(14,5))
u=uni.head(14).iloc[::-1]
ax[0].barh(u['variable'],u['pr_auc_solo'],color='#00897B',alpha=.88)
ax[0].set_title('Señal univariada por variable (PR-AUC sola)',fontweight='bold')
ax[0].set_xlabel('PR-AUC (OOF)')
ax[1].plot(fwd['paso'],fwd['pr_auc'],'o-',color='#1565C0',lw=2)
for _,r in fwd.iterrows():
    ax[1].annotate(r['variable'],(r['paso'],r['pr_auc']),fontsize=7,rotation=25,
                   textcoords='offset points',xytext=(4,-10))
ax[1].set_title('Forward selection: PR-AUC acumulado',fontweight='bold')
ax[1].set_xlabel('Paso');ax[1].set_ylabel('PR-AUC')
plt.tight_layout();plt.savefig(FIGURES_DIR/'busqueda_seleccion.png',dpi=140,bbox_inches='tight');plt.show()

### Sección 21.3 — Fase 2: reconstrucción del universo

Con las variables ganadoras se redefinió el universo de entrenamiento: para cada conjunto S, U(S) = leads calificados con los componentes base de S completos. Se reentrenó dentro de cada universo y se comparó (a) en su propio universo y (b) **sobre las mismas filas del universo de 13K** (comparación directa contra el campeón de fase 1).

| Universo | Requiere completas | Leads | Conv. | %leads p/ recall≥0.70 (propio) |
|---|---|---|---|---|
| S_A solo canal | canal (2) | 42,374 | 729 | 48.8% |
| S_B canal+edad | canal, edad | 20,987 | 638 | 49.4% |
| S_C +lugar | canal, edad, lugar | 19,142 | 631 | 46.1% |
| S_D backward-11 | canal, edad, razón, ingreso, participantes, lugar, campaña | 10,112 | 268 | 52.9% |
| **E sin hijos/núcleo** | 8 de las 10 originales | 18,067 | 617 | **30.4%** |
| F sin estado_civil | 7 | 18,758 | 617 | 29.0% |
| G sin sexo ni e.civil | 6 | 18,992 | 618 | 29.3% |
| **H canal+edad+ingreso+fuente** | **5** | **20,304** | **625** | **26.6–29.0%** |

**Hallazgos:** (1) exigir `campania_tipo` completa (S_D) descarta a la mayoría de los convertidores — los que convierten llegan por canales sin campaña rastreada; (2) relajar `tiene_hijos`/`nucleo_familiar`/`sexo`/`estado_civil` (familias con aporte ~0 en la ablación) **agranda el universo sin costo**; (3) el óptimo es la variante **H**: +56% de leads vs fase 1, con mejor punto operativo.

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
Ya definidas las variables importantes, quedaba decidir <b>a qué leads aplicar el modelo</b>. Cada "universo" exige que ciertos campos estén completos: mientras más campos se exigen, menos leads quedan (y se pierden conversiones), pero el modelo trabaja con mejor información. Se probaron 8 variantes buscando el equilibrio; ganó la variante <b>H</b> (canal + edad + ingreso + fuente): cubre 20,304 leads —56% más que en la fase 1— con el mejor punto operativo.
</div>

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b><br><br>
Lee <code>results_v6_universes.csv</code> (universos S_A–S_D), arma la tabla <code>fase2</code> con los resultados de las 8 variantes (los de E–H provienen de los scripts externos de la fase 2) y dibuja un gráfico de dispersión con burbujas: tamaño del universo vs. costo operativo.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer el gráfico</b>
<ul>
<li><b>Eje X:</b> tamaño del universo — a cuántos leads se les puede aplicar el modelo.</li>
<li><b>Eje Y:</b> % de leads que hay que contactar para capturar el 70% de las conversiones. <b>Menor = mejor</b> (menos esfuerzo comercial para el mismo resultado).</li>
<li><b>Tamaño de la burbuja:</b> cuántas conversiones reales contiene ese universo (universos chicos pueden dejar fuera a los convertidores).</li>
<li>El punto ideal está <b>abajo y a la derecha, con burbuja grande</b>. Verde oscuro = universo H (el elegido); verde claro = variante E (segunda mejor); gris = fase 1 y variantes descartadas.</li>
</ul>
</div>


In [ ]:
# ── §21.3  Comparación de universos (fase 2) ──────────────────────────────
univ6 = pd.read_csv(EXPLOR_DIR/'results_v6_universes.csv')
print("Universos S_A–S_D (script explore_v6_universe.py):")
print(univ6.to_string(index=False))

fase2 = pd.DataFrame([
 {'universo':'Fase1 (10 vars)','n':13038,'conv':563,'pctleads_r70':36.2},
 {'universo':'S_A solo canal','n':42374,'conv':729,'pctleads_r70':48.8},
 {'universo':'S_B canal+edad','n':20987,'conv':638,'pctleads_r70':49.4},
 {'universo':'S_C +lugar','n':19142,'conv':631,'pctleads_r70':46.1},
 {'universo':'S_D backward11','n':10112,'conv':268,'pctleads_r70':52.9},
 {'universo':'E sin hijos/nucleo','n':18067,'conv':617,'pctleads_r70':30.4},
 {'universo':'H canal+edad+ingreso','n':20304,'conv':625,'pctleads_r70':27.8},
])
fig,ax=plt.subplots(figsize=(9,5.5))
colors_u=['#78909C']+['#B0BEC5']*4+['#26A69A','#2E7D32']
sc=ax.scatter(fase2['n'],fase2['pctleads_r70'],s=fase2['conv']*0.6,
              c=colors_u,alpha=.85,edgecolors='k',linewidths=.5)
for _,r in fase2.iterrows():
    ax.annotate(r['universo'],(r['n'],r['pctleads_r70']),fontsize=8,
                textcoords='offset points',xytext=(8,4))
ax.set_xlabel('Tamaño del universo (leads)')
ax.set_ylabel('% de leads para recall ≥ 0.70 (menor = mejor)')
ax.set_title('Fase 2: tamaño del universo vs. costo operativo\n(tamaño de burbuja = conversiones)',fontweight='bold')
plt.tight_layout();plt.savefig(FIGURES_DIR/'busqueda_universos.png',dpi=140,bbox_inches='tight');plt.show()

### Sección 21.4 — Modelo final (reproducible en el notebook)

**Universo H:** leads con `fuente_pt2`, `canal_de_anuencia`, `canal_de_contacto`, `ingreso_familiar` y `edad` completas (20,304 leads, 625 conversiones, tasa 3.08%). Las demás variables entran como opcionales (categoría `_missing_` cuando faltan — señal de completitud de perfil, metodología aceptada desde la Sección 14; las variables de fuga de la Sección 20 siguen excluidas).

**Modelo:** Regresión Logística L2, `C=0.55` (tuneada en fase 1b), OHE con agrupación de categorías raras (`min_frequency=30`) — este agrupamiento fue clave para destapar la señal de canal que el hold-out simple de la Sección 19 no veía.

**Generalización a proyectos nuevos (decisión final):** `fuente_pt2` y `proyecto` —la identidad del proyecto— quedan **fuera de las variables predictoras**, porque el objetivo del modelo es aplicarse a proyectos futuros que no existen en los datos de entrenamiento. `fuente_pt2` se mantiene únicamente como requisito de completitud del universo H. Costo medido de excluirlas (OOF 5-fold, 2 semillas, mismas filas): PR-AUC 0.221 → 0.210, lift@10% 4.3 → 4.0–4.1, y el punto de recall 0.70 pasa de 26.6–29.0% a 29.0–30.0% de leads — un costo pequeño que se acepta a cambio de un modelo **portable a proyectos nuevos**.

### ¿Qué hace el código de la celda siguiente? (§21.4)

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b><br><br>
<b>En palabras simples:</b> esta celda construye el modelo final y lo evalúa de forma honesta — cada lead se califica con un modelo que <b>nunca lo vio</b> durante el entrenamiento.
<ol>
<li><b>Crea variables derivadas:</b> agrupa los canales de anuencia en familias (<code>pauta_digital</code>, <code>portal</code>, <code>directo</code>, <code>referido</code>, <code>presencial</code>, …), marca si el lead es referido o compra por inversión, y construye interacciones (p. ej. <code>edad × ingreso</code>) que capturan combinaciones que una variable sola no muestra.</li>
<li><b>Define el universo H:</b> se queda solo con los leads que tienen completos fuente, canal de anuencia, canal de contacto, ingreso y edad (20,304 leads). Las demás variables pueden faltar: el faltante se convierte en la categoría <code>_missing_</code>. Ojo: la fuente y el proyecto solo se usan para definir el universo, <b>no como variables predictoras</b> — así el modelo generaliza a proyectos nuevos.</li>
<li><b>Arma el pipeline:</b> imputación de faltantes + one-hot encoding (agrupando categorías con menos de 30 casos) + escalado de numéricas + Regresión Logística con regularización L2 (<code>C=0.55</code>).</li>
<li><b>Evalúa con validación cruzada OOF (out-of-fold):</b> divide los datos en 5 partes; cada parte se predice con un modelo entrenado en las otras 4. Se repite con 2 semillas distintas para confirmar que el resultado no depende del azar de la partición.</li>
<li><b>Construye la tabla de umbrales operativos:</b> para cada recall objetivo (50%, 60%, 70%, 80%) calcula qué umbral de probabilidad hay que usar, cuántos leads implica contactar y con qué precisión.</li>
</ol>
</div>

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Resultados clave para las secciones siguientes:</b> <code>prob_H</code> (probabilidad OOF de cada lead), <code>y_H</code> (si convirtió o no) y <code>thr_final</code> (tabla de umbrales operativos).
</div>


In [ ]:
# ── §21.4  Modelo final: universo H + LR tuneada, OOF con 2 semillas ─────
from sklearn.model_selection import cross_val_predict

CANAL_FAMILIA = {
 'Facebook':'pauta_digital','Instagram':'pauta_digital','Fanpage Líderes':'pauta_digital',
 'Inbox':'pauta_digital','portal_demo':'portal','Página Web':'portal',
 'WhatsApp':'directo','Llamada':'directo',
 'Referencia':'referido','Friends/Family':'referido','Shareholders Sale':'referido',
 'Sede':'presencial','ExpoCasa':'presencial','ExpoCasa Virtual':'presencial',
 'Visita Inédita':'presencial','Open House':'presencial','OpenSale':'presencial',
 'Ferias':'presencial','Evento':'presencial','Actividades MKT':'presencial',
 'Actividad Posicionamiento':'presencial','Lanzamiento':'presencial',
 'Vallas':'exterior','Corporación':'otro','usa':'otro','Prensa':'otro','Radio/tv':'otro'}

df_f = df_model.copy()
df_f['edad_n'] = pd.to_numeric(df_f['edad'], errors='coerce')
df_f['es_inversion'] = (df_f['razon_de_compra']=='Inversión').astype(int)
df_f['canal_familia'] = df_f['canal_de_anuencia'].map(CANAL_FAMILIA).fillna('otro')
df_f['es_referido'] = (df_f['canal_familia']=='referido').astype(int)
df_f['edad_x_inversion'] = df_f['edad_n']*df_f['es_inversion']
df_f['edad_x_ingreso'] = df_f['edad_n']*df_f['ingreso_mid']/100.0
df_f['edad_x_referido'] = df_f['edad_n']*df_f['es_referido']
df_f['mayor_45'] = (df_f['edad_n']>=45).astype(int)
df_f['ingreso_x_inversion'] = df_f['ingreso_mid']*df_f['es_inversion']
df_f['ingreso_alto'] = (df_f['ingreso_mid']>=32).astype(int)

REQ_H = ['fuente_pt2','canal_de_anuencia','canal_de_contacto','ingreso_familiar','edad']
# fuente_pt2 y proyecto (identidad de proyecto) se EXCLUYEN como predictoras:
# el objetivo es generalizar a proyectos nuevos. fuente_pt2 permanece en REQ_H
# solo como requisito de completitud del universo, no como variable del modelo.
FEATS_H = ['edad_n','sexo','estado_civil','ingreso_mid','ingreso_familiar','razon_de_compra',
 'canal_de_anuencia','canal_de_contacto','canal_familia','tipo_unidad','opciones_visitadas',
 'participantes_en_credito','lugar_trabajo','lugar_vivienda','campania_tipo','campania_audiencia',
 'es_inversion','es_referido','edad_x_inversion','ingreso_x_inversion','edad_x_ingreso',
 'edad_x_referido','mayor_45','ingreso_alto']

mask_H = df_f[REQ_H].notna().all(axis=1)
df_H = df_f.loc[mask_H].reset_index(drop=True)
y_H = df_H['target_won'].values
print(f"Universo H: {len(df_H):,} leads | {int(y_H.sum())} conversiones ({y_H.mean()*100:.2f}%)")

cat_H = [c for c in FEATS_H if not pd.api.types.is_numeric_dtype(df_H[c])]
num_H = [c for c in FEATS_H if c not in cat_H]
prep_H = ColumnTransformer([
  ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='_missing_')),
                    ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                                          min_frequency=30))]), cat_H),
  ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                    ('sc', StandardScaler())]), num_H)], remainder='drop')
pipe_H = Pipeline([('prep', prep_H),
                   ('clf', LogisticRegression(C=0.55, penalty='l2', max_iter=2000,
                                              random_state=SEED))])

met_rows=[]
prob_H = None
for tag,seed in [('selección',SEED),('confirmación',2025)]:
    skf_f = StratifiedKFold(5, shuffle=True, random_state=seed)
    prob = cross_val_predict(pipe_H, df_H[FEATS_H], y_H, cv=skf_f,
                             method='predict_proba', n_jobs=-1)[:,1]
    if prob_H is None: prob_H = prob
    bm = business_at_k(y_H, prob)
    prc,rcc,thr = precision_recall_curve(y_H, prob)
    i70 = np.where(rcc[:-1]>=0.70)[0][-1]
    met_rows.append({'CV':tag,'pr_auc':round(average_precision_score(y_H,prob),4),
        'roc_auc':round(roc_auc_score(y_H,prob),4),
        'lift@10%':round(bm[0.10]['lift'],2),'recall@20%':round(bm[0.20]['rec'],3),
        'pct_leads_r70':round((prob>=thr[i70]).mean()*100,1),
        'prec_r70':round(prc[i70],4)})
met_final = pd.DataFrame(met_rows)
print("\nMÉTRICAS DEL MODELO FINAL (OOF 5-fold, dos semillas):")
print(met_final.to_string(index=False))
met_final.to_csv(RESULTS_DIR/'modelo_final_metricas.csv', index=False)

thr_rows=[]
prc,rcc,thr = precision_recall_curve(y_H, prob_H)
for target in [0.50,0.60,0.70,0.80]:
    idx=np.where(rcc[:-1]>=target)[0]
    if len(idx):
        i=idx[-1]; sel=prob_H>=thr[i]
        thr_rows.append({'recall_objetivo':target,'umbral':round(thr[i],4),
            'recall':round(rcc[i],3),'precision':round(prc[i],4),
            'lift_vs_base':round(prc[i]/y_H.mean(),2),
            'leads':int(sel.sum()),'pct_leads':round(sel.mean()*100,1)})
thr_final = pd.DataFrame(thr_rows)
print("\nUMBRALES OPERATIVOS (OOF selección):")
print(thr_final.to_string(index=False))
thr_final.to_csv(RESULTS_DIR/'modelo_final_umbrales.csv', index=False)

### Sección 21.5 — Curvas del modelo final

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo</b><br><br>
Se grafican cuatro vistas complementarias del desempeño del modelo final, usando las probabilidades OOF (cada lead evaluado por un modelo que no lo vio al entrenar).
</div>

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b><br><br>
Ordena los leads de mayor a menor probabilidad y acumula las conversiones capturadas (curvas de ganancia y lift); calcula la curva precisión-recall con <code>precision_recall_curve</code> y la curva de calibración con <code>calibration_curve</code> (10 grupos de igual tamaño).
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer cada gráfico</b>
<table style="border-collapse:collapse; margin-top:8px;">
<tr><th style="border:1px solid #D9C46B; padding:6px 10px; background:#FCF0C8; text-align:left; color:#212121;">Gráfico</th><th style="border:1px solid #D9C46B; padding:6px 10px; background:#FCF0C8; text-align:left; color:#212121;">Cómo leerlo</th></tr>
<tr><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;"><b>Ganancia acumulada</b> (arriba izq.)</td><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;">Eje X: % de leads contactados, empezando por los de mayor probabilidad. Eje Y: % de las conversiones reales capturadas. Cuanto más se despega la curva azul de la diagonal ("aleatorio"), mejor. La línea roja punteada marca la meta de capturar el 70%.</td></tr>
<tr><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;"><b>Lift</b> (arriba der.)</td><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;">Cuántas veces más conversiones se obtienen respecto a contactar al azar. Un lift de 4 en el 10% significa: contactando solo al 10% mejor rankeado, se consiguen 4× más conversiones que eligiendo leads al azar.</td></tr>
<tr><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;"><b>Precisión-Recall</b> (abajo izq.)</td><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;">El intercambio entre capturar más conversiones (recall, eje X) y acertar más entre los contactados (precisión, eje Y). La línea gris es la tasa base (3.08%): mientras la curva esté por encima, el modelo aporta valor sobre no usar modelo.</td></tr>
<tr><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;"><b>Calibración</b> (abajo der.)</td><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;">Compara la probabilidad predicha con la frecuencia real de conversión. Si los puntos siguen la diagonal, las probabilidades son confiables: un lead con probabilidad 0.10 realmente convierte ≈10% de las veces.</td></tr>
</table>
</div>


In [ ]:
# ── §21.5  Curvas del modelo final ────────────────────────────────────────
from sklearn.calibration import calibration_curve
fig,ax=plt.subplots(2,2,figsize=(14,10))
order=np.argsort(prob_H)[::-1]; cg=np.cumsum(y_H[order])/y_H.sum()
pct=np.arange(1,len(y_H)+1)/len(y_H)
ax[0,0].plot(pct,cg,lw=2,color='#1565C0',label='Modelo final')
ax[0,0].plot([0,1],[0,1],'k--',lw=.8,label='Aleatorio')
ax[0,0].axhline(0.70,color='red',ls=':',lw=1,label='Recall 70%')
ax[0,0].set_title('Curva de ganancia acumulada',fontweight='bold')
ax[0,0].set_xlabel('% leads contactados');ax[0,0].set_ylabel('% conversiones capturadas');ax[0,0].legend(fontsize=8)
lift=cg/np.clip(pct,1e-9,None)
ax[0,1].plot(pct,lift,lw=2,color='#6A1B9A');ax[0,1].axhline(1,color='k',ls='--',lw=.8)
ax[0,1].set_title('Curva de lift',fontweight='bold')
ax[0,1].set_xlabel('% leads contactados');ax[0,1].set_ylabel('Lift');ax[0,1].set_xlim(0,1)
prc,rcc,_=precision_recall_curve(y_H,prob_H)
ax[1,0].plot(rcc,prc,lw=2,color='#2E7D32',label=f'AP={average_precision_score(y_H,prob_H):.3f}')
ax[1,0].axhline(y_H.mean(),color='gray',ls='--',lw=.8,label=f'Base={y_H.mean():.3f}')
ax[1,0].set_title('Curva Precision-Recall',fontweight='bold')
ax[1,0].set_xlabel('Recall');ax[1,0].set_ylabel('Precision');ax[1,0].legend(fontsize=8)
fp,mpv=calibration_curve(y_H,prob_H,n_bins=10,strategy='quantile')
ax[1,1].plot(mpv,fp,'o-',color='#C62828',label='Modelo')
ax[1,1].plot([0,max(mpv)],[0,max(mpv)],'k--',lw=.8,label='Perfecta')
ax[1,1].set_title('Curva de calibración',fontweight='bold')
ax[1,1].set_xlabel('Prob. predicha media');ax[1,1].set_ylabel('Frac. positivos observada');ax[1,1].legend(fontsize=8)
plt.suptitle('Modelo final — LR (C=0.55) | Universo H (20,304 leads) | OOF 5-fold',
             fontsize=13,fontweight='bold')
plt.tight_layout();plt.savefig(FIGURES_DIR/'modelo_final_curvas.png',dpi=140,bbox_inches='tight');plt.show()

### Sección 21.6 — Importancia de variables y serialización del modelo

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo</b><br><br>
Se midió qué variables sostienen al modelo final con dos métodos independientes, y se guardó el modelo entrenado (<code>.pkl</code>) para poder usarlo fuera del notebook.
</div>

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b>
<ol>
<li>Separa un 20% de hold-out y entrena el pipeline con el 80% restante.</li>
<li><b>Permutation importance:</b> desordena una variable a la vez y mide cuánto cae el PR-AUC. Si al desordenarla el modelo empeora mucho, esa variable era importante. Se repite 15 veces para estimar la variabilidad (barras de error).</li>
<li><b>SHAP:</b> calcula la contribución media de cada <i>feature transformada</i> (cada categoría del one-hot por separado) a las predicciones individuales.</li>
<li>Guarda el pipeline completo con <code>pickle</code>, junto con la lista de features, los requisitos del universo y los umbrales operativos.</li>
</ol>
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer las gráficas</b>
<ul>
<li><b>Izquierda (permutation):</b> barras más largas = variables más importantes, a nivel de variable original. La barra de error indica estabilidad: si es pequeña respecto a la barra, la importancia es confiable.</li>
<li><b>Derecha (SHAP):</b> la misma idea pero a nivel de categoría individual (p. ej., <code>canal_de_anuencia_Referencia</code> por separado). Permite ver <i>qué categoría específica</i> de una variable es la que mueve al modelo.</li>
</ul>
</div>


In [ ]:
# ── §21.6  Importancia (permutación + SHAP) y serialización ──────────────
import pickle
tr_H, te_H = train_test_split(np.arange(len(df_H)), test_size=0.20,
                              random_state=SEED, stratify=y_H)
pipe_H.fit(df_H.iloc[tr_H][FEATS_H], y_H[tr_H])
pi_H = permutation_importance(pipe_H, df_H.iloc[te_H][FEATS_H], y_H[te_H],
                              scoring='average_precision', n_repeats=15,
                              random_state=SEED, n_jobs=-1)
imp_H = pd.DataFrame({'variable':FEATS_H,'imp_media':pi_H.importances_mean,
                      'imp_std':pi_H.importances_std}).sort_values('imp_media',ascending=False)
print("PERMUTATION IMPORTANCE (hold-out 20% del universo H) — top 15:")
print(imp_H.head(15).round(4).to_string(index=False))
imp_H.to_csv(RESULTS_DIR/'modelo_final_importancia.csv', index=False)

import shap
Xt_H = pipe_H.named_steps['prep'].transform(df_H.iloc[te_H][FEATS_H])
names_H = pipe_H.named_steps['prep'].get_feature_names_out()
expl = shap.LinearExplainer(pipe_H.named_steps['clf'], Xt_H)
sv = expl.shap_values(Xt_H)
shap_H = pd.DataFrame({'feature':names_H,'mean_abs_shap':np.abs(sv).mean(axis=0)})\
    .sort_values('mean_abs_shap',ascending=False)
print("\nSHAP |valor| medio — top 15 features transformadas:")
print(shap_H.head(15).round(4).to_string(index=False))
shap_H.to_csv(RESULTS_DIR/'modelo_final_shap.csv', index=False)

fig,ax=plt.subplots(1,2,figsize=(14,6))
t=imp_H.head(14).iloc[::-1]
ax[0].barh(t['variable'],t['imp_media'],xerr=t['imp_std'],color='#00897B',alpha=.88,capsize=3)
ax[0].set_title('Permutation importance (± std)',fontweight='bold')
ax[0].set_xlabel('Caída de PR-AUC')
s=shap_H.head(14).iloc[::-1]
ax[1].barh(s['feature'],s['mean_abs_shap'],color='#5C6BC0',alpha=.88)
ax[1].set_title('SHAP |valor| medio (features transformadas)',fontweight='bold')
ax[1].set_xlabel('|SHAP| medio')
plt.tight_layout();plt.savefig(FIGURES_DIR/'modelo_final_importancia.png',dpi=140,bbox_inches='tight');plt.show()

with open(MODELS_DIR/'modelo_final_busqueda.pkl','wb') as f:
    pickle.dump({'pipeline':pipe_H,'features':FEATS_H,'requisitos_universo':REQ_H,
                 'metricas':met_final.to_dict('records'),
                 'umbrales':thr_final.to_dict('records')}, f)
print(f"\nModelo serializado en {MODELS_DIR/'modelo_final_busqueda.pkl'}")

## Conclusión — Sección 21

**La búsqueda exhaustiva encontró un modelo sustancialmente mejor que todos los anteriores, sin variables de fuga y sin identidad de proyecto (generalizable a proyectos nuevos):**

| | Sección 20 (honesto, hold-out) | **Sección 21 (búsqueda + CV, 24 vars)** |
|---|---|---|
| Universo | 13,038 leads | **20,304 leads (+56%)** |
| PR-AUC | ~0.07 (hold-out) | **0.210 (OOF, 2 semillas)** |
| % leads para recall≥0.70 | 64.2% | **29.0–30.0%** |
| Precisión en ese punto | 5.1% (1.2× base) | **7.2–7.4% (~2.4× base)** |
| Lift@10% | ~2.6 | **4.0–4.1** |

**Generalización a proyectos nuevos.** `fuente_pt2` y `proyecto` se excluyeron de las variables predictoras: la identidad del proyecto no existe para proyectos futuros, así que un modelo que dependa de ella no es aplicable donde más se necesita. El costo medido fue pequeño (PR-AUC 0.221 → 0.210; lift@10% 4.3 → 4.0–4.1; recall 0.70 pasa de costar 26.6–29.0% a 29.0–30.0% de los leads) y se acepta a cambio de la portabilidad. La búsqueda ya lo anticipaba: la permutation importance de ambas era ~0 individualmente (información duplicada entre sí).

**Por qué la Sección 19–20 subestimó la señal.** Tres causas identificadas: (1) el hold-out simple de 20% con 113 conversiones es muy ruidoso — la CV 5-fold out-of-fold usa las 625; (2) sin agrupación de categorías raras (`min_frequency`), las decenas de niveles poco frecuentes de canal/lugar diluían la señal en la regularización; (3) faltaban las interacciones (`edad×ingreso`, `edad×referido`) que portan la señal conjunta.

**Variables que realmente importan** (permutation + SHAP + ablación coinciden): la familia **canal** (`es_referido`, `canal_de_contacto`, `canal_de_anuencia`) es la columna vertebral; le siguen las **interacciones de edad** (`edad×ingreso`, `edad×inversión`) y aportes menores de `lugar_trabajo` y `campania_tipo`. Demografía pura (sexo, estado civil, hijos, núcleo) no aporta sobre lo anterior — por eso pudieron salir de los requisitos del universo.

**Universo final (H).** Exigir completas solo `fuente`, `canal_de_anuencia`, `canal_de_contacto`, `ingreso_familiar` y `edad` da 20,304 leads (48% del calificado) reteniendo 625 de 737 conversiones (85%). Ampliar más (solo canal, 42K) degrada el desempeño; exigir más (universos de las Secciones 15/17) lo encarece sin mejorarlo.

**Recomendación operativa** (umbrales OOF, modelo de 24 variables): con el 10% de leads mejor rankeados se captura ~41% de conversiones (lift 4.1×); recall 0.70 cuesta ~29–30% de los leads con precisión ~7.2%; recall 0.60 cuesta ~21% con precisión ~8.9% (2.9× base); recall 0.50 cuesta ~14.5% con precisión ~10.6% (3.5× base). El pipeline serializado (`modelo_final_busqueda.pkl`) incluye features, requisitos de universo y umbrales.

> **Nota de honestidad metodológica.** La selección de configuraciones usó CV sobre el mismo dataset (riesgo de sobre-selección); como mitigación, todos los números reportados se **confirmaron con una segunda semilla de CV** y el patrón (canal ≫ resto) es consistente entre metodologías independientes (univariado, forward, backward, ablación, permutación, SHAP). La validación definitiva sería con leads futuros (out-of-time), que no existen aún en este corte de datos.

---

## Sección 22 — Matriz de confusión y recall total del modelo final

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
Hasta aquí el modelo entrega una <b>probabilidad</b> por lead. Para usarlo en la práctica hay que convertir esa probabilidad en una decisión sí/no, aplicando el umbral operativo (recall ≥ 0.70) ya definido en la §21.4, y viendo la matriz de confusión y el recall sobre <b>todo</b> el universo H.
</div>

In [ ]:
# ── §22  Matriz de confusión + recall TOTAL del modelo final ─────────────
# Umbral operativo: el menor umbral que logra recall >= 0.70 sobre las
# probabilidades OOF del universo H (mismo criterio de la §21.4).
prc_f, rcc_f, thr_f = precision_recall_curve(y_H, prob_H)
i70 = np.where(rcc_f[:-1] >= 0.70)[0][-1]
umbral_op = thr_f[i70]

# Decisión sí/no por lead y conteo de aciertos/errores en TODO el universo
y_pred_H = (prob_H >= umbral_op).astype(int)
cm = confusion_matrix(y_H, y_pred_H)
vn, fp, fn, vp = cm.ravel()

recall_total  = vp / (vp + fn)          # sobre todo el universo, no @K
precision_op  = vp / (vp + fp)
especificidad = vn / (vn + fp)
exactitud     = (vp + vn) / cm.sum()
f1_op         = 2 * precision_op * recall_total / (precision_op + recall_total)

print(f"Umbral operativo: {umbral_op:.4f}")
print(f"Leads marcados 'contactar': {int(y_pred_H.sum()):,} de {len(y_H):,} "
      f"({y_pred_H.mean()*100:.1f}% del universo)")
print(f"VN={vn:,} | FP={fp:,} | FN={fn:,} | VP={vp:,}")

met_cm = pd.DataFrame([
    {'métrica': 'Recall TOTAL  VP/(VP+FN)', 'valor': round(recall_total, 4),
     'lectura': f'detecta {vp} de {vp+fn} conversiones reales del universo'},
    {'métrica': 'Precisión     VP/(VP+FP)', 'valor': round(precision_op, 4),
     'lectura': f'{precision_op*100:.1f}% de los contactados convierte (base: {y_H.mean()*100:.2f}%)'},
    {'métrica': 'Especificidad VN/(VN+FP)', 'valor': round(especificidad, 4),
     'lectura': f'descarta correctamente al {especificidad*100:.1f}% de los que no convierten'},
    {'métrica': 'Exactitud', 'valor': round(exactitud, 4),
     'lectura': 'poco informativa con clases desbalanceadas (3% de positivos)'},
    {'métrica': 'F1', 'valor': round(f1_op, 4),
     'lectura': 'media armónica de precisión y recall'},
])
print("\nMÉTRICAS GLOBALES AL UMBRAL OPERATIVO (todo el universo H):")
print(met_cm.to_string(index=False))
met_cm.to_csv(RESULTS_DIR/'modelo_final_matriz_confusion_metricas.csv', index=False)

# Mapa de calor: color = % dentro de cada fila (si no, el desbalance 97/3
# dejaría casi invisibles las celdas de la clase positiva)
cm_rowpct = cm / cm.sum(axis=1, keepdims=True)
etiquetas = np.array([
    [f"VN\n{vn:,}\n({vn/(vn+fp)*100:.1f}% de los reales 'No')",
     f"FP\n{fp:,}\n({fp/(vn+fp)*100:.1f}% de los reales 'No')"],
    [f"FN\n{fn:,}\n({fn/(fn+vp)*100:.1f}% de los reales 'Sí')",
     f"VP\n{vp:,}\n({vp/(fn+vp)*100:.1f}% de los reales 'Sí')"]])

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(cm_rowpct, annot=etiquetas, fmt='', cmap='Blues', vmin=0, vmax=1,
            cbar=False, linewidths=2, linecolor='white',
            annot_kws={'fontsize': 11}, ax=ax,
            xticklabels=['No convierte', 'Convierte'],
            yticklabels=['No convirtió', 'Convirtió'])
ax.set_xlabel('Predicción del modelo', fontweight='bold')
ax.set_ylabel('Realidad', fontweight='bold')
ax.set_title(f'Matriz de confusión — modelo final (universo H, OOF)\n'
             f'umbral={umbral_op:.4f} | recall total={recall_total:.3f} | '
             f'precisión={precision_op:.3f}', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR/'modelo_final_matriz_confusion.png', dpi=140, bbox_inches='tight')
plt.show()


---

## Sección 23 — Importancia consolidada de variables del modelo final

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
Se consolida en una sola vista qué variables usa el modelo final y en qué dirección actúan, reutilizando la importancia por permutación (§21.6) junto con los coeficientes (odds ratio) del modelo entrenado.
</div>

In [ ]:
# ── §23  Importancia consolidada de variables del modelo final ───────────
# Reutiliza imp_H (permutación) y pipe_H (ya entrenado) de la §21.6.
top_imp = imp_H.head(12).iloc[::-1]

coef_df = pd.DataFrame({
    'feature': [n.replace('cat__', '').replace('num__', '') for n in names_H],
    'coef': pipe_H.named_steps['clf'].coef_[0]})
coef_df['odds_ratio'] = np.exp(coef_df['coef'])
coef_df = coef_df.sort_values('coef', ascending=False).reset_index(drop=True)
top_dir = pd.concat([coef_df.head(8), coef_df.tail(8)])
top_dir = top_dir[~top_dir.index.duplicated()].sort_values('coef')

print("TOP 12 VARIABLES POR IMPORTANCIA (permutación, caída de PR-AUC):")
print(imp_H.head(12).round(4).to_string(index=False))
print("\nCARACTERÍSTICAS CON MAYOR EFECTO (coeficientes del modelo final):")
print(top_dir.iloc[::-1].round(4).to_string(index=False))
coef_df.to_csv(RESULTS_DIR/'modelo_final_coeficientes.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(15, 6.5))
ax[0].barh(top_imp['variable'], top_imp['imp_media'],
           xerr=top_imp['imp_std'], color='#00897B', alpha=.88, capsize=3)
ax[0].set_title('¿Cuánto pesa cada variable?\n(permutation importance ± std)',
                fontweight='bold')
ax[0].set_xlabel('Caída de PR-AUC al desordenar la variable')

colores = ['#2E7D32' if c > 0 else '#C62828' for c in top_dir['coef']]
ax[1].barh(top_dir['feature'], top_dir['coef'], color=colores, alpha=.88)
ax[1].axvline(0, color='k', lw=.8)
ax[1].tick_params(axis='y', labelsize=9)
ax[1].set_title('¿En qué dirección empuja cada característica?\n'
                '(coeficientes de la regresión logística)', fontweight='bold')
ax[1].set_xlabel('Coeficiente   (← reduce  |  aumenta →  la prob. de conversión)')

plt.suptitle('Importancia de variables — Modelo final (LR C=0.55, universo H)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR/'modelo_final_importancia_consolidada.png',
            dpi=140, bbox_inches='tight')
plt.show()


---

## Sección 24 — Fórmula matemática del modelo final

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
La regresión logística no es una caja negra: es una fórmula cerrada que se puede escribir en papel. Se renderiza aquí la ecuación completa del modelo final con sus coeficientes entrenados (pipe_H, §21.6).
</div>

In [ ]:
# ── §24  La fórmula matemática del modelo final (renderizada) ─────────────
from IPython.display import display, Math, Markdown

# Coeficientes del modelo entrenado (pipe_H, §21.6)
clf_f = pipe_H.named_steps['clf']
beta0 = clf_f.intercept_[0]
betas = clf_f.coef_[0]
nombres_f = [n.replace('cat__', '').replace('num__', '') for n in names_H]

def _tex(s):
    """Nombre de variable seguro para modo texto de LaTeX/KaTeX."""
    s = s.replace('\\', ' ')
    for a, b in [('_', r'\_'), ('%', r'\%'), ('&', r'\&'), ('#', r'\#'),
                 ('$', r'\$'), ('{', r'\{'), ('}', r'\}'), ('^', ' '), ('~', ' ')]:
        s = s.replace(a, b)
    return s

# 1) Forma general (símbolos)
display(Markdown("**Forma general** — regresión logística (función sigmoide):"))
display(Math(r'\hat{P}(\text{conversión}) = \sigma(z) = \frac{1}{1 + e^{-z}}'
             r'\qquad\text{con}\qquad z = \beta_0 + \sum_{i=1}^{%d}\beta_i\,x_i'
             % len(betas)))

# 2) Fórmula completa con los coeficientes reales, ordenada por |beta|.
#    Se parte en bloques de CHUNK términos: KaTeX limita el nº de expansiones
#    y un único bloque con ~80 términos lo supera ("Too many expansions").
orden = list(np.argsort(-np.abs(betas)))
CHUNK = 12
partes = [orden[i:i+CHUNK] for i in range(0, len(orden), CHUNK)]
display(Markdown(f"**Fórmula completa** con los {len(betas)} coeficientes del modelo final "
                 f"(términos ordenados de mayor a menor peso $|\\beta|$; se muestra en "
                 f"{len(partes)} partes por su longitud):"))
for p_i, part in enumerate(partes):
    filas = [r'z = {}& %+.4f \\' % beta0] if p_i == 0 else []  # intercepto en la 1ª parte
    for i in part:
        filas.append(r'& %+.4f\;\text{%s} \\' % (betas[i], _tex(nombres_f[i])))
    display(Math(r'\begin{aligned}' + '\n' + '\n'.join(filas) + '\n' + r'\end{aligned}'))

# Copia en texto plano
lineas_txt = [f"z = {beta0:+.4f}   (intercepto)"] + \
             [f"    {betas[i]:+.4f} * [{nombres_f[i]}]" for i in orden]
(RESULTS_DIR/'modelo_final_formula.txt').write_text(
    "P(conversion) = 1 / (1 + e^(-z))\n\n" + "\n".join(lineas_txt), encoding='utf-8')

# 3) Ejemplo aplicado a un lead real, también como fórmula
i_lead = int(np.argmax(prob_H))                       # el lead mejor rankeado
x_fila = pipe_H.named_steps['prep'].transform(df_H[FEATS_H].iloc[[i_lead]])[0]
z_lead = beta0 + float(x_fila @ betas)
p_lead = 1 / (1 + np.exp(-z_lead))
p_pipe = pipe_H.predict_proba(df_H[FEATS_H].iloc[[i_lead]])[0, 1]

contrib = x_fila * betas
activos = np.where(np.abs(contrib) > 1e-9)[0]
activos = activos[np.argsort(-np.abs(contrib[activos]))]
filas = [r'z = {}& %+.4f & \text{(intercepto)} \\' % beta0]
for j in activos[:10]:
    filas.append(r'& %+.4f \cdot %.3f & \text{(%s)} \\'
                 % (betas[j], x_fila[j], _tex(nombres_f[j])))
if len(activos) > 10:
    filas.append(r'& %+.4f & \text{(suma de los %d términos restantes)} \\'
                 % (contrib[activos[10:]].sum(), len(activos) - 10))
filas.append(r'= {}& %.4f & \\' % z_lead)
display(Markdown(f"**Ejemplo** — el lead mejor rankeado del universo H (índice {i_lead}). "
                 f"Solo se activan {len(activos)} de {len(betas)} términos (el resto vale 0):"))
display(Math(r'\begin{aligned}' + '\n' + '\n'.join(filas) + '\n' + r'\end{aligned}'))
display(Math(r'\hat{P}(\text{conversión}) = \frac{1}{1 + e^{-(%.4f)}} = %.4f'
             % (z_lead, p_lead)))

# Verificación: la fórmula reproduce exactamente el pipeline
print(f"Verificación:  fórmula = {p_lead:.6f}   |   pipeline.predict_proba = {p_pipe:.6f}"
      f"   |   diferencia = {abs(p_lead - p_pipe):.1e}")
assert abs(p_lead - p_pipe) < 1e-9, "La fórmula debe coincidir con el pipeline"
print("La fórmula reproduce exactamente el modelo.")


---

## Sección 25 — Red Neuronal (Perceptrón Multicapa)

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
Se replica, adaptado a este proyecto, el ejercicio de referencia "Regresión con RNA": una red neuronal densa con capas ocultas ReLU, <i>early stopping</i> para prevenir el sobreajuste y curvas de aprendizaje, comparando su desempeño contra el modelo final (regresión logística).
</div>

In [ ]:
# ── §25.1  Red neuronal (MLP): construcción, ajuste y curvas de aprendizaje ─
# Inspirado en el ejercicio "Regresión con RNA" (Keras Sequential + Dense +
# EarlyStopping), adaptado a clasificación binaria con MLPClassifier.
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),  # 3 capas ocultas ReLU
    activation='relu',
    solver='adam',                     # optimizador (análogo a rmsprop/Adam del doc)
    alpha=1e-4,                        # regularización L2
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=500,                      # techo de épocas
    early_stopping=True,               # análogo a EarlyStopping de Keras
    validation_fraction=0.15,
    n_iter_no_change=25,               # 'patience'
    random_state=SEED,
)
pipe_mlp = Pipeline([('prep', clone(prep_H)), ('clf', mlp)])

# Mismo hold-out 80/20 de la §21.6 (tr_H / te_H) para las curvas de aprendizaje
t0 = time.time()
pipe_mlp.fit(df_H.iloc[tr_H][FEATS_H], y_H[tr_H])
t_fit = time.time() - t0

mlp_f  = pipe_mlp.named_steps['clf']
n_in   = mlp_f.coefs_[0].shape[0]
n_par  = sum(c.size for c in mlp_f.coefs_) + sum(b.size for b in mlp_f.intercepts_)
print(f"Arquitectura: {n_in} entradas → "
      f"{' → '.join(map(str, mlp.hidden_layer_sizes))} → 1 (sigmoide)")
print(f"Parámetros entrenables: {n_par:,} | épocas efectivas: {mlp_f.n_iter_} "
      f"| tiempo de ajuste: {t_fit:.1f}s")

# Curvas de aprendizaje (equivalente a 'Training and validation loss' del doc)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(mlp_f.loss_curve_, color='#1565C0', lw=2)
ax[0].set_title('Pérdida de entrenamiento (log-loss)', fontweight='bold')
ax[0].set_xlabel('Época'); ax[0].set_ylabel('Loss')
ax[1].plot(mlp_f.validation_scores_, color='#EF6C00', lw=2)
best_ep = int(np.argmax(mlp_f.validation_scores_))
ax[1].axvline(best_ep, color='red', ls=':', lw=1, label=f'Mejor época = {best_ep}')
ax[1].set_title('Score en validación interna (15% del train)', fontweight='bold')
ax[1].set_xlabel('Época'); ax[1].set_ylabel('Accuracy'); ax[1].legend(fontsize=8)
plt.suptitle('Red neuronal (MLP) — curvas de aprendizaje', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR/'rna_curvas_aprendizaje.png', dpi=140, bbox_inches='tight')
plt.show()

# Chequeo rápido en el hold-out 20% (mismas filas que la §21.6)
prob_mlp_te = pipe_mlp.predict_proba(df_H.iloc[te_H][FEATS_H])[:, 1]
print(f"\nHold-out 20% — PR-AUC: {average_precision_score(y_H[te_H], prob_mlp_te):.4f} "
      f"| ROC-AUC: {roc_auc_score(y_H[te_H], prob_mlp_te):.4f}")

### Sección 25.2 — Evaluación honesta (OOF) y comparación con el modelo final

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo</b><br><br>
En el documento de referencia, la evaluación consiste en <b>predecir</b> sobre el conjunto de prueba y calcular <b>métricas</b> (MAE, MSE, R²). Aquí se hace lo mismo pero con el protocolo más exigente de este proyecto: probabilidades <b>out-of-fold</b> (5-fold, dos semillas) — cada lead es calificado por una red que <b>nunca lo vio</b> al entrenar — y las métricas propias de clasificación desbalanceada (PR-AUC, ROC-AUC, lift y costo de alcanzar recall 0.70), <b>idénticas a las de la §21.4</b> para que la comparación con la regresión logística sea directa.
</div>

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b>
<ol>
<li>Calcula probabilidades OOF con <code>cross_val_predict</code> (5-fold estratificado) para dos semillas: <i>selección</i> y <i>confirmación</i> — igual que en la §21.4. ⏱️ Son 10 ajustes de la red; la celda puede tardar varios minutos.</li>
<li>Con esas probabilidades computa PR-AUC, ROC-AUC, lift@10%, recall@20% y el % de leads necesario para capturar el 70% de las conversiones.</li>
<li>Une la tabla con <code>met_final</code> (métricas OOF del modelo final LR) y guarda la comparación en <code>results/rna_vs_lr_metricas.csv</code>.</li>
<li>Imprime un <b>veredicto</b> automático según la diferencia media de PR-AUC entre ambos modelos.</li>
</ol>
</div>


In [ ]:
# ── §25.2  RNA: evaluación OOF 5-fold y comparación contra el modelo final ─
met_rows_mlp = []
prob_mlp = None
for tag, seed in [('selección', SEED), ('confirmación', 2025)]:
    skf_m = StratifiedKFold(5, shuffle=True, random_state=seed)
    prob = cross_val_predict(pipe_mlp, df_H[FEATS_H], y_H, cv=skf_m,
                             method='predict_proba', n_jobs=-1)[:, 1]
    if prob_mlp is None:
        prob_mlp = prob
    bm = business_at_k(y_H, prob)
    prc, rcc, thr = precision_recall_curve(y_H, prob)
    i70 = np.where(rcc[:-1] >= 0.70)[0][-1]
    met_rows_mlp.append({'CV': tag,
        'pr_auc'       : round(average_precision_score(y_H, prob), 4),
        'roc_auc'      : round(roc_auc_score(y_H, prob), 4),
        'lift@10%'     : round(bm[0.10]['lift'], 2),
        'recall@20%'   : round(bm[0.20]['rec'], 3),
        'pct_leads_r70': round((prob >= thr[i70]).mean()*100, 1),
        'prec_r70'     : round(prc[i70], 4)})
met_mlp = pd.DataFrame(met_rows_mlp)
print("MÉTRICAS DE LA RED NEURONAL (OOF 5-fold, dos semillas):")
print(met_mlp.to_string(index=False))

# Comparación directa contra el modelo final (LR C=0.55, §21.4)
comp = pd.concat([met_final.assign(modelo='LR (modelo final)'),
                  met_mlp.assign(modelo='Red neuronal (MLP)')], ignore_index=True)
comp = comp[['modelo'] + [c for c in comp.columns if c != 'modelo']]
print("\nCOMPARACIÓN LR vs RNA (mismo universo H, mismo esquema OOF):")
print(comp.to_string(index=False))
comp.to_csv(RESULTS_DIR/'rna_vs_lr_metricas.csv', index=False)

d_pr = met_mlp['pr_auc'].mean() - met_final['pr_auc'].mean()
if d_pr > 0.005:
    veredicto = "la red neuronal SUPERA a la regresión logística en PR-AUC"
elif d_pr < -0.005:
    veredicto = ("la regresión logística sigue siendo superior — "
                 "la RNA no aporta sobre el modelo final")
else:
    veredicto = "ambos modelos son prácticamente equivalentes en PR-AUC"
print(f"\nVeredicto (ΔPR-AUC medio = {d_pr:+.4f}): {veredicto}.")

### Sección 25.3 — Curvas comparativas: LR vs Red Neuronal

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo</b><br><br>
El documento de referencia cierra graficando <b>real vs. predicho</b> en entrenamiento y prueba. El análogo para clasificación es superponer, para ambos modelos y con las mismas probabilidades OOF, las tres curvas que resumen el desempeño: <b>ganancia acumulada</b>, <b>precisión-recall</b> y <b>calibración</b>.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer cada gráfico</b>
<ul>
<li><b>Ganancia acumulada:</b> la curva que se despega más de la diagonal captura más conversiones contactando menos leads. La línea gris punteada marca la meta de recall 70%.</li>
<li><b>Precision-Recall:</b> la curva más alta domina; la leyenda muestra el AP (= PR-AUC) de cada modelo. La línea gris es la tasa base (3.08%).</li>
<li><b>Calibración:</b> el modelo cuyos puntos siguen mejor la diagonal produce probabilidades más confiables (un lead con prob. 0.10 realmente convierte ≈10% de las veces). Las redes neuronales sin calibrar tienden a desviarse más que la LR.</li>
</ul>
</div>


In [ ]:
# ── §25.3  Curvas comparativas: LR (modelo final) vs Red Neuronal ──────────
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.8))

for prob, nombre, color in [(prob_H,   'LR (modelo final)',  '#1565C0'),
                            (prob_mlp, 'Red neuronal (MLP)', '#C62828')]:
    order = np.argsort(prob)[::-1]
    cg  = np.cumsum(y_H[order]) / y_H.sum()
    pct = np.arange(1, len(y_H)+1) / len(y_H)
    ax[0].plot(pct, cg, lw=2, color=color, label=nombre)
    prc_c, rcc_c, _ = precision_recall_curve(y_H, prob)
    ax[1].plot(rcc_c, prc_c, lw=1.8, color=color,
               label=f'{nombre} (AP={average_precision_score(y_H, prob):.3f})')
    fp_c, mpv_c = calibration_curve(y_H, prob, n_bins=10, strategy='quantile')
    ax[2].plot(mpv_c, fp_c, 'o-', color=color, label=nombre)

ax[0].plot([0, 1], [0, 1], 'k--', lw=.8, label='Aleatorio')
ax[0].axhline(0.70, color='gray', ls=':', lw=1)
ax[0].set_title('Ganancia acumulada', fontweight='bold')
ax[0].set_xlabel('% leads contactados'); ax[0].set_ylabel('% conversiones capturadas')
ax[1].axhline(y_H.mean(), color='gray', ls='--', lw=.8)
ax[1].set_title('Precision-Recall', fontweight='bold')
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
lim = ax[2].get_xlim()[1]
ax[2].plot([0, lim], [0, lim], 'k--', lw=.8)
ax[2].set_title('Calibración', fontweight='bold')
ax[2].set_xlabel('Prob. predicha media'); ax[2].set_ylabel('Frac. positivos observada')
for a in ax:
    a.legend(fontsize=8)
plt.suptitle('Modelo final (LR) vs Red neuronal — probabilidades OOF, universo H',
             fontsize=12.5, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR/'rna_vs_lr_curvas.png', dpi=140, bbox_inches='tight')
plt.show()

## Conclusión — Sección 25

**Cómo interpretar los resultados de esta sección** (el análisis puntual depende de las tablas impresas arriba, análogo al análisis de métricas del documento de referencia):

- **Sobreajuste (equivalente al análisis MAE train vs. prueba del documento).** El *early stopping* congela los pesos de la mejor época de validación; si el PR-AUC del hold-out (§25.1) y el OOF (§25.2) son similares entre sí y entre las dos semillas, la red generaliza de forma consistente, igual que se concluía en el documento cuando el MAE de entrenamiento y prueba coincidían.
- **¿Aporta la RNA sobre el modelo final?** El veredicto automático de la §25.2 compara el PR-AUC medio de ambos modelos. En datos tabulares con pocas variables — y con las interacciones relevantes (`edad×ingreso`, `edad×referido`, `ingreso×inversión`) ya construidas a mano en `FEATS_H` — es habitual que una red neuronal **empate o quede por debajo** de una regresión logística bien regularizada: la ventaja de la red (aprender interacciones no lineales automáticamente) ya está cubierta por la ingeniería de características de la §21.
- **Costo de adoptar la RNA.** Aun si empatara en métricas, la red pierde frente a la LR en tres frentes operativos: (1) **interpretabilidad** — no existe la fórmula cerrada de la §24 ni coeficientes con odds ratio; (2) **calibración** — las probabilidades del MLP suelen desviarse más de la diagonal (§25.3), y en este proyecto los umbrales operativos (§21.4) dependen de probabilidades confiables; (3) **tiempo de entrenamiento** mayor.
- **Decisión.** El modelo final del proyecto **sigue siendo la regresión logística (§21.4)** salvo que la RNA muestre una mejora consistente de PR-AUC en *ambas* semillas de validación. Esta sección queda como verificación de que un modelo no lineal más expresivo no encuentra señal adicional que la LR estuviera dejando sobre la mesa.


---

## Sección 26 — Métricas consolidadas del modelo final (OOF)

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
A lo largo de las Secciones 21 a 24 el modelo final se evaluó con distintas métricas, pero repartidas en varias celdas. Aquí se reúnen todas en una sola tabla — discriminación (PR-AUC, ROC-AUC), calibración (Brier score) y desempeño al umbral operativo (accuracy, F1, F-beta, MCC, kappa) — sobre las mismas probabilidades OOF del universo H usadas en toda la Parte III.
</div>

In [ ]:
# ── §26  Métricas consolidadas del modelo final (universo H, OOF) ─────────
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
    accuracy_score, f1_score, fbeta_score, balanced_accuracy_score,
    matthews_corrcoef, cohen_kappa_score, precision_recall_curve, confusion_matrix)

# Umbral operativo: menor umbral con recall >= 0.70 sobre las probs OOF (mismo criterio §21.4/§22)
prc_c, rcc_c, thr_c = precision_recall_curve(y_H, prob_H)
i70_c = np.where(rcc_c[:-1] >= 0.70)[0][-1]
umbral_c = thr_c[i70_c]
y_pred_c = (prob_H >= umbral_c).astype(int)

metricas_consolidadas = pd.DataFrame([
    {'métrica': 'PR-AUC',            'valor': round(average_precision_score(y_H, prob_H), 4)},
    {'métrica': 'ROC-AUC',           'valor': round(roc_auc_score(y_H, prob_H), 4)},
    {'métrica': 'Brier score',       'valor': round(brier_score_loss(y_H, prob_H), 4)},
    {'métrica': 'Accuracy',          'valor': round(accuracy_score(y_H, y_pred_c), 4)},
    {'métrica': 'Balanced accuracy', 'valor': round(balanced_accuracy_score(y_H, y_pred_c), 4)},
    {'métrica': 'F1',                'valor': round(f1_score(y_H, y_pred_c), 4)},
    {'métrica': 'F2 (recall-weighted)', 'valor': round(fbeta_score(y_H, y_pred_c, beta=2), 4)},
    {'métrica': 'MCC',               'valor': round(matthews_corrcoef(y_H, y_pred_c), 4)},
    {'métrica': "Cohen's kappa",     'valor': round(cohen_kappa_score(y_H, y_pred_c), 4)},
])
print(f"Umbral operativo (recall >= 0.70): {umbral_c:.4f}\n")
print("MÉTRICAS CONSOLIDADAS — MODELO FINAL (universo H, OOF):")
print(metricas_consolidadas.to_string(index=False))
metricas_consolidadas.to_csv(RESULTS_DIR/'modelo_final_metricas_consolidadas.csv', index=False)

---

## Sección 27 — Modelo final vs. selección aleatoria (Monte Carlo)

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
Se replica la simulación Monte Carlo de la Sección 9.3, pero ahora sobre el modelo final (universo H). La pregunta de negocio es: si ventas solo puede atender un Top X% de los leads, ¿cuántas conversiones reales se capturan usando el modelo, frente a elegir ese mismo % de leads al azar? Se comparan contra un intervalo de confianza del 95% de la selección aleatoria (1000 iteraciones).
</div>

In [ ]:
# ── §27  Modelo final vs selección aleatoria (Monte Carlo) ────────────────
N_MC = 1000; rng = np.random.default_rng(SEED)
p = prob_H; yt = y_H
order = np.argsort(p)[::-1]; cum = np.cumsum(yt[order]); n = len(yt); base = yt.mean()

mc_rows = []
for k in (0.05, 0.10, 0.20, 0.30):
    tn = int(np.ceil(n * k)); tp_m = int(cum[tn - 1])
    rand = np.array([yt[rng.choice(n, tn, replace=False)].sum() for _ in range(N_MC)])
    mc_rows.append({
        'top_%': int(k * 100), 'n_leads': tn,
        'modelo_conversiones': tp_m, 'modelo_recall': round(tp_m / yt.sum(), 3),
        'mc_media': round(rand.mean(), 2), 'mc_ci95_lo': round(np.percentile(rand, 2.5), 2),
        'mc_ci95_hi': round(np.percentile(rand, 97.5), 2),
        'supera_azar': '***' if tp_m > np.percentile(rand, 97.5) else ''})
mc_final = pd.DataFrame(mc_rows)
print("MODELO FINAL vs. SELECCIÓN ALEATORIA (Monte Carlo, 1000 iteraciones):")
print(mc_final.to_string(index=False))
print("\nNota: *** = el modelo captura más conversiones que el límite superior del IC95%% de elegir al azar.")
mc_final.to_csv(RESULTS_DIR/'modelo_final_montecarlo.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(mc_final['top_%'], mc_final['modelo_conversiones'], 'o-', color='#1565C0',
        lw=2, label='Modelo final')
ax.plot(mc_final['top_%'], mc_final['mc_media'], 's--', color='#9E9E9E',
        lw=2, label='Selección aleatoria (media)')
ax.fill_between(mc_final['top_%'], mc_final['mc_ci95_lo'], mc_final['mc_ci95_hi'],
                 color='#9E9E9E', alpha=.25, label='IC 95% aleatorio')
ax.set_xlabel('Top % de leads contactados', fontweight='bold')
ax.set_ylabel('Conversiones capturadas', fontweight='bold')
ax.set_title('Modelo final vs. selección aleatoria (universo H, OOF)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR/'modelo_final_montecarlo.png', dpi=140, bbox_inches='tight')
plt.show()